# Deliverable: rebuild, reconcile, check price inputs and ledger arithmetic, measure error severity

This notebook is the code deliverable (brief §7, "Code and data package"): it loads the full release through `manifest.csv`, reconstructs and reconciles positions, checks price inputs and ledger arithmetic, writes the defect register and measures the severity of what it found. Everything downstream reads the `outputs/02_*` files it writes: `docs/FINDINGS.md` (via `scripts/build_findings.py`), the dashboard store (`scripts/build_store.py`), the deck and figure scripts, and the tests.

Sections 1 to 10 were written for the Check-in 2 method gate and answer its three questions on the **full data**:

1. Does the rebuild run on the full data?
2. How many breaks, classified how?
3. What did we assume about REVERSED, and what else could it mean?

They also run the price and ledger checks we chose, sort corporate actions into supported and quarantined, trace one record hop by hop, and write the defect register. Section 11 measures the severity of the defects the register holds (NAV sensitivity in basis points of fund net assets, and distinct accounts reached) and writes `02_severity.csv` with its fund, record and account drill-downs plus the severity plot. Section 12 is the run record.

**How to run:** from the repo root, `uv run jupyter lab`, open this file, then *Run All* (headless: `tests/run_notebook.sh`). Thresholds and treatment choices live in `config/reconciliation.toml`; change a value there and rerun. Results go to `outputs/02_*.csv` (`02_` is this notebook's output prefix, kept from its Check-in 2 name so that every script and test reading the files is unchanged) and `outputs/02_severity.png` / `.pdf`. Nothing in `data/` is changed. The plan behind every choice is `docs/checkins/checkin2/CHECKIN2_PLAN.md`.

**Reading the checks:** cells that start with `# CHECK` stop the notebook if something does not add up (for example, groups that do not sum to their denominator). Checks against notebook 01's numbers (`notebooks/additional/01_explore_tables.ipynb`, an independent exploration of the same data) only stop the notebook when the settings **and** the data are the same as notebook 01's; after a settings change, on a test file, or when notebook 01 has not been run, they report instead.

**Test options** (environment variables, all optional): `RECONCILIATION_CONFIG` = another config file; `RECONCILIATION_DATA` = another `manifest.csv` (its files are read relative to the manifest's folder, for example the small made-up fixture in `tests/fixtures/`); `RECONCILIATION_OUTPUTS` = another output folder. Without them the notebook reads the release through `manifest.csv` and writes to `outputs/`.

## 1. Setup and loading

Every table is read **as text** (DuckDB, `all_varchar=true`), then converted explicitly. Values that fail to convert are kept and counted, never dropped. The row ledger records rows in and out at every load, filter and join.

In [ ]:
import csv
import os
import platform
import time
import tomllib
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)
RUN_START = time.time()


def find_repo_root(start=None):
    """Walk up from `start` until we find the folder that holds manifest.csv and data/."""
    folder = Path(start or Path.cwd()).resolve()
    for candidate in [folder, *folder.parents]:
        if (candidate / 'manifest.csv').is_file() and (candidate / 'data').is_dir():
            return candidate
    raise FileNotFoundError(f'No folder with manifest.csv and data/ above {folder}')


ROOT = find_repo_root()


def path_from_env(variable, default):
    """A path from an optional environment variable (for tests), else the default; relative paths start at the repo root."""
    path = Path(os.environ.get(variable) or default)
    return path if path.is_absolute() else ROOT / path


CONFIG_PATH = path_from_env('RECONCILIATION_CONFIG', 'config/reconciliation.toml')
MANIFEST_PATH = path_from_env('RECONCILIATION_DATA', 'manifest.csv')   # its files are read relative to its folder
OUTPUTS = path_from_env('RECONCILIATION_OUTPUTS', 'outputs')
OUTPUTS.mkdir(parents=True, exist_ok=True)
NB01_OUTPUTS = ROOT / 'outputs'          # where notebook 01 writes; read only by the reproduction checks
with CONFIG_PATH.open('rb') as stream:
    CONFIG = tomllib.load(stream)

AS_OF_DATE = CONFIG['snapshot']['as_of_date']
TOLERANCE = CONFIG['tolerances']['quantity_match_tolerance']
GROSS_TOLERANCE = CONFIG['tolerances']['gross_amount_tolerance']
DERIVED_UNITS_TOLERANCE = CONFIG['tolerances']['derived_units_tolerance']
POLICY = CONFIG['policies']
REPORTING_CURRENCY = CONFIG['fx']['reporting_currency']
FEED_LATENCY_LIMITS = CONFIG['timeliness']['feed_latency_limit_minutes']              # per source_system
PROCESSING_LATENCY_LIMITS = CONFIG['timeliness']['processing_latency_limit_minutes']  # per source_system
LAG_LIMIT = int(CONFIG['tolerances']['settlement_lag_limit_business_days'])   # T+n, business days (Mon-Fri)
EXTENSION_NOTE = str(CONFIG['tolerances']['settlement_extension_note'])       # settlement_note that clears a longer lag
NAV_CUTOFF_TIME = str(CONFIG['prices']['nav_cutoff_time'])                   # a quote stamped at it is on time
FLAT_RUN_DATES = int(CONFIG['prices']['flat_run_dates'])                     # identical chosen price on this many dates
FX_BAND = float(CONFIG['fx']['day_on_day_band'])                             # |rate / previous business day's - 1|
FLOAT_DECIMALS = int(CONFIG['tolerances']['float_decimals'])                 # rounding before a threshold comparison

ROW_LEDGER = []   # one line per load, filter and join: rows in, rows out, why


def log_rows(step, rows_in, rows_out, why):
    ROW_LEDGER.append({'step': step, 'rows in': rows_in, 'rows out': rows_out, 'why': why})


print('Python', platform.python_version(), '| pandas', pd.__version__, '| duckdb', duckdb.__version__)
SHUFFLE_SEED = CONFIG['robustness']['shuffle_seed']
for label, path in [('config', CONFIG_PATH), ('data manifest', MANIFEST_PATH), ('outputs', OUTPUTS)]:
    print(f'{label}:', path.relative_to(ROOT) if path.is_relative_to(ROOT) else path)
print('Snapshot', AS_OF_DATE, '| quantity tolerance', TOLERANCE, '| shuffle seed', SHUFFLE_SEED, '| policies', POLICY)

In [ ]:
with MANIFEST_PATH.open(newline='', encoding='utf-8') as stream:
    MANIFEST = {row['table']: row for row in csv.DictReader(stream)}

con = duckdb.connect()


def load_text(name):
    """Read one table with every column as text. A blank field stays '' (as in notebook 01), never NULL or NaN."""
    path = (MANIFEST_PATH.parent / MANIFEST[name]['file']).as_posix()
    frame = con.sql(f"SELECT * FROM read_csv('{path}', all_varchar=true, header=true)").df()
    frame = frame.fillna('')
    log_rows(f'load {name}', int(MANIFEST[name]['rows']), len(frame), 'read as text; manifest rows in, loaded rows out')
    return frame


tables = {name: load_text(name) for name in MANIFEST}
if SHUFFLE_SEED:   # robustness test: every result must be the same whatever the row order
    tables = {name: frame.sample(frac=1, random_state=SHUFFLE_SEED).reset_index(drop=True) for name, frame in tables.items()}
    print(f'every table shuffled with seed {SHUFFLE_SEED}')
accounts, securities = tables['accounts'], tables['securities']
transactions, pipeline_events = tables['transactions'], tables['pipeline_events']
positions, comparison_positions = tables['positions'], tables['comparison_positions']
corporate_actions, prices = tables['corporate_actions'], tables['prices']
fx_rates, ledger_nav = tables['fx_rates'], tables['ledger_nav']
print(f'loaded {len(tables)} tables in {time.time() - RUN_START:.0f}s')

In [ ]:
# CHECK: every table loaded with the row and column counts in the jury's manifest.
load_check = pd.DataFrame([
    {'table': name,
     'manifest rows': int(MANIFEST[name]['rows']), 'loaded rows': len(tables[name]),
     'manifest columns': int(MANIFEST[name]['columns']), 'loaded columns': tables[name].shape[1]}
    for name in MANIFEST])
load_check['result'] = np.where((load_check['manifest rows'] == load_check['loaded rows'])
                                & (load_check['manifest columns'] == load_check['loaded columns']), 'PASS', 'FAIL')
assert set(MANIFEST) == set(tables), 'the manifest and the loaded tables name different tables'
assert (load_check['result'] == 'PASS').all(), load_check[load_check['result'] == 'FAIL']
load_check

## 2. The rebuild: raw benchmark first, then each justified step

We rebuild every account + security from zero (the Guide's shared convention): BUY adds, SELL subtracts. The **raw benchmark** uses every row as delivered. Each later step is one justified change, and the table shows its effect. The main chain is fixed in `config/reconciliation.toml` before looking at results:

| Step | What it does | Why |
|---|---|---|
| C0 | every row, raw signed quantity | the raw benchmark |
| C1 | drop the second copy of exact and date-format-only duplicates | a copy of the same record is not a second trade (evidence in 2b) |
| C2 | keys touched by a **conflicting** duplicate pair: *not evaluable* | we cannot know which copy is right |
| C3 | drop CANCELLED rows | a cancelled trade is not a holding; the Guide's BOOKED-only benchmark drops them too |
| C4 | drop REVERSED rows (**R1**: the original stands) | R1 = BOOKED-only, the Guide's shared benchmark |
| C5 | negative quantities kept **raw**; their keys are flagged later | never take absolute values silently |
| C6 | keys with a **blank** quantity: *not evaluable* | unknown never becomes zero; blank-as-zero and gross_amount / price are sensitivities (evidence in 2b) |
| C7 | trades with trade date up to the snapshot | `positions` is a trade-date book |
| C7s | same, by scheduled settlement date | reported beside C7; a plan, not proof of settlement |

**R3** (the REVERSED reading shown beside R1) offsets each original once, with its *first* REVERSED row: the earliest `event_timestamp`, then the earliest trade date, then the lowest `transaction_id`. It is chosen among the rows still in the chain after C1 to C3, never by file order, so shuffling rows cannot change it.

A key's group against a reported table: *matched*, *different*, *only in rebuild* (nets to zero, or nonzero), *only in reported table*, or *not evaluable*. Rate A = matched / (matched + different); rate B = matched / keys in union.

In [ ]:
def normalise_date(raw_dates):
    """Turn YYYY-MM-DD and DD/MM/YYYY text into YYYY-MM-DD text. Blanks stay blank; other values are returned unchanged."""
    iso = pd.to_datetime(raw_dates, format='%Y-%m-%d', errors='coerce')
    slash = pd.to_datetime(raw_dates, format='%d/%m/%Y', errors='coerce')
    return iso.fillna(slash).dt.strftime('%Y-%m-%d').fillna(raw_dates)


def normalise_timestamp(raw_timestamps):
    """Normalise the date part (first 10 characters) and keep the time part as it is."""
    return normalise_date(raw_timestamps.str.slice(0, 10)) + raw_timestamps.str.slice(10)


def to_day(raw_dates):
    """Text date in either format -> Timestamp; NaT where blank or unreadable (the raw text is kept in its own column)."""
    return pd.to_datetime(normalise_date(raw_dates), format='%Y-%m-%d', errors='coerce')


def to_number(raw_numbers):
    """Text -> number; NaN where blank, unreadable or not finite ('inf' and 'nan' are unreadable, never a value)."""
    number = pd.to_numeric(raw_numbers, errors='coerce')
    return number.where(np.isfinite(number))


# Explicit conversions; the raw text columns stay next to the converted ones.
trades = transactions.copy()
trades['row_number'] = np.arange(len(trades))                     # lineage only, never a key
trades['trade_day'] = to_day(trades['trade_date'])
trades['settlement_day'] = to_day(trades['settlement_date'])
trades['quantity_number'] = to_number(trades['quantity'])
trades['quantity_blank'] = trades['quantity'] == ''
trades['price_number'] = to_number(trades['price'])
trades['gross_number'] = to_number(trades['gross_amount'])
direction = trades['transaction_type'].map({'BUY': 1, 'SELL': -1})   # missing for any other type
trades['signed_quantity'] = direction * trades['quantity_number']
trades['direction'] = direction

# A blank quantity whose size the row still carries: gross_amount / price lies within the config tolerance of a whole
# number of units, and the row has no related_transaction_id (a rebook can keep the cancelled trade's gross_amount).
# Only the "derived" sensitivity uses it (D-047); the main chain never fills a blank.
implied_units = trades['gross_number'] / trades['price_number'].where(trades['price_number'] > 0)
whole_units = implied_units.round()
trades['derivable_quantity'] = whole_units.where(trades['quantity_blank'] & (whole_units > 0)
                                                 & ((implied_units - whole_units).abs() <= DERIVED_UNITS_TOLERANCE)
                                                 & (trades['related_transaction_id'] == ''))

# Repeated transaction_ids: each pair is an exact copy, a copy that differs only by date format, or a real conflict.
copies_per_id = trades['transaction_id'].value_counts()
assert copies_per_id.max() <= 2, 'an ID appears more than twice; the pair logic below compares only two copies'
data_columns = [column for column in transactions.columns if column != 'transaction_id']
repeated = trades[trades['transaction_id'].duplicated(keep=False)]
repeated_copy = repeated.groupby('transaction_id').cumcount()
copy_a = repeated[repeated_copy == 0].set_index('transaction_id')[data_columns]
copy_b = repeated[repeated_copy == 1].set_index('transaction_id')[data_columns].loc[copy_a.index]


def normalise_transaction_dates(frame):
    frame = frame.copy()
    frame['trade_date'] = normalise_date(frame['trade_date'])
    frame['settlement_date'] = normalise_date(frame['settlement_date'])
    frame['event_timestamp'] = normalise_timestamp(frame['event_timestamp'])
    return frame


raw_difference = (copy_a != copy_b).any(axis=1)
real_difference = (normalise_transaction_dates(copy_a) != normalise_transaction_dates(copy_b)).any(axis=1)
pair_type = pd.Series('exact copy', index=copy_a.index)
pair_type[raw_difference & ~real_difference] = 'differs only by date format'
pair_type[real_difference] = 'real conflict'
trades['pair_type'] = trades['transaction_id'].map(pair_type).fillna('not repeated')
trades['copy_number'] = trades.groupby('transaction_id').cumcount()   # 0 = first copy in file order

# REVERSED rows point to a BOOKED original. Which REVERSED row comes first (for R3) is decided inside the R3 step,
# on the rows that survive the earlier steps, from data fields only (see reversed_R3), never from file order.
is_reversed = trades['status'] == 'REVERSED'
trades['is_reversed_original'] = ((trades['status'] == 'BOOKED')
                                  & trades['transaction_id'].isin(trades.loc[is_reversed, 'related_transaction_id']))
trades['event_time'] = pd.to_datetime(normalise_timestamp(trades['event_timestamp']), format='%Y-%m-%d %H:%M:%S',
                                      errors='coerce')   # NaT where blank or unreadable

conversion_report = pd.Series({
    'rows': len(trades),
    'trade_date blank or unreadable': int(trades['trade_day'].isna().sum()),
    'settlement_date blank': int((trades['settlement_date'] == '').sum()),
    'settlement_date unreadable (not blank)': int((trades['settlement_day'].isna() & (trades['settlement_date'] != '')).sum()),
    'quantity blank': int(trades['quantity_blank'].sum()),
    'quantity blank, size derivable from gross_amount / price': int(trades['derivable_quantity'].notna().sum()),
    'quantity unreadable (not blank)': int((trades['quantity_number'].isna() & ~trades['quantity_blank']).sum()),
    'quantity negative': int((trades['quantity_number'] < 0).sum()),
    'transaction_type not BUY or SELL': int(direction.isna().sum()),
    'price blank or unreadable': int(trades['price_number'].isna().sum()),
    'gross_amount blank or unreadable': int(trades['gross_number'].isna().sum()),
})
print('duplicate pairs:', pair_type.value_counts().to_dict())
conversion_report.to_frame('count')

In [ ]:
# The steps (ported from notebook 01 section 9). Each takes and returns trade rows.
def raw_rows(rows):
    """C0: every row, raw signed quantity, nothing marked."""
    return rows.assign(quantity_used=rows['signed_quantity'], mark_not_evaluable=False)

def drop_second_copies(rows, pair_types):
    return rows[~((rows['copy_number'] >= 1) & rows['pair_type'].isin(pair_types))]

def drop_exact_and_format_copies(rows):
    return drop_second_copies(rows, ['exact copy', 'differs only by date format'])

def copies_kept_sensitivity(rows):
    """Sensitivity only: both copies count, as the book's transaction_count does (D-044)."""
    return rows

def conflicts_not_evaluable(rows):
    return rows.assign(mark_not_evaluable=rows['mark_not_evaluable'] | (rows['pair_type'] == 'real conflict'))

def conflicts_keep_first(rows):
    return drop_second_copies(rows, ['real conflict'])

def conflicts_keep_second(rows):
    return rows[~((rows['pair_type'] == 'real conflict') & (rows['copy_number'] == 0))]

def conflicts_drop_both(rows):
    return rows[rows['pair_type'] != 'real conflict']

def drop_cancelled(rows):
    return rows[rows['status'] != 'CANCELLED']

def reversed_R1(rows):
    """R1: a REVERSED row is a status record and does not count; the BOOKED original stands."""
    return rows[rows['status'] != 'REVERSED']

def reversed_R2(rows):
    """R2: neither the REVERSED rows nor their BOOKED original count (same quantity as R3, but the key can vanish)."""
    return rows[(rows['status'] != 'REVERSED') & ~rows['is_reversed_original']]

REVERSAL_ORDER = ['event_time', 'trade_day', 'transaction_id']   # "first" = earliest event, then trade date, then lowest ID


def first_reversals(rows):
    """Index of the first REVERSED row per original among `rows`, ordered by data fields (unreadable times last).
    The remaining raw columns break any tie left (only possible between copies of one transaction_id), so file order
    never decides. A REVERSED row with no link is its own original."""
    reversals = rows[rows['status'] == 'REVERSED']
    original = reversals['related_transaction_id'].where(reversals['related_transaction_id'] != '',
                                                         'unlinked:' + reversals['transaction_id'])
    ordered = reversals.assign(original=original).sort_values(REVERSAL_ORDER + data_columns, na_position='last',
                                                              kind='mergesort')
    return ordered.drop_duplicates('original').index


def reversed_R3(rows):
    """R3: the first REVERSED row per original (by data fields, among the rows still in the chain) offsets it once;
    any later REVERSED row for the same original does not count. A REVERSED row whose BOOKED original is not among
    the rows (no link, or a link to an ID not there) has nothing to offset: it counts 0 and marks its key (D-049)."""
    rows = rows[(rows['status'] != 'REVERSED') | rows.index.isin(first_reversals(rows))]
    offset = rows['status'] == 'REVERSED'
    orphan = offset & ~rows['related_transaction_id'].isin(rows.loc[rows['status'] == 'BOOKED', 'transaction_id'])
    return rows.assign(quantity_used=rows['quantity_used'].where(~offset, -rows['quantity_used']).where(~orphan, 0.0),
                       mark_not_evaluable=rows['mark_not_evaluable'] | orphan, orphan_reversal=orphan)

def reversed_R4_evidence_only(rows):
    """R4: every REVERSED row offsets: a double offset. Evidence of how the book seems built, never a treatment."""
    offset = rows['status'] == 'REVERSED'
    return rows.assign(quantity_used=rows['quantity_used'].where(~offset, -rows['quantity_used']))

def negatives_raw(rows):
    return rows                                    # nothing changes; the keys get a sign_conflict flag in section 4

def negatives_not_evaluable(rows):
    return rows.assign(mark_not_evaluable=rows['mark_not_evaluable'] | (rows['quantity_number'] < 0))

def negatives_absolute_sensitivity(rows):
    """Sensitivity only: a negative raw quantity counted as its absolute value."""
    negative = rows['quantity_number'] < 0
    return rows.assign(quantity_used=rows['quantity_used'].where(~negative, -rows['quantity_used']))

def blanks_not_evaluable(rows):
    return rows                                    # a row without a number already marks its key (see rebuild)

def blanks_zero_sensitivity(rows):
    """Sensitivity only: a blank quantity counted as 0 units."""
    return rows.assign(quantity_used=rows['quantity_used'].where(~rows['quantity_blank'], 0.0))

def blanks_derived_sensitivity(rows):
    """Sensitivity only: a blank quantity rebuilt as gross_amount / price where the row supports it (derivable_quantity);
    any other blank keeps no number, so its key stays not evaluable."""
    derived = rows['derivable_quantity'].notna()
    return rows.assign(quantity_used=rows['quantity_used'].where(~derived, rows['direction'] * rows['derivable_quantity']))

CUTOFF = pd.Timestamp(AS_OF_DATE)

def trade_date_cutoff(rows):
    """Trades with trade_date up to the snapshot; an unreadable date marks its key."""
    unknown = rows['trade_day'].isna()
    rows = rows[unknown | (rows['trade_day'] <= CUTOFF)]
    return rows.assign(mark_not_evaluable=rows['mark_not_evaluable'] | rows['trade_day'].isna())

def settlement_date_cutoff(rows):
    """Trades with scheduled settlement_date up to the snapshot; a blank or unreadable date marks its key."""
    unknown = rows['settlement_day'].isna()
    rows = rows[unknown | (rows['settlement_day'] <= CUTOFF)]
    return rows.assign(mark_not_evaluable=rows['mark_not_evaluable'] | rows['settlement_day'].isna())


# Policy switches from the config. R4 and the two sensitivity-only treatments are refused as main choices.
CONFLICT_STEPS = {'not_evaluable': conflicts_not_evaluable, 'keep_first': conflicts_keep_first,
                  'keep_second': conflicts_keep_second, 'drop_both': conflicts_drop_both}
REVERSED_STEPS = {'R1': reversed_R1, 'R2': reversed_R2, 'R3': reversed_R3}
NEGATIVE_STEPS = {'raw': negatives_raw, 'not_evaluable': negatives_not_evaluable}
BLANK_STEPS = {'not_evaluable': blanks_not_evaluable}
DATE_STEPS = {'trade': trade_date_cutoff, 'settlement': settlement_date_cutoff}
for setting, allowed in [('conflicting_pairs', CONFLICT_STEPS), ('reversed_main', REVERSED_STEPS),
                         ('reversed_side', REVERSED_STEPS), ('negatives', NEGATIVE_STEPS),
                         ('blanks', BLANK_STEPS), ('holding_date', DATE_STEPS)]:
    if POLICY[setting] not in allowed:
        raise ValueError(f'{setting} = {POLICY[setting]!r} is not allowed here; choose one of {sorted(allowed)}')
OTHER_DATE = 'settlement' if POLICY['holding_date'] == 'trade' else 'trade'

CHAIN = [  # (code, what the step does, function), fixed before looking at results
    ('C0', 'raw benchmark: every row', raw_rows),
    ('C1', 'drop exact and date-format-only copies', drop_exact_and_format_copies),
    ('C2', f"conflicting pairs: {POLICY['conflicting_pairs']}", CONFLICT_STEPS[POLICY['conflicting_pairs']]),
    ('C3', 'drop CANCELLED', drop_cancelled),
    ('C4', f"REVERSED: {POLICY['reversed_main']}", REVERSED_STEPS[POLICY['reversed_main']]),
    ('C5', f"negative quantities: {POLICY['negatives']}", NEGATIVE_STEPS[POLICY['negatives']]),
    ('C6', f"blank quantities: {POLICY['blanks']}", BLANK_STEPS[POLICY['blanks']]),
    ('C7', f"{POLICY['holding_date']}-date cut-off", DATE_STEPS[POLICY['holding_date']]),
]
OTHER_DATE_CODE = 'C7s' if OTHER_DATE == 'settlement' else 'C7t'
MAIN_STEPS = [function for _, _, function in CHAIN]
print('main chain:', ' -> '.join(f'{code} {what}' for code, what, _ in CHAIN))

In [ ]:
KEY = ['account_id', 'security_id']
POSITION_KEY = ['account_id', 'security_id', 'as_of_date']
GROUPS = ['matched', 'different', 'only in rebuild, nonzero', 'only in rebuild, nets to zero',
          'only in reported table', 'not evaluable (in both)', 'not evaluable (only in rebuild)']

REPORTED = {}
for reported_name, table in [('positions', positions), ('comparison_positions', comparison_positions)]:
    side = table[POSITION_KEY].copy()
    side['as_of_date'] = normalise_date(side['as_of_date'])      # either format joins; unreadable text stays as it is
    side['reported_quantity'] = pd.to_numeric(table['quantity'], errors='coerce')
    if 'transaction_count' in table.columns:
        side['transaction_count'] = pd.to_numeric(table['transaction_count'], errors='coerce')
    # One snapshot: a row dated another day, or whose date does not parse, is set aside so no comparison on account +
    # security can join two dates of one key (Dataset Guide §1: one row per account x security x date). Both counts
    # are logged in the row ledger and printed; neither row is read as a zero or a missing holding.
    readable = to_day(table['as_of_date']).notna()
    other_day = readable & (side['as_of_date'] != AS_OF_DATE)
    log_rows(f'{reported_name}: rows dated another day set aside', len(side), len(side) - int(other_day.sum()),
             'as_of_date (either format) is another day than the snapshot; kept out of every comparison')
    log_rows(f'{reported_name}: rows with an unreadable as_of_date set aside', len(side) - int(other_day.sum()),
             int((readable & ~other_day).sum()), 'as_of_date does not parse in either format; the snapshot is unknown')
    print(f'{reported_name}: set aside {int(other_day.sum()):,} rows dated another day, '
          f'{int((~readable).sum()):,} with an unreadable as_of_date')
    REPORTED[reported_name] = side[readable & ~other_day]


def apply_steps(rows, steps):
    for step in steps:
        rows = step(rows)
    return rows


def rebuild(rows):
    """One row per key: summed quantity, contributing rows, and whether the key is evaluable."""
    rows = rows.assign(no_number=rows['quantity_used'].isna())
    rebuilt = rows.groupby(KEY, as_index=False).agg(
        rebuilt_quantity=('quantity_used', 'sum'), contributing_rows=('quantity_used', 'size'),
        rows_without_number=('no_number', 'sum'), marked=('mark_not_evaluable', 'any'))
    rebuilt['not_evaluable'] = (rebuilt['rows_without_number'] > 0) | rebuilt['marked']
    return rebuilt


def compare_groups(rebuilt, reported):
    """Full outer comparison of a rebuild with one reported table; every key gets exactly one group."""
    merged = rebuilt.assign(as_of_date=AS_OF_DATE).merge(reported, on=POSITION_KEY, how='outer',
                                                         indicator=True, validate='one_to_one')
    in_both = merged['_merge'] == 'both'
    only_rebuild = merged['_merge'] == 'left_only'
    not_evaluable = merged['not_evaluable'].eq(True) | (in_both & merged['reported_quantity'].isna())
    merged['difference'] = merged['rebuilt_quantity'] - merged['reported_quantity']
    within = merged['difference'].abs() <= TOLERANCE
    nets_to_zero = merged['rebuilt_quantity'].abs() <= TOLERANCE
    group = pd.Series('', index=merged.index)
    group[merged['_merge'] == 'right_only'] = 'only in reported table'
    group[only_rebuild & not_evaluable] = 'not evaluable (only in rebuild)'
    group[only_rebuild & ~not_evaluable & nets_to_zero] = 'only in rebuild, nets to zero'
    group[only_rebuild & ~not_evaluable & ~nets_to_zero] = 'only in rebuild, nonzero'
    group[in_both & not_evaluable] = 'not evaluable (in both)'
    group[in_both & ~not_evaluable & within] = 'matched'
    group[in_both & ~not_evaluable & ~within] = 'different'
    assert (group != '').all(), 'a key was left without a group'
    merged['group'] = group
    return merged


def summarise(merged):
    counts = merged['group'].value_counts().reindex(GROUPS, fill_value=0)
    assert counts.sum() == len(merged), 'groups do not add up to the keys in the union'
    summary = {group: int(counts[group]) for group in GROUPS}
    summary['keys in union'] = len(merged)
    comparable = counts['matched'] + counts['different']
    summary['rate A: matched / (matched + different)'] = round(counts['matched'] / comparable, 6) if comparable else np.nan
    summary['rate B: matched / keys in union'] = round(counts['matched'] / len(merged), 6)
    return summary


def run_variant(code, variant, rows):
    """Rebuild from `rows` and compare with both reported tables: one result line per reported table."""
    rebuilt = rebuild(rows)
    lines = []
    for reported_name, reported in REPORTED.items():
        lines.append({'code': code, 'variant': variant, 'compared with': reported_name, 'rows used': len(rows),
                      'keys rebuilt': len(rebuilt), **summarise(compare_groups(rebuilt, reported))})
    return lines, rebuilt

In [ ]:
# The waterfall: C0 raw benchmark, the Guide's BOOKED-only benchmark, then each step of our chain.
waterfall_lines = []
lines, _ = run_variant('BOOKED-only', 'Guide shared benchmark: BOOKED rows only, nothing else changed',
                       apply_steps(trades, [raw_rows, drop_cancelled, reversed_R1]))
waterfall_lines += lines

rows = trades
for code, what, step in CHAIN:
    rows_in = len(rows)
    rows = step(rows)
    log_rows(f'chain {code}', rows_in, len(rows), what)
    lines, rebuilt = run_variant(code, what, rows)
    waterfall_lines += lines
MAIN_ROWS, MAIN_REBUILT = rows, rebuilt                       # the main chain, used by every later section

OTHER_ROWS = apply_steps(trades, MAIN_STEPS[:-1] + [DATE_STEPS[OTHER_DATE]])
lines, OTHER_REBUILT = run_variant(OTHER_DATE_CODE, f'{OTHER_DATE}-date cut-off (reported beside C7)', OTHER_ROWS)
waterfall_lines += lines

waterfall = pd.DataFrame(waterfall_lines)
waterfall.to_csv(OUTPUTS / '02_waterfall.csv', index=False, lineterminator='\n')
waterfall[waterfall['compared with'] == 'positions'].set_index('code').drop(columns='compared with')

In [ ]:
# Sensitivities: the main chain with ONE step swapped for an alternative. Delta = change against the main chain.
def main_chain_with(code, step):
    steps = list(MAIN_STEPS)
    steps[[c for c, _, _ in CHAIN].index(code)] = step
    return apply_steps(trades, steps)


alternatives = [('C1', 'copies: both counted, as the book does (sensitivity only)', copies_kept_sensitivity)]
alternatives += [('C2', f'conflicting pairs: {name}', function) for name, function in CONFLICT_STEPS.items()
                if name != POLICY['conflicting_pairs']]
alternatives += [('C4', f'REVERSED: {name}', function) for name, function in REVERSED_STEPS.items()
                 if name != POLICY['reversed_main']]
alternatives += [('C4', 'REVERSED: R4 every REVERSED row offsets (double offset; evidence only)', reversed_R4_evidence_only),
                 ('C5', 'negatives: absolute value (sensitivity only)', negatives_absolute_sensitivity),
                 ('C6', 'blanks: zero (sensitivity only)', blanks_zero_sensitivity),
                 ('C6', 'blanks: derived from gross_amount / price (sensitivity only)', blanks_derived_sensitivity)]
alternatives += [('C5', f'negatives: {name}', function) for name, function in NEGATIVE_STEPS.items()
                 if name != POLICY['negatives']]

main_lines = waterfall[waterfall['code'] == 'C7'].set_index('compared with')
sensitivity_lines = []
for code, alternative, step in alternatives:
    lines, _ = run_variant(code, alternative, main_chain_with(code, step))
    for line in lines:
        main = main_lines.loc[line['compared with']]
        line['matched vs main chain'] = line['matched'] - main['matched']
        line['different vs main chain'] = line['different'] - main['different']
        line['not evaluable vs main chain'] = ((line['not evaluable (in both)'] + line['not evaluable (only in rebuild)'])
                                               - (main['not evaluable (in both)'] + main['not evaluable (only in rebuild)']))
    sensitivity_lines += lines
sensitivities = pd.DataFrame(sensitivity_lines).rename(columns={'code': 'step swapped', 'variant': 'alternative'})
sensitivities.to_csv(OUTPUTS / '02_sensitivities.csv', index=False, lineterminator='\n')
sensitivities[['step swapped', 'alternative', 'compared with', 'matched', 'different', 'matched vs main chain',
               'different vs main chain', 'not evaluable vs main chain']]

In [ ]:
# CHECK: the chain reproduces notebook 01 (outputs/01_rebuild_steps.csv) when the settings are the same.
NB01_POLICY = {'conflicting_pairs': 'not_evaluable', 'reversed_main': 'R1', 'negatives': 'raw',
               'blanks': 'not_evaluable', 'holding_date': 'trade'}   # notebook 01's chain, section 9.8
NB01_NAMES = {'C0': 'C0 raw benchmark', 'BOOKED-only': 'c alone: BOOKED only (Guide shared benchmark)',
              'C1': 'C1 drop exact and format-only copies', 'C2': 'C2 conflicting keys not evaluable',
              'C3': 'C3 drop CANCELLED', 'C4': 'C4 drop REVERSED (R1, BOOKED-only)', 'C5': 'C5 negatives kept raw',
              'C6': 'C6 blanks not evaluable'}
NB01_DATE_NAMES = {'trade': 'C7 trade-date cut-off', 'settlement': 'C7s settlement-date cut-off'}


def nb01_variant(code):
    """Notebook 01's name for one of our lines, matched by what the line does (C7 follows holding_date)."""
    if code == 'C7':
        return NB01_DATE_NAMES[POLICY['holding_date']]
    if code == OTHER_DATE_CODE:
        return NB01_DATE_NAMES[OTHER_DATE]
    return NB01_NAMES.get(code)


COMPARED = ['rows used', 'keys rebuilt', 'matched', 'different', 'only in rebuild, nonzero',
            'only in rebuild, nets to zero', 'only in reported table', 'not evaluable (in both)',
            'not evaluable (only in rebuild)', 'keys in union']
with (ROOT / 'config' / 'exploration.toml').open('rb') as stream:
    NB01_CONFIG = tomllib.load(stream)
same_settings = (NB01_CONFIG['snapshot']['as_of_date'] == AS_OF_DATE
                 and NB01_CONFIG['reconstruction']['quantity_match_tolerance'] == TOLERANCE
                 and all(POLICY[k] == v for k, v in NB01_POLICY.items()))
same_data = False            # set below only when notebook 01's outputs exist and were made from the same rows
nb01_file = NB01_OUTPUTS / '01_rebuild_steps.csv'
if not nb01_file.is_file():
    print('Not checked: run notebooks/additional/01_explore_tables.ipynb first to create outputs/01_rebuild_steps.csv.')
else:
    nb01 = pd.read_csv(nb01_file).set_index(['variant', 'compared with'])
    ours = (waterfall.assign(variant=waterfall['code'].map(nb01_variant)).dropna(subset=['variant'])
            .set_index(['variant', 'compared with']))
    common = ours.index.intersection(nb01.index)
    not_in_nb01 = ours.index.difference(nb01.index)
    reproduction = (ours.loc[common, COMPARED] - nb01.loc[common, COMPARED]).abs().sum(axis=1).rename('total abs difference')
    if ('C0 raw benchmark', 'positions') in nb01.index:
        same_data = int(nb01.loc[('C0 raw benchmark', 'positions'), 'rows used']) == len(trades)   # not a test fixture
    print(f'settings equal to notebook 01: {same_settings}; same data: {same_data}; variants compared: '
          f'{len(reproduction)}; variants that differ: {int((reproduction > 0).sum())}; '
          f'lines with no notebook 01 counterpart: {list(not_in_nb01)}')
    if same_settings and same_data:
        assert not len(not_in_nb01), f'lines with no notebook 01 counterpart: {list(not_in_nb01)}'
        assert (reproduction == 0).all(), reproduction[reproduction > 0]

# CHECK (made-up rows): R3 offsets an original once; a REVERSED row with nothing to offset (no link, or a link to an
# ID not among the rows) does not count and makes its key not evaluable (D-049).
made_up_r3 = pd.DataFrame({column: [''] * 4 for column in transactions.columns}).assign(
    transaction_id=['O1', 'V1', 'V2', 'V3'], account_id='A', security_id=['S1', 'S1', 'S2', 'S3'],
    status=['BOOKED', 'REVERSED', 'REVERSED', 'REVERSED'], related_transaction_id=['', 'O1', '', 'GONE'],
    quantity_used=[10.0, 10.0, 5.0, 5.0], mark_not_evaluable=False, event_time=pd.NaT, trade_day=pd.NaT)
r3_keys = rebuild(reversed_R3(made_up_r3)).set_index('security_id')
assert r3_keys.loc['S1', 'rebuilt_quantity'] == 0 and not r3_keys.loc['S1', 'not_evaluable'], r3_keys
assert r3_keys.loc['S2', 'not_evaluable'] and r3_keys.loc['S3', 'not_evaluable'], r3_keys


### 2b. The evidence behind two treatments: duplicate copies and blank quantities

Two steps of the chain rest on a reading of the data, so the evidence is written out here (D-044, D-047).

- **Copies (C1).** Do the two copies of a `transaction_id` look like one trade delivered twice? The pipeline says when each copy was received, booked and published. The book's own `transaction_count` says whether the book counted the copy as a transaction.
- **Blank quantities (C6).** Does a blank-quantity row look like a trade that never happened, or like a real trade with one field missing? If the size is still in the row (`gross_amount / price`), how well does that rebuild the quantity on rows where we *can* check it (the backtest)? And what does the book hold on these keys: the trade's units, or nothing?

Everything is counted from field values, never from file order, so shuffling rows changes nothing.

In [ ]:
def to_time(raw):
    return pd.to_datetime(normalise_timestamp(raw), format='%Y-%m-%d %H:%M:%S', errors='coerce')


def shares(values):
    """Share of each value, as text sorted by value (same style as the trace)."""
    return str({k: round(float(v), 4) for k, v in values.value_counts(normalize=True).sort_index().items()})


def evidence(part, measure, count, out_of='', note=''):
    return {'part': part, 'measure': measure, 'count': count, 'out of': out_of, 'note': note}


pipeline = pipeline_events.assign(received=to_time(pipeline_events['received_timestamp']),
                                  booked=to_time(pipeline_events['booked_timestamp']),
                                  published=to_time(pipeline_events['published_timestamp']),
                                  cutoff=to_time(pipeline_events['cutoff_timestamp']))

# --- Copies: exact and date-format-only pairs (the ones C1 removes)
copy_ids = pair_type.index[pair_type != 'real conflict']
per_id = (pipeline[pipeline['transaction_id'].isin(copy_ids)].groupby('transaction_id')
          .agg(records=('transaction_id', 'size'), batches=('batch_id', 'nunique'), received=('received', 'nunique'),
               booked=('booked', 'nunique'), published=('published', 'nunique'), first=('published', 'min'),
               last=('published', 'max'), cutoff=('cutoff', 'max')))
gap_minutes = (per_id['last'] - per_id['first']).dt.total_seconds() / 60
two_records = per_id['records'] == 2
counted_rows = trades[trades['status'] != 'CANCELLED'].assign(
    is_copy=lambda f: f['pair_type'].isin(['exact copy', 'differs only by date format']),
    is_blank=lambda f: f['quantity_blank'])
book_counts = (counted_rows.groupby(KEY).agg(rows=('transaction_id', 'size'), ids=('transaction_id', 'nunique'),
                                             has_copy=('is_copy', 'any'), has_blank=('is_blank', 'any'))
               .reset_index().merge(REPORTED['positions'][KEY + ['transaction_count']], on=KEY, validate='one_to_one'))
book_counts_rows = book_counts['transaction_count'] == book_counts['rows']
with_copy = book_counts['has_copy']
COPY_EVIDENCE = [
    evidence('copies', 'repeated transaction_ids that are copies (exact or date format only)', len(copy_ids),
             trades['transaction_id'].nunique(), 'distinct transaction_ids'),
    evidence('copies', 'copies with two pipeline records', int(two_records.sum()), len(copy_ids)),
    evidence('copies', 'both records in one batch', int((two_records & (per_id['batches'] == 1)).sum()), int(two_records.sum())),
    evidence('copies', 'both records received at the same time', int((two_records & (per_id['received'] == 1)).sum()),
             int(two_records.sum())),
    evidence('copies', 'both records booked at the same time', int((two_records & (per_id['booked'] == 1)).sum()),
             int(two_records.sum())),
    evidence('copies', 'both records published at the same time', int((two_records & (per_id['published'] == 1)).sum()),
             int(two_records.sum()), 'otherwise the second publish comes later'),
    evidence('copies', 'minutes between the two publishes: median', float(gap_minutes[two_records].median()), '',
             f'90th percentile {gap_minutes[two_records].quantile(0.9):.0f}, longest {gap_minutes[two_records].max():.0f}'),
    evidence('copies', 'second publish after the cutoff, first one before it',
             int((two_records & (per_id['last'] > per_id['cutoff']) & (per_id['first'] <= per_id['cutoff'])).sum()),
             int(two_records.sum())),
    evidence('copies', 'share of copies by source_system', shares(trades.loc[trades['transaction_id'].isin(copy_ids)
                                                                            & (trades['copy_number'] == 0), 'source_system']),
             '', f"share of all rows by source_system: {shares(trades['source_system'])}"),
    evidence('copies', 'book positions whose transaction_count = rows not CANCELLED, copies counted',
             int(book_counts_rows.sum()), len(book_counts), 'book positions that have trades'),
    evidence('copies', 'of the book positions holding a copy: transaction_count counts both copies',
             int((with_copy & book_counts_rows).sum()), int(with_copy.sum()),
             f"counts each copy once: {int((with_copy & (book_counts['transaction_count'] == book_counts['ids'])).sum())}"),
]

# --- Blank quantities
blank = trades[trades['quantity_blank']]
blank_ids = set(blank['transaction_id'])
pipe_blank = pipeline[pipeline['transaction_id'].isin(blank_ids)]
booked_and_published = set(pipe_blank.loc[pipe_blank['booked'].notna() & pipe_blank['published'].notna(), 'transaction_id'])
priced = blank['price_number'] > 0
linked = blank['related_transaction_id'] != ''
derivable = blank['derivable_quantity'].notna()
not_whole = priced & ~linked & ~derivable
# Backtest: on rows that HAVE a positive quantity and price, rebuild the quantity as gross_amount / price and compare.
testable = trades[(trades['quantity_number'] > 0) & (trades['price_number'] > 0)]
rebuilt_units = (testable['gross_number'] / testable['price_number']).round()
exact = (rebuilt_units - testable['quantity_number']).abs() <= TOLERANCE
miss_linked = ~exact & (testable['related_transaction_id'] != '')
# Keys: what the book holds where a blank row counts in the main chain
blank_keys = MAIN_ROWS.loc[MAIN_ROWS['quantity_blank'], KEY].drop_duplicates()
derived_vs_book = compare_groups(rebuild(main_chain_with('C6', blanks_derived_sensitivity)), REPORTED['positions'])
zero_vs_book = compare_groups(rebuild(main_chain_with('C6', blanks_zero_sensitivity)), REPORTED['positions'])
blank_key_groups = (blank_keys.merge(derived_vs_book[KEY + ['group', 'reported_quantity']], on=KEY, validate='one_to_one')
                    .merge(zero_vs_book[KEY + ['group']].rename(columns={'group': 'group if zero'}), on=KEY,
                           validate='one_to_one')
                    .merge(REPORTED['comparison_positions'][KEY + ['reported_quantity']]
                           .rename(columns={'reported_quantity': 'custodian_quantity'}), on=KEY, how='left'))
# Evaluable once blanks are derived: in the book (matched / different) or only in the rebuild (a nonzero or zero total).
# The book is "consistent with blank-as-zero" when that rebuild matches it, or nets to zero where the book has no row.
derived_testable = blank_key_groups['group'].isin(['matched', 'different', 'only in rebuild, nonzero',
                                                   'only in rebuild, nets to zero'])
derived_differs = blank_key_groups['group'].isin(['different', 'only in rebuild, nonzero'])
book_as_zero = blank_key_groups['group if zero'].isin(['matched', 'only in rebuild, nets to zero'])
BLANK_OMITTED = blank_key_groups[derived_differs & book_as_zero]
BLANK_KEYS_POPULATION, BLANK_KEYS_TESTABLE = len(blank_key_groups), int(derived_testable.sum())
in_book = BLANK_OMITTED[BLANK_OMITTED['reported_quantity'].notna()]
custodian_same = int(((in_book['custodian_quantity'] - in_book['reported_quantity']).abs() <= TOLERANCE).sum())
with_blank = book_counts['has_blank']
BLANK_EVIDENCE = [
    evidence('blank quantities', 'rows with a blank quantity', len(blank), len(trades)),
    evidence('blank quantities', 'status BOOKED', int((blank['status'] == 'BOOKED').sum()), len(blank),
             f"share BOOKED among all rows: {(trades['status'] == 'BOOKED').mean():.4f}"),
    evidence('blank quantities', 'gross_amount above zero', int((blank['gross_number'] > 0).sum()), len(blank),
             'money moved'),
    evidence('blank quantities', 'booked and published in the pipeline', len(booked_and_published), len(blank_ids),
             'distinct transaction_ids'),
    evidence('blank quantities', 'share by source_system', shares(blank['source_system']), '',
             f"share of all rows by source_system: {shares(trades['source_system'])}"),
    evidence('blank quantities', 'book positions with a blank row: transaction_count counts it',
             int((with_blank & book_counts_rows).sum()), int(with_blank.sum()), 'the book records the trade'),
    evidence('size from gross_amount / price', 'derivable: whole number of units, no link to another trade',
             int(derivable.sum()), len(blank), f'within {DERIVED_UNITS_TOLERANCE} units of a whole number'),
    evidence('size from gross_amount / price', 'held: linked to another trade', int(linked.sum()), len(blank),
             'a rebook can keep the cancelled gross_amount'),
    evidence('size from gross_amount / price', 'not derivable: no price above zero', int((~priced & ~linked).sum()), len(blank)),
    evidence('size from gross_amount / price', 'not derivable: not a whole number of units', int(not_whole.sum()), len(blank)),
    evidence('backtest', 'rows with a quantity: gross_amount / price gives it exactly', int(exact.sum()), len(testable),
             'rows with quantity > 0 and price > 0'),
    evidence('backtest', 'misses on rows linked to another trade', int(miss_linked.sum()), len(testable),
             'mostly rebooks that keep the cancelled gross_amount'),
    evidence('backtest', 'misses on rows with no link', int((~exact & ~miss_linked).sum()), len(testable), 'not yet explained'),
    evidence('positions', 'keys with a blank row in the main chain: evaluable once blanks are derived', BLANK_KEYS_TESTABLE,
             BLANK_KEYS_POPULATION, 'the rest keep a blank that cannot be derived, or a conflicting copy'),
    evidence('positions', 'derived rebuild equals the book', int((blank_key_groups['group'] == 'matched').sum()),
             BLANK_KEYS_TESTABLE),
    evidence('positions', 'derived rebuild differs from the book, which equals blank-as-zero',
             int((derived_differs & book_as_zero & (blank_key_groups['group'] == 'different')).sum()), BLANK_KEYS_TESTABLE,
             f'the custodian holds the same as the book on {custodian_same} of these'),
    evidence('positions', 'derived rebuild holds units; the book has no row (blank-as-zero nets to zero)',
             int((derived_differs & book_as_zero & (blank_key_groups['group'] == 'only in rebuild, nonzero')).sum()),
             BLANK_KEYS_TESTABLE, 'the position is missing from the book'),
    evidence('positions', 'derived rebuild differs, and the book is not blank-as-zero either',
             int((derived_differs & ~book_as_zero).sum()), BLANK_KEYS_TESTABLE),
]
duplicate_profile = pd.DataFrame(COPY_EVIDENCE)
duplicate_profile.to_csv(OUTPUTS / '02_duplicate_profile.csv', index=False, lineterminator='\n')
blank_quantity = pd.DataFrame(BLANK_EVIDENCE)
blank_quantity.to_csv(OUTPUTS / '02_blank_quantity.csv', index=False, lineterminator='\n')
display(duplicate_profile)
blank_quantity

In [ ]:
# CHECK: every blank row lands in exactly one size group, the backtest adds up, and every blank key is in one group.
size_groups = int(derivable.sum()) + int(linked.sum()) + int((~priced & ~linked).sum()) + int(not_whole.sum())
assert size_groups == len(blank), f'blank rows {len(blank)} but the size groups hold {size_groups}'
assert int(exact.sum()) + int(miss_linked.sum()) + int((~exact & ~miss_linked).sum()) == len(testable)
assert len(BLANK_OMITTED) <= int(derived_differs.sum()) <= BLANK_KEYS_TESTABLE <= BLANK_KEYS_POPULATION
assert BLANK_KEYS_POPULATION == len(blank_keys), 'a blank key was lost in the joins'
print(f'blank rows {len(blank):,}: derivable {int(derivable.sum()):,} | backtest exact {int(exact.sum()):,} of {len(testable):,} '
      f'| book leaves out the units on {len(BLANK_OMITTED):,} of {BLANK_KEYS_TESTABLE:,} testable keys')

## 3. Book against custodian, on the full set of keys

`positions` (our book) and `comparison_positions` (the custodian) are joined with a **full outer join** on account + security + as-of date, after checking that each side is unique at that grain. A key on one side only is reported as missing on the other side, never as a zero holding. This comparison does not use the rebuild, so keys the rebuild cannot evaluate still get an answer.

In [ ]:
# Grain check first: each side must be unique on account + security + as-of date.
for name, side in REPORTED.items():
    repeated_keys = int(side.duplicated(POSITION_KEY).sum())
    unreadable = int(side['reported_quantity'].isna().sum())
    print(f'{name}: {len(side):,} rows | repeated keys: {repeated_keys} | quantity unreadable: {unreadable} | '
          f'as_of_date values: {sorted(side["as_of_date"].unique())}')
    assert repeated_keys == 0, f'{name} is not unique at its grain; the comparison below would multiply rows'

book = REPORTED['positions'].rename(columns={'reported_quantity': 'book_quantity',
                                             'transaction_count': 'book_transaction_count'})
custodian = REPORTED['comparison_positions'].rename(columns={'reported_quantity': 'custodian_quantity'})
book_vs_custodian = book.merge(custodian, on=POSITION_KEY, how='outer', indicator='side', validate='one_to_one')
book_vs_custodian['side'] = book_vs_custodian['side'].map(
    {'both': 'both', 'left_only': 'only in positions', 'right_only': 'only in comparison_positions'})
log_rows('join positions <-> comparison_positions', len(book) + len(custodian), len(book_vs_custodian),
         'full outer join on account + security + as_of_date; rows out = keys in the union')

both_sides = book_vs_custodian['side'] == 'both'
unreadable = book_vs_custodian['book_quantity'].isna() | book_vs_custodian['custodian_quantity'].isna()
book_vs_custodian['book_minus_custodian'] = book_vs_custodian['book_quantity'] - book_vs_custodian['custodian_quantity']
agree = book_vs_custodian['book_minus_custodian'].abs() <= TOLERANCE
book_vs_custodian['group'] = np.select(
    [~both_sides, both_sides & unreadable, both_sides & agree],
    [book_vs_custodian['side'], 'not evaluable (in both)', 'matched'], default='different')

BVC_GROUPS = ['matched', 'different', 'not evaluable (in both)', 'only in positions', 'only in comparison_positions']
bvc_summary = (book_vs_custodian['group'].value_counts().reindex(BVC_GROUPS, fill_value=0)
               .rename('keys').to_frame())
bvc_summary['share of union'] = (bvc_summary['keys'] / len(book_vs_custodian)).round(6)
comparable = bvc_summary.loc[['matched', 'different'], 'keys'].sum()
print(f"keys in union: {len(book_vs_custodian):,} | rate A (matched / comparable): "
      f"{bvc_summary.loc['matched', 'keys'] / comparable:.6f} | rate B (matched / union): "
      f"{bvc_summary.loc['matched', 'keys'] / len(book_vs_custodian):.6f}")
differing = book_vs_custodian.loc[book_vs_custodian['group'] == 'different', 'book_minus_custodian']
print('book - custodian on differing keys: min / median / max =',
      f'{differing.min():,.2f} / {differing.median():,.2f} / {differing.max():,.2f}' if len(differing) else 'none')

In [ ]:
# CHECK: the book-vs-custodian groups add up to the union, and the keys in both add up to matched + different + not evaluable.
bvc_counts = book_vs_custodian['group'].value_counts()
in_both = int((book_vs_custodian['side'] == 'both').sum())
assert int(bvc_counts.sum()) == len(book_vs_custodian), 'a key has no group'
assert (bvc_counts.get('matched', 0) + bvc_counts.get('different', 0)
        + bvc_counts.get('not evaluable (in both)', 0)) == in_both, 'keys in both do not add up'
assert len(book_vs_custodian) == len(REPORTED['positions']) + len(REPORTED['comparison_positions']) - in_both
cross_file = NB01_OUTPUTS / '01_cross_check.csv'
if cross_file.is_file() and same_data:
    nb01_cross = pd.read_csv(cross_file).set_index('check')['this notebook']
    expected = {'merge: in both': in_both,
                'merge: only in positions': int(bvc_counts.get('only in positions', 0)),
                'merge: only in comparison_positions': int(bvc_counts.get('only in comparison_positions', 0))}
    for check, ours in expected.items():
        assert int(float(nb01_cross[check])) == ours, f'{check}: notebook 01 {nb01_cross[check]}, here {ours}'
    print('join counts equal notebook 01:', expected)
bvc_summary

## 4. Three sources: who disagrees? (layer 1 of the break classification)

Each key now has up to three numbers: our **rebuild** (main chain), the **book** (`positions`) and the **custodian** (`comparison_positions`). Layer 1 gives every key exactly one pattern:

- all three agree, or one source is the odd one out (book, custodian or rebuild), or all three differ;
- a one-sided pattern when a source has no record (missing is reported as missing, never as zero);
- *not evaluable* when the rebuild cannot be trusted for that key (conflicting duplicate, blank quantity, or another unreadable input), or when the book or the custodian has the key but no readable quantity (an unknown is never counted as a difference).

When the book and the custodian agree with each other and only the rebuild differs, the custodian is **not** independent evidence against the book: both may share the same convention, or our reading may be wrong. Those keys stay *unresolved*.

In [ ]:
# Why a rebuilt key is not evaluable, from the rows behind it (only causes the main chain actually marks).
CAUSE_NAMES = {'conflict': 'conflicting duplicate', 'blank': 'blank quantity', 'negative': 'negative quantity',
               'no_number_other': 'unreadable quantity or type', 'date_unknown': 'unreadable date',
               'orphan_reversal': 'REVERSED row with nothing to offset'}


def key_cause_flags(rows):
    """Per key: which causes the chain marks, from the rows of the chain that rebuilt it (main or side view)."""
    flags = rows.assign(
        conflict=(rows['pair_type'] == 'real conflict') & (POLICY['conflicting_pairs'] == 'not_evaluable'),
        blank=rows['quantity_blank'],
        negative=(rows['quantity_number'] < 0) & (POLICY['negatives'] == 'not_evaluable'),
        no_number_other=rows['quantity_used'].isna() & ~rows['quantity_blank'],
        date_unknown=rows[f"{POLICY['holding_date']}_day"].isna(),
        orphan_reversal=rows['orphan_reversal'].fillna(False) if 'orphan_reversal' in rows else False)   # R3 only
    return flags.groupby(KEY, as_index=False)[list(CAUSE_NAMES)].any()


key_causes = key_cause_flags(MAIN_ROWS)

three_way = (MAIN_REBUILT[KEY + ['rebuilt_quantity', 'contributing_rows', 'not_evaluable']]
             .assign(in_rebuild=True)
             .merge(book[KEY + ['book_quantity', 'book_transaction_count']].assign(in_book=True), on=KEY, how='outer')
             .merge(custodian[KEY + ['custodian_quantity']].assign(in_custodian=True), on=KEY, how='outer')
             .merge(key_causes, on=KEY, how='left'))
for column in ['in_rebuild', 'in_book', 'in_custodian', 'not_evaluable', *CAUSE_NAMES]:
    three_way[column] = three_way[column].astype('boolean').fillna(False).astype(bool)
log_rows('three-way union (rebuild, book, custodian)', len(MAIN_REBUILT) + len(book) + len(custodian), len(three_way),
         'full outer joins on account + security; rows out = keys in the union')

def layer1_patterns(frame):
    """One pattern per key from the three quantities, which sources have the key, and evaluability. A key is not
    evaluable when the rebuild cannot be trusted OR a source that has the key gives no readable quantity: an
    unknown is never counted as a difference."""
    r, b, c = frame['rebuilt_quantity'], frame['book_quantity'], frame['custodian_quantity']
    R, B, C = frame['in_rebuild'], frame['in_book'], frame['in_custodian']
    rebuild_unknown = R & frame['not_evaluable']
    book_unreadable, custodian_unreadable = B & b.isna(), C & c.isna()
    not_evaluable = rebuild_unknown | book_unreadable | custodian_unreadable
    evaluable = R & ~frame['not_evaluable']
    rb, rc, bc = (r - b).abs() <= TOLERANCE, (r - c).abs() <= TOLERANCE, (b - c).abs() <= TOLERANCE
    all_three = evaluable & B & C
    reasons = pd.DataFrame({name: frame[column] & rebuild_unknown for column, name in CAUSE_NAMES.items()})
    reasons['unreadable book quantity'] = book_unreadable
    reasons['unreadable custodian quantity'] = custodian_unreadable
    ne_reason = pd.Series('', index=frame.index)
    if not_evaluable.any():
        ne_reason[not_evaluable] = reasons[not_evaluable].apply(
            lambda row: ' + '.join(name for name, hit in row.items() if hit) or 'other', axis=1)
    pattern = np.select(
        [not_evaluable,
         all_three & rb & rc & bc, all_three & rc & ~rb, all_three & rb & ~rc, all_three & bc & ~rb & ~rc,
         all_three & ~rb & ~rc & ~bc, all_three,
         ~R & B & C, evaluable & B & ~C, evaluable & ~B & C,
         evaluable & ~B & ~C & (r.abs() <= TOLERANCE), evaluable & ~B & ~C, ~R & B & ~C, ~R & ~B & C],
        ['not evaluable: ' + ne_reason,
         'all three agree', 'book is the odd one out', 'custodian is the odd one out', 'rebuild is the odd one out',
         'all three differ', 'tolerance edge case',
         'book + custodian, no trades', 'rebuild + book, no custodian', 'rebuild + custodian, no book',
         'rebuild only, nets to zero', 'rebuild only, nonzero', 'book only', 'custodian only'],
        default='')
    two_present_agree = np.select([~R & B & C, evaluable & B & ~C, evaluable & ~B & C], [bc, rb, rc], default=False)
    return pattern, two_present_agree


three_way['pattern'], three_way['two_present_agree'] = layer1_patterns(three_way)

pattern_summary = three_way['pattern'].value_counts().rename('keys').to_frame()
pattern_summary['share of union'] = (pattern_summary['keys'] / len(three_way)).round(6)
pattern_summary['of which the two present agree'] = (three_way[three_way['pattern'].isin(
    ['book + custodian, no trades', 'rebuild + book, no custodian', 'rebuild + custodian, no book'])]
    .groupby('pattern')['two_present_agree'].sum().reindex(pattern_summary.index).astype('Int64'))

three_way_summary = pd.concat([
    bvc_summary.reset_index().rename(columns={'group': 'group or pattern'}).assign(comparison='book vs custodian'),
    pattern_summary.reset_index().rename(columns={'pattern': 'group or pattern'}).assign(
        comparison='rebuild vs book vs custodian')], ignore_index=True)
three_way_summary.to_csv(OUTPUTS / '02_three_way_summary.csv', index=False, lineterminator='\n')

In [ ]:
# CHECK: every key in the union of the three sources has exactly one pattern.
rebuild_keys = set(map(tuple, MAIN_REBUILT[KEY].to_numpy()))
book_keys = set(map(tuple, book[KEY].to_numpy()))
custodian_keys = set(map(tuple, custodian[KEY].to_numpy()))
assert len(three_way) == len(rebuild_keys | book_keys | custodian_keys), 'the union is the wrong size'
assert not three_way.duplicated(KEY).any(), 'a key appears twice'
assert (three_way['pattern'] != '').all() and three_way['pattern'].notna().all(), 'a key has no pattern'
assert int(pattern_summary['keys'].sum()) == len(three_way), 'patterns do not add up to the union'
print(f'keys in the union of rebuild, book and custodian: {len(three_way):,}')
pattern_summary

## 5. Why do they disagree? (layer 2: cause rules, one main class per key)

For every key where the sources disagree, the cause rules from `config/reconciliation.toml` are tested **in order** on the difference between the odd source and the rebuild; the first rule that fits sets the key's main class. A key that no rule explains is **unexplained** and stays visible. Each rule is a reproducible test, not a judgment:

| Rule | The difference equals... |
|---|---|
| book_counts_duplicate_copies | the signed quantity of the duplicate copies our chain removed (the source counts both copies) |
| book_double_offsets_reversed | what an R4 rebuild adds: every REVERSED row subtracted (a double offset) |
| duplicates_and_reversed | both of the above together |
| custodian_settlement_date_basis | the custodian quantity equals our settlement-date rebuild |
| sign_flipped_position | minus twice the rebuild: the source holds the opposite sign to the trades' net (added after the first run, when it showed up in the unexplained group; written as a rule and counted like the others) |

Keys that agree, one-sided keys and not-evaluable keys keep their layer-1 pattern as their class. **Flags** sit on top and never change the class: `sign_conflict` (a negative quantity on the key, kept raw; every such key is flagged, as the plan says, and the trace and the register report whether `gross_amount` fits the absolute value, which confirms the size but not the side; section 6 looks for the side in other records of the same trade) and `rebook_old_gross` (a rebook whose `gross_amount` still fits the cancelled quantity).

In [ ]:
# Each cause rule compares with the main chain where ONE convention is switched on.
def chain_variant(drop=(), swap=None):
    """The main chain without the steps in `drop`, and with `swap` = (code, function) replacing one step."""
    return rebuild(chain_rows(drop, swap))


def chain_rows(drop=(), swap=None):
    """The rows of that chain, before they are summed per key."""
    steps = []
    for code, _, function in CHAIN:
        if code in drop:
            continue
        steps.append(swap[1] if swap and code == swap[0] else function)
    return apply_steps(trades, steps)


def delta_against_main(alternative, name):
    """Per key: alternative rebuild minus main rebuild (missing in the alternative -> missing, never zero)."""
    merged = MAIN_REBUILT[KEY + ['rebuilt_quantity']].merge(
        alternative[KEY + ['rebuilt_quantity']], on=KEY, how='left', suffixes=('', '_alternative'))
    merged[name] = merged['rebuilt_quantity_alternative'] - merged['rebuilt_quantity']
    return merged[KEY + [name]]


SETTLEMENT_REBUILT = OTHER_REBUILT if OTHER_DATE == 'settlement' else MAIN_REBUILT
alternatives = [
    delta_against_main(chain_variant(drop=['C1']), 'duplicate_delta'),
    delta_against_main(chain_variant(swap=('C4', reversed_R4_evidence_only)), 'reversed_delta'),
    delta_against_main(chain_variant(drop=['C1'], swap=('C4', reversed_R4_evidence_only)), 'both_delta'),
    SETTLEMENT_REBUILT[KEY + ['rebuilt_quantity', 'not_evaluable']].rename(
        columns={'rebuilt_quantity': 'settlement_quantity', 'not_evaluable': 'settlement_not_evaluable'})]
for frame in alternatives:
    rows_in = len(three_way)
    three_way = three_way.drop(columns=[c for c in frame.columns if c not in KEY and c in three_way.columns]).merge(
        frame, on=KEY, how='left', validate='one_to_one')
    assert len(three_way) == rows_in, 'a cause-rule join changed the number of keys'


def fits(difference, explained):
    """The difference equals the explained amount, and that amount is not zero."""
    return ((difference - explained).abs() <= TOLERANCE) & (explained.abs() > TOLERANCE)


RULE_TESTS = {
    'book_counts_duplicate_copies': lambda diff, source: fits(diff, three_way['duplicate_delta']),
    'book_double_offsets_reversed': lambda diff, source: fits(diff, three_way['reversed_delta']),
    'duplicates_and_reversed': lambda diff, source: (fits(diff, three_way['both_delta'])
                                                     & (three_way['duplicate_delta'].abs() > TOLERANCE)
                                                     & (three_way['reversed_delta'].abs() > TOLERANCE)),
    'custodian_settlement_date_basis': lambda diff, source: (
        (source == 'custodian') & ~three_way['settlement_not_evaluable'].astype('boolean').fillna(True).astype(bool)
        & ((three_way['custodian_quantity'] - three_way['settlement_quantity']).abs() <= TOLERANCE)),
    'sign_flipped_position': lambda diff, source: fits(diff, -2 * three_way['rebuilt_quantity']),
}
CAUSE_LABELS = {'book_counts_duplicate_copies': 'counts duplicate copies',
                'book_double_offsets_reversed': 'double-offsets REVERSED',
                'duplicates_and_reversed': 'duplicate copies + REVERSED double offset',
                'custodian_settlement_date_basis': 'custodian on settlement-date basis',
                'sign_flipped_position': 'opposite sign to the trades',
                'unexplained': 'unexplained'}
unknown_rules = set(CONFIG['classification']['cause_rules']) - set(RULE_TESTS)
assert not unknown_rules, f'config names cause rules that do not exist: {unknown_rules}'

r, b, c = three_way['rebuilt_quantity'], three_way['book_quantity'], three_way['custodian_quantity']
ODD_SOURCES = {'rebuild is the odd one out': [('book', b - r)],          # book = custodian here
               'book is the odd one out': [('book', b - r)],
               'custodian is the odd one out': [('custodian', c - r)],
               'all three differ': [('book', b - r), ('custodian', c - r)]}
cause = pd.Series('n/a', index=three_way.index)
for pattern, sources in ODD_SOURCES.items():
    open_keys = three_way['pattern'] == pattern
    for rule in CONFIG['classification']['cause_rules']:          # first rule that fits wins
        hit = pd.Series(False, index=three_way.index)
        for source, difference in sources:
            hit = hit | RULE_TESTS[rule](difference, source)
        cause[open_keys & hit] = rule
        open_keys = open_keys & ~hit
    cause[open_keys] = 'unexplained'
three_way['cause'] = cause

BREAK_KIND = {'all three agree': 'no', 'rebuild only, nets to zero': 'cleared-flag candidate',
              'book + custodian, no trades': 'one-sided', 'rebuild + book, no custodian': 'one-sided',
              'rebuild + custodian, no book': 'one-sided', 'rebuild only, nonzero': 'one-sided',
              'book only': 'one-sided', 'custodian only': 'one-sided'}
three_way['is a break'] = np.where(three_way['pattern'].str.startswith('not evaluable'), 'not evaluable',
                                   three_way['pattern'].map(BREAK_KIND).fillna('yes'))
three_way['main_class'] = np.where(three_way['cause'] == 'n/a', three_way['pattern'],
                                   three_way['pattern'] + ': ' + three_way['cause'].map(CAUSE_LABELS))

In [ ]:
# Flags: they sit on top of the class and never change it.
negative_rows = MAIN_ROWS[MAIN_ROWS['quantity_number'] < 0]
negative_rows = negative_rows.assign(gross_fits_absolute=(
    negative_rows['gross_number'] - negative_rows['quantity_number'].abs() * negative_rows['price_number']).abs()
    <= GROSS_TOLERANCE)
# The flag covers every key with a negative-quantity row of any status (D-008, D-032, D-048); negative_rows (main chain
# only) gives the sizes, because only those rows move the rebuilt quantity.
sign_conflict = (trades[trades['quantity_number'] < 0].groupby(KEY, as_index=False)
                 .agg(sign_conflict_rows=('transaction_id', 'size')))
print(f'negative-quantity rows in the main chain: {len(negative_rows):,} | types: '
      f'{negative_rows["transaction_type"].value_counts().to_dict()} | gross_amount fits the absolute value: '
      f'{int(negative_rows["gross_fits_absolute"].sum()):,}')

# The quantity each CANCELLED ID was booked with. An ID delivered with two different quantities has no one quantity to
# compare a rebook with: blank (not evaluable, D-048), never whichever copy comes first in the file. A CANCELLED row
# with a blank transaction_id can be no row's original: every unlinked trade has a blank related_transaction_id.
cancelled = (trades[(trades['status'] == 'CANCELLED') & (trades['transaction_id'] != '')]
             .groupby('transaction_id')['quantity_number']
             .agg(lambda q: q.iloc[0] if q.nunique(dropna=False) == 1 else np.nan))
rebooks = MAIN_ROWS[MAIN_ROWS['related_transaction_id'].isin(cancelled.index)].copy()
rebooks['cancelled_quantity'] = rebooks['related_transaction_id'].map(cancelled)


def rebook_tests(rows):
    """For rebooks with cancelled_quantity: evaluable = both quantities readable; new = evaluable and different;
    old_gross = new, and gross_amount fits the cancelled quantity but not its own. A blank on either side is not
    evaluable, never a new quantity (D-048)."""
    evaluable = rows['quantity_number'].notna() & rows['cancelled_quantity'].notna()
    new = evaluable & (rows['quantity_number'] != rows['cancelled_quantity'])
    fits_old = (rows['gross_number'] - rows['cancelled_quantity'].abs() * rows['price_number']).abs() <= GROSS_TOLERANCE
    fits_own = (rows['gross_number'] - rows['quantity_number'].abs() * rows['price_number']).abs() <= GROSS_TOLERANCE
    return evaluable, new, new & fits_old & ~fits_own


rebook_evaluable, rebook_new, rebook_old = rebook_tests(rebooks)
REBOOK_OLD_GROSS_ROWS = rebooks[rebook_old]
rebook_old_gross = REBOOK_OLD_GROSS_ROWS.groupby(KEY, as_index=False).agg(rebook_old_gross_rows=('transaction_id', 'size'))
print(f'rebooks in the main chain: {len(rebooks):,} | with a new quantity: {int(rebook_new.sum()):,} | blank on either '
      f'side: {int((~rebook_evaluable).sum()):,} | gross_amount fits the old quantity, not its own: '
      f'{int(rebook_old.sum()):,}')

for flags in [sign_conflict, rebook_old_gross]:
    three_way = three_way.drop(columns=[c for c in flags.columns if c not in KEY and c in three_way.columns]).merge(
        flags, on=KEY, how='left', validate='one_to_one')
three_way['sign_conflict'] = three_way['sign_conflict_rows'].fillna(0) > 0
three_way['rebook_old_gross'] = three_way['rebook_old_gross_rows'].fillna(0) > 0

class_summary = (three_way.groupby(['is a break', 'pattern', 'cause', 'main_class'], as_index=False)
                 .agg(keys=('account_id', 'size'), sign_conflict_keys=('sign_conflict', 'sum'),
                      rebook_old_gross_keys=('rebook_old_gross', 'sum'))
                 .sort_values('keys', ascending=False))
class_summary['share of union'] = (class_summary['keys'] / len(three_way)).round(6)
class_summary.to_csv(OUTPUTS / '02_break_classes.csv', index=False, lineterminator='\n')

In [ ]:
# Side view: the keys with a REVERSED row, rebuilt under the other REVERSED reading (R3: one offset per original).
reversal_keys = trades.loc[trades['status'] == 'REVERSED', KEY].drop_duplicates()
if POLICY['reversed_side'] != POLICY['reversed_main']:
    side_rows = chain_rows(swap=('C4', REVERSED_STEPS[POLICY['reversed_side']]))
    side_rebuilt = rebuild(side_rows)
    # The side reading's own causes: a key it cannot rebuild is named from its rows, not from the main chain's.
    side = three_way.merge(reversal_keys, on=KEY).drop(columns=['rebuilt_quantity', 'not_evaluable', 'in_rebuild',
                                                                *CAUSE_NAMES])
    side = (side.merge(side_rebuilt[KEY + ['rebuilt_quantity', 'not_evaluable']], on=KEY, how='left')
            .merge(key_cause_flags(side_rows), on=KEY, how='left'))
    side[list(CAUSE_NAMES)] = side[list(CAUSE_NAMES)].astype('boolean').fillna(False).astype(bool)
    side['in_rebuild'] = side['rebuilt_quantity'].notna() | side['not_evaluable'].notna()
    side['not_evaluable'] = side['not_evaluable'].astype('boolean').fillna(False).astype(bool)
    side['side_pattern'], _ = layer1_patterns(side)
    reversed_side_view = (side.groupby(['pattern', 'side_pattern']).size().rename('keys').reset_index()
                          .rename(columns={'pattern': f"main reading ({POLICY['reversed_main']})",
                                           'side_pattern': f"side reading ({POLICY['reversed_side']})"}))
    reversed_side_view.to_csv(OUTPUTS / '02_reversed_side_view.csv', index=False, lineterminator='\n')
    print(f'keys with a REVERSED row: {len(reversal_keys):,}')
    display(reversed_side_view)

In [ ]:
# CHECK: one main class per key; the classes add up to the union; the unexplained count is shown.
assert three_way['main_class'].notna().all() and (three_way['main_class'] != '').all(), 'a key has no class'
assert int(class_summary['keys'].sum()) == len(three_way), 'classes do not add up to the union'
per_pattern_layer2 = class_summary.groupby('pattern')['keys'].sum()
per_pattern_layer1 = three_way['pattern'].value_counts()
assert set(per_pattern_layer2.index) == set(per_pattern_layer1.index) and \
    (per_pattern_layer2 == per_pattern_layer1.reindex(per_pattern_layer2.index)).all(), \
    'layer 2 moved a key out of its layer-1 pattern'
# CHECK (made-up rows): a rebook with a blank quantity on either side is not evaluable, never a "new quantity" (D-048).
made_up_rebooks = pd.DataFrame({'quantity_number': [np.nan, 5.0, 10.0, 5.0], 'cancelled_quantity': [10.0, 10.0, 10.0, np.nan],
                                'gross_number': [20.0, 20.0, 20.0, 20.0], 'price_number': [2.0, 2.0, 2.0, 2.0]})
evaluable, new, old_gross = rebook_tests(made_up_rebooks)
assert list(evaluable) == [False, True, True, False] and list(new) == [False, True, False, False], (evaluable, new)
assert list(old_gross) == [False, True, False, False], old_gross
# CHECK: every key in the union that has a negative-quantity row, of any status, carries sign_conflict (D-008, D-048).
negative_trade_keys = set(map(tuple, trades.loc[trades['quantity_number'] < 0, KEY].to_numpy()))
union_keys = set(map(tuple, three_way[KEY].to_numpy()))
assert set(map(tuple, three_way.loc[three_way['sign_conflict'], KEY].to_numpy())) == negative_trade_keys & union_keys, \
    'a key with a negative-quantity row is not flagged sign_conflict, or a key without one is'
unexplained = int(class_summary.loc[class_summary['cause'] == 'unexplained', 'keys'].sum())
print(f'unexplained breaks: {unexplained:,} of {int(class_summary.loc[class_summary["is a break"] == "yes", "keys"].sum()):,} breaks')
class_summary

## 6. One evidenced trace: a negative-quantity trade

The record is **chosen by a rule** from `config/reconciliation.toml`, never typed in: among keys flagged `sign_conflict` that are present and evaluable in all three sources and carry no other flag, take the first by account, security and transaction ID. We then follow it hop by hop (Working Guide §4) and mark each hop **observed** (read directly from the data) or **inferred** (our reasoning or an assumption).

First, one question the trace depends on. `gross_amount / price` gives the size of a trade, but does it give the side? We count the signs of `quantity` and `gross_amount` by transaction type (`02_sign_by_type.csv`). A negative BUY then has two readings: **wrong sign** (it should be BUY +q; the book is off by 2q) or **wrong type** (it should be SELL q; the book is right). Other records of the same trade can tell them apart (rule `TXN_NEGATIVE_SIDE_EVIDENCE`, `02_negative_partners.csv`): a copy with the same `transaction_id` and another quantity, rows whose `related_transaction_id` points at it, and the row it points at. A partner of the same type with a positive quantity supports **wrong sign**; one of the opposite type supports **wrong type**.

In [ ]:
# Which reading fits a negative-quantity row: wrong sign or wrong type? (TXN_NEGATIVE_SIDE_EVIDENCE in docs/RULES.md)
# Run on every trade row, the population of the register line. No threshold: partners are matched on exact IDs.
OPPOSITE_TYPE = {'BUY': 'SELL', 'SELL': 'BUY'}


def sign_label(numbers):
    """negative / zero / positive, or 'blank or unreadable' where the text did not parse."""
    return pd.Series(np.select([numbers < 0, numbers == 0, numbers > 0], ['negative', 'zero', 'positive'],
                               'blank or unreadable'), index=numbers.index)


# 1. Does gross_amount carry a side? Signs of quantity and gross_amount by transaction type.
sign_by_type = (pd.concat([trades.assign(field=field, sign=sign_label(trades[column]))
                           .groupby(['transaction_type', 'field', 'sign'], as_index=False)
                           .agg(rows=('transaction_id', 'size'))
                           for field, column in [('quantity', 'quantity_number'), ('gross_amount', 'gross_number')]])
                .sort_values(['transaction_type', 'field', 'sign']).reset_index(drop=True))
sign_by_type.to_csv(OUTPUTS / '02_sign_by_type.csv', index=False, lineterminator='\n')
gross_signs = sign_by_type[(sign_by_type['field'] == 'gross_amount') & sign_by_type['transaction_type'].isin(OPPOSITE_TYPE)]
GROSS_HAS_NO_SIDE = (set(gross_signs['transaction_type']) == set(OPPOSITE_TYPE)) and set(gross_signs['sign']) == {'positive'}
print('signs by type:', {(t, f): dict(zip(g['sign'], g['rows'])) for (t, f), g in sign_by_type.groupby(['transaction_type', 'field'])})
print('gross_amount is positive on every BUY and SELL row (carries no side):', GROSS_HAS_NO_SIDE)

# 2. Other records of the same trade, for each negative-quantity row.
PARTNER_FIELDS = ['transaction_id', 'transaction_type', 'status', 'quantity', 'quantity_number', 'related_transaction_id']
PARTNER_COLUMNS = ['transaction_id', 'transaction_type', 'status', 'quantity', 'link', 'partner_transaction_id',
                   'partner_transaction_type', 'partner_status', 'partner_quantity', 'partner_quantity_sign', 'same_size',
                   'reading_supported']


def negative_partners(rows):
    """One line per (negative-quantity row, partner record): a row with the same transaction_id and another quantity
    (blank counts as another), a row whose related_transaction_id points at it, or the row it points at. Copies of
    one record count once. A partner of the same type with a positive quantity supports 'wrong sign'; one of the
    opposite type with a positive quantity supports 'wrong type'; anything else 'neither'."""
    negatives = rows.loc[rows['quantity_number'] < 0, PARTNER_FIELDS].drop_duplicates(['transaction_id', 'quantity'])
    others = rows[PARTNER_FIELDS].rename(columns=lambda c: f'partner_{c}')
    same_id = negatives.merge(others, left_on='transaction_id', right_on='partner_transaction_id')
    same_id = same_id[same_id['partner_quantity'] != same_id['quantity']].assign(link='same transaction_id, different quantity')
    points_at_it = negatives.merge(others, left_on='transaction_id', right_on='partner_related_transaction_id')
    points_at_it = points_at_it[points_at_it['partner_transaction_id'] != points_at_it['transaction_id']].assign(link='points at it')
    it_points_at = negatives[negatives['related_transaction_id'] != ''].merge(
        others, left_on='related_transaction_id', right_on='partner_transaction_id')
    it_points_at = it_points_at[it_points_at['partner_transaction_id'] != it_points_at['transaction_id']].assign(link='it points at')
    links = pd.concat([same_id, points_at_it, it_points_at], ignore_index=True)
    positive = links['partner_quantity_number'] > 0
    links = links.assign(
        partner_quantity_sign=sign_label(links['partner_quantity_number']),
        same_size=(links['partner_quantity_number'].abs() - links['quantity_number'].abs()).abs() <= TOLERANCE,
        reading_supported=np.select(
            [positive & (links['partner_transaction_type'] == links['transaction_type']),
             positive & (links['partner_transaction_type'] == links['transaction_type'].map(OPPOSITE_TYPE))],
            ['wrong sign', 'wrong type'], 'neither'))
    return links[PARTNER_COLUMNS].drop_duplicates().sort_values(PARTNER_COLUMNS).reset_index(drop=True)


negative_partner_links = negative_partners(trades)
negative_partner_links.to_csv(OUTPUTS / '02_negative_partners.csv', index=False, lineterminator='\n')
negative_transaction_ids = trades.loc[trades['quantity_number'] < 0, 'transaction_id'].nunique()
readings_per_negative = (negative_partner_links.groupby('transaction_id')['reading_supported']
                         .agg(lambda s: ' and '.join(sorted(set(s)))))
reading_examples = readings_per_negative.reset_index().groupby('reading_supported')['transaction_id'].agg(['size', 'min'])
PARTNER_SUMMARY = (f'other records of the same trade (TXN_NEGATIVE_SIDE_EVIDENCE, section 6): {len(readings_per_negative)} '
                   f'of {negative_transaction_ids} negative transaction_ids have one'
                   + ''.join(f'; partners support {reading} on {row["size"]} (example {row["min"]})'
                             for reading, row in reading_examples.iterrows())
                   + '; wrong sign = a partner of the same type with a positive quantity, wrong type = one of the opposite type')
print(PARTNER_SUMMARY)

# 3. The custodian on positions with a negative-quantity trade: does it hold the book's value or the wrong-sign
# value (the rebuild with the negative rows' sign flipped)? And does it copy the book's other conventions?
def wrong_sign_quantity(rebuilt, negative_used):
    """The holding if the negative rows had the opposite sign: each moves the holding by -2 x what the rebuild used
    (a negative BUY of -q adds 2q; a negative SELL of -q, which the rebuild counts as +q, takes 2q away)."""
    return rebuilt - 2 * negative_used


# Keys with a negative row in the main chain (the flag also covers rows of other statuses, which move nothing).
negative_used = negative_rows.groupby(KEY, as_index=False).agg(negative_used=('quantity_used', 'sum'))
negative_keys = three_way[three_way['sign_conflict']].merge(negative_used, on=KEY, how='inner', validate='one_to_one')
negative_keys = negative_keys[negative_keys[['rebuilt_quantity', 'book_quantity', 'custodian_quantity']].notna().all(axis=1)
                              & ~negative_keys['not_evaluable'].fillna(True).astype(bool)
                              & ~negative_keys['pattern'].str.startswith('not evaluable')]
wrong_sign_value = wrong_sign_quantity(negative_keys['rebuilt_quantity'], negative_keys['negative_used'])
CUSTODIAN_ON_NEGATIVE_KEYS = {
    'keys': len(negative_keys),
    'equals the book': int(((negative_keys['custodian_quantity'] - negative_keys['book_quantity']).abs() <= TOLERANCE).sum()),
    'equals the wrong-sign value': int(((negative_keys['custodian_quantity'] - wrong_sign_value).abs() <= TOLERANCE).sum())}
CUSTODIAN_COPIES_BOOK = int((three_way['pattern'].eq('rebuild is the odd one out') & three_way['cause'].isin(
    ['book_counts_duplicate_copies', 'book_double_offsets_reversed', 'duplicates_and_reversed'])).sum())
print(f"positions with a negative-quantity trade, evaluable in all three sources: {CUSTODIAN_ON_NEGATIVE_KEYS['keys']:,} | "
      f"custodian equals the book: {CUSTODIAN_ON_NEGATIVE_KEYS['equals the book']:,} | equals the wrong-sign value: "
      f"{CUSTODIAN_ON_NEGATIVE_KEYS['equals the wrong-sign value']:,} | positions where book and custodian share the "
      f"duplicate or REVERSED convention: {CUSTODIAN_COPIES_BOOK:,}")
negative_partner_links


In [ ]:
# CHECK: the partner search on small made-up rows: each link kind, a partner delivered twice counted once, a copy with
# the same quantity is not a partner, a partner of the opposite type supports the wrong-type reading.
made_up = pd.DataFrame([('N1', 'BUY', 'BOOKED', '-10', ''),
                        ('N1', 'BUY', 'BOOKED', '-10', ''),        # exact copy of the negative row: not a partner
                        ('N1', 'BUY', 'BOOKED', '10', ''),         # same ID, another quantity
                        ('R1', 'BUY', 'REVERSED', '10', 'N1'),     # points at it ...
                        ('R1', 'BUY', 'REVERSED', '10', 'N1'),     # ... delivered twice
                        ('N2', 'BUY', 'REVERSED', '-5', 'O2'),     # points at O2
                        ('O2', 'SELL', 'BOOKED', '5', ''),
                        ('N3', 'BUY', 'BOOKED', '-7', '')],        # no partner
                       columns=['transaction_id', 'transaction_type', 'status', 'quantity', 'related_transaction_id'])
made_up['quantity_number'] = pd.to_numeric(made_up['quantity'])
made_up_links = negative_partners(made_up)
found = sorted(zip(made_up_links['transaction_id'], made_up_links['link'], made_up_links['partner_transaction_id'],
                   made_up_links['reading_supported']))
assert found == [('N1', 'points at it', 'R1', 'wrong sign'),
                 ('N1', 'same transaction_id, different quantity', 'N1', 'wrong sign'),
                 ('N2', 'it points at', 'O2', 'wrong type')], found
assert negative_partners(made_up[made_up['transaction_id'] == 'O2']).empty, 'a row with no negative quantity has partners'
# The wrong-sign holding: a negative BUY of -10 (signed -10) adds 20; a negative SELL of -10 (signed +10) takes 20 away.
assert wrong_sign_quantity(100.0, -10.0) == 120.0 and wrong_sign_quantity(100.0, 10.0) == 80.0
assert list(sign_label(pd.Series([-1.0, 0.0, 2.0, np.nan]))) == ['negative', 'zero', 'positive', 'blank or unreadable']


In [ ]:
def booked_before_cutoff(booked, cutoff):
    """True when the booked timestamp is at or before the cutoff; both normalised to YYYY-MM-DD first."""
    booked_text, cutoff_text = normalise_timestamp(pd.Series([booked, cutoff]))
    return booked_text <= cutoff_text


def fund_net_assets(ledger, fund_id, as_of_date):
    """Net assets of one fund, summed over its share classes (never sum nav_per_share), and the rows used."""
    rows = ledger[(ledger['fund_id'] == fund_id) & (normalise_date(ledger['as_of_date']) == as_of_date)]
    return pd.to_numeric(rows['net_assets'], errors='coerce').sum(), len(rows)


TRACE = CONFIG['trace']
FLAG_ROWS = {'sign_conflict': negative_rows,                                      # the rows behind each flag
             'rebook_old_gross': REBOOK_OLD_GROSS_ROWS}
OTHER_FLAGS = [flag for flag in FLAG_ROWS if flag != TRACE['flag']]
state = ['in_rebuild', 'in_book', 'in_custodian', 'not_evaluable', 'sign_conflict', 'rebook_old_gross',
         'pattern', 'main_class', 'rebuilt_quantity', 'book_quantity', 'custodian_quantity']
flagged = FLAG_ROWS[TRACE['flag']][KEY + ['transaction_id']].merge(three_way[KEY + state], on=KEY)
# Present in all three sources, and evaluable in all three (a not-evaluable pattern also covers an unreadable
# book or custodian quantity).
rule = (flagged[TRACE['flag']] & flagged['in_rebuild'] & flagged['in_book'] & flagged['in_custodian']
        & ~flagged['not_evaluable'] & ~flagged['pattern'].str.startswith('not evaluable'))
no_other_flag = ~flagged[OTHER_FLAGS].any(axis=1)
trace_candidates = flagged[rule & no_other_flag].sort_values(TRACE['sort_by']).reset_index(drop=True)
TRACE_RULE_USED = 'full rule'
if trace_candidates.empty:
    print('No key meets the full rule; dropping the "no other flag" condition, as the plan allows.')
    trace_candidates = flagged[rule].sort_values(TRACE['sort_by']).reset_index(drop=True)
    TRACE_RULE_USED = 'without "no other flag"'
TRACE_FOUND = not trace_candidates.empty
print(f"keys flagged {TRACE['flag']}: {flagged[KEY].drop_duplicates().shape[0]:,} | meeting the trace rule "
      f"({TRACE_RULE_USED}): {trace_candidates[KEY].drop_duplicates().shape[0]:,}")
if not TRACE_FOUND:
    print(f"NO TRACE: no key flagged {TRACE['flag']} is present and evaluable in all three sources, even without the "
          '"no other flag" condition. 02_trace.csv is written empty; change [trace] flag in the config or check the data.')

trace_lines = []
def add(hop, table, field, value, evidence, note=''):
    trace_lines.append({'hop': hop, 'table': table, 'field': field, 'value': value, 'evidence': evidence, 'note': note})

if TRACE_FOUND:
    chosen = trace_candidates.iloc[0]
    TXN_ID, ACCOUNT_ID, SECURITY_ID = chosen['transaction_id'], chosen['account_id'], chosen['security_id']

    # Hop 1, source: the transaction row as delivered (every raw field), and what its own amount says.
    source_rows = transactions[transactions['transaction_id'] == TXN_ID]
    for copy_number, (_, row) in enumerate(source_rows.iterrows()):
        for field, value in row.items():
            add('source', 'transactions', field if copy_number == 0 else f'{field} (copy {copy_number + 1})', value, 'observed')
    src = trades[trades['transaction_id'] == TXN_ID].iloc[0]
    implied = src['gross_number'] / src['price_number']
    gross_fits_absolute = abs(src['gross_number'] - abs(src['quantity_number']) * src['price_number']) <= GROSS_TOLERANCE
    size = abs(src['quantity_number'])
    other_type = OPPOSITE_TYPE.get(src['transaction_type'], 'the other type')
    add('source', 'transactions', 'gross_amount / price', round(implied, 6), 'observed',
        f"quantity is {src['quantity']} on a {src['transaction_type']}; the amount implies {implied:,.2f} units, so "
        'gross_amount confirms the size of the trade, not its side' +
        (': it is positive on every BUY and SELL row (02_sign_by_type.csv)' if GROSS_HAS_NO_SIDE else '') +
        f". Two readings fit: wrong sign ({src['transaction_type']} +{size:,.0f}; the book is off by 2 x {size:,.0f} "
        f"units) or wrong type ({other_type} {size:,.0f}; the book is right)" if gross_fits_absolute else
        f"quantity is {src['quantity']} on a {src['transaction_type']}; the amount implies {implied:,.2f} units, which "
        'fits neither sign, so gross_amount does not show which sign is right')
    own_partners = negative_partner_links[negative_partner_links['transaction_id'] == TXN_ID]
    add('source', 'transactions', 'other records of this trade', len(own_partners), 'observed',
        '; '.join(f'{p.link}: {p.partner_transaction_id} {p.partner_transaction_type} {p.partner_quantity} '
                  f'({p.partner_status}), supports {p.reading_supported}' for p in own_partners.itertuples())
        or 'none (same transaction_id with another quantity, a row linked to it, the row it links to): this record '
           'alone cannot show which reading is right')
    add('source', 'transactions', 'where it entered', src['source_system'], 'inferred',
        'the negative quantity is already in the row as delivered by this source system; nothing downstream changes it')

    # Hop 2, pipeline: the matching pipeline record, and whether negatives cluster in one source or batch.
    # pipeline_events is not one-to-one with transactions: the record shown is the first by its own fields, never by file order.
    pipe_rows = pipeline_events[pipeline_events['transaction_id'] == TXN_ID]
    pipe_rows = pipe_rows.sort_values(list(pipe_rows.columns), kind='mergesort')
    for field, value in (pipe_rows.iloc[0].items() if len(pipe_rows) else []):
        add('pipeline', 'pipeline_events', field, value, 'observed')
    add('pipeline', 'pipeline_events', 'pipeline records for this transaction_id', len(pipe_rows), 'observed',
        '' if len(pipe_rows) else 'no pipeline record: the path from capture to booking cannot be followed for this trade')
    if len(pipe_rows):
        add('pipeline', 'pipeline_events', 'booked before cutoff',
            bool(booked_before_cutoff(pipe_rows.iloc[0]['booked_timestamp'], pipe_rows.iloc[0]['cutoff_timestamp'])),
            'observed', 'timestamps compared in one platform clock')
    negatives_by_source = (negative_rows['source_system'].value_counts(normalize=True).round(4).to_dict())
    all_by_source = trades['source_system'].value_counts(normalize=True).round(4).to_dict()
    add('pipeline', 'transactions', 'share of negative quantities by source_system', negatives_by_source, 'observed',
        f'share of all rows by source_system: {all_by_source}')
    if len(pipe_rows):
        same_batch = pipeline_events['batch_id'] == pipe_rows.iloc[0]['batch_id']
        negatives_in_batch = int(pipeline_events.loc[same_batch, 'transaction_id'].isin(negative_rows['transaction_id']).sum())
        add('pipeline', 'pipeline_events', 'negative-quantity records in the same batch', negatives_in_batch, 'observed',
            f'out of {int(same_batch.sum()):,} records in that batch')

    # Hop 3, position: the book against our rebuild.
    book_row = positions[(positions['account_id'] == ACCOUNT_ID) & (positions['security_id'] == SECURITY_ID)
                         & (normalise_date(positions['as_of_date']) == AS_OF_DATE)].iloc[0]
    key_rows = MAIN_ROWS[(MAIN_ROWS['account_id'] == ACCOUNT_ID) & (MAIN_ROWS['security_id'] == SECURITY_ID)].sort_values(
        ['trade_day', 'transaction_id'])                                   # listed in date order, never file order
    for field in ['position_id', 'quantity', 'transaction_count', 'as_of_date', 'source']:
        add('position', 'positions', field, book_row[field], 'observed')
    add('position', 'rebuild (main chain)', 'rebuilt quantity', chosen['rebuilt_quantity'], 'observed',
        f"{len(key_rows)} contributing rows: " + ', '.join(f"{t} {q}" for t, q in zip(key_rows['transaction_type'], key_rows['quantity'])))
    book_equals_rebuild = bool(abs(chosen['book_quantity'] - chosen['rebuilt_quantity']) <= TOLERANCE)
    add('position', 'rebuild (main chain)', 'book equals the raw rebuild', book_equals_rebuild, 'observed',
        'the book carries the negative quantity as delivered, so the reconciliation matches and hides it'
        if book_equals_rebuild else 'the book differs from the raw rebuild on this key')
    # What the rebuild used for this ID's negative rows (as in section 6c), not the raw signed quantity: they differ
    # when the main chain offsets a REVERSED row (R3).
    used_rows = negative_rows.loc[negative_rows['transaction_id'] == TXN_ID, 'quantity_used']
    assert len(used_rows) == 1, f'{len(used_rows)} main-chain rows for the traced ID: the wrong-sign value would be ambiguous'
    used = used_rows.iloc[0]
    corrected = wrong_sign_quantity(chosen['rebuilt_quantity'], used)
    add('position', 'rebuild (absolute-value sensitivity)', 'quantity under the wrong-sign reading', corrected, 'inferred',
        f"if the row should be {src['transaction_type']} +{size:,.0f}: differs from the book by 2 x {size:,.0f} units")
    add('position', 'rebuild (main chain)', 'quantity under the wrong-type reading', chosen['rebuilt_quantity'], 'inferred',
        f"if the row should be {other_type} {size:,.0f}: it moves the holding the same way as the negative "
        f"{src['transaction_type']}, so the book is right")

    # Hop 4, custodian: does the second source catch it?
    custodian_row = comparison_positions[(comparison_positions['account_id'] == ACCOUNT_ID)
                                         & (comparison_positions['security_id'] == SECURITY_ID)
                                         & (normalise_date(comparison_positions['as_of_date']) == AS_OF_DATE)].iloc[0]
    for field in ['comparison_id', 'quantity', 'as_of_date', 'source']:
        add('custodian', 'comparison_positions', field, custodian_row[field], 'observed')
    shares_error = bool(abs(chosen['custodian_quantity'] - chosen['book_quantity']) <= TOLERANCE)
    add('custodian', 'comparison_positions', 'custodian equals the book', shares_error, 'observed',
        'if the sign is wrong, the custodian shares the error, so a book-vs-custodian check cannot catch it' if shares_error
        else 'the custodian differs from the book: independent evidence on this key')
    add('custodian', 'comparison_positions', 'positions with a negative-quantity trade: custodian equals the book',
        f"{CUSTODIAN_ON_NEGATIVE_KEYS['equals the book']} of {CUSTODIAN_ON_NEGATIVE_KEYS['keys']}", 'observed',
        f"evaluable in all three sources; the custodian holds the wrong-sign value on "
        f"{CUSTODIAN_ON_NEGATIVE_KEYS['equals the wrong-sign value']}. It also shares the book's duplicate or REVERSED "
        f"convention on {CUSTODIAN_COPIES_BOOK:,} positions (02_break_classes.csv), so its agreement is not independent "
        'evidence for either reading')

    # Hop 5, downstream: the fund the account feeds, and an indicative exposure (labelled; not a loss).
    account = accounts[accounts['account_id'] == ACCOUNT_ID]
    fund_total, fund_rows = np.nan, 0
    if len(account) == 1:
        account = account.iloc[0]
        for field in ['fund_id', 'share_class', 'account_type', 'base_currency', 'status']:
            add('downstream', 'accounts', field, account[field], 'observed')
        fund_total, fund_rows = fund_net_assets(ledger_nav, account['fund_id'], AS_OF_DATE)
        add('downstream', 'ledger_nav', 'fund net_assets (all share classes, USD)', round(fund_total, 2), 'observed',
            f"{fund_rows} share-class rows; data_dictionary: net_assets is USD for the exercise")
    else:
        add('downstream', 'accounts', 'account_id', 'not in accounts' if account.empty else f'{len(account)} rows in accounts',
            'observed', 'the fund this account feeds cannot be named, so the exposure is not stated in bps of fund net assets')
    security = securities[securities['security_id'] == SECURITY_ID]
    asset_class = security.iloc[0]['asset_class'] if len(security) else 'not in securities'
    add('downstream', 'securities', 'asset_class', asset_class, 'observed')
    quotes = prices[(prices['security_id'] == SECURITY_ID) & (normalise_date(prices['price_date']) == AS_OF_DATE)].copy()
    quotes['price_number'] = pd.to_numeric(quotes['price'], errors='coerce')
    quotes['priority'] = pd.to_numeric(quotes['vendor_priority'], errors='coerce')
    # Lowest priority first; a tie is broken by the quote's own fields, never by file order.
    quotes = quotes[quotes['price_number'] > 0].sort_values(['priority', *prices.columns], kind='mergesort')
    units_misstated = 2 * abs(src['quantity_number'])
    add('downstream', 'rebuild', 'units misstated in the book', units_misstated, 'inferred',
        'under the wrong-sign reading (the book holds -X where +X is expected); 0 under the wrong-type reading')
    if len(quotes):
        quote = quotes.iloc[0]
        add('downstream', 'prices + fx_rates', 'price used', f"{quote['price']} {quote['currency']} ({quote['vendor']}, "
            f"priority {quote['vendor_priority']})", 'inferred', 'lowest vendor_priority with a positive price on the snapshot date')
        # Same convention as FX_RATE_COVERAGE (section 10): the reporting currency needs no rate; any other currency
        # needs exactly one usable rate (readable and > 0) on the snapshot date.
        if quote['currency'] == REPORTING_CURRENCY:
            rate, rate_note = 1.0, f'{REPORTING_CURRENCY} is the reporting currency: no rate applied'
        else:
            rates = pd.to_numeric(fx_rates.loc[(fx_rates['currency'] == quote['currency'])
                                               & (normalise_date(fx_rates['rate_date']) == AS_OF_DATE), 'rate_to_usd'],
                                  errors='coerce')
            rates = rates[rates > 0].unique()
            rate = float(rates[0]) if len(rates) == 1 else np.nan
            rate_note = ('USD per one unit of the price currency' if len(rates) == 1 else
                         'no usable rate on the snapshot date' if not len(rates) else
                         f'{len(rates)} different rates on the snapshot date: none chosen')
        add('downstream', 'fx_rates', 'rate_to_usd used', rate if pd.notna(rate) else 'none', 'inferred', rate_note)
        if pd.notna(rate):
            exposure_usd = units_misstated * quote['price_number'] * rate
            add('downstream', 'derived', 'indicative exposure (USD)', round(exposure_usd, 2), 'inferred',
                'exposure, not a loss: units misstated x price x FX, under the wrong-sign reading (0 under the '
                'wrong-type reading); quote basis assumed per unit' +
                ('; BOND: percent-of-par basis not verified' if asset_class == 'BOND' else ''))
            if fund_total > 0:
                add('downstream', 'derived', 'indicative exposure (bps of fund net assets)',
                    round(exposure_usd / fund_total * 1e4, 4), 'inferred',
                    'under the wrong-sign reading (0 under the wrong-type reading); one basis point = 0.01 percent')
        else:
            add('downstream', 'derived', 'indicative exposure (USD)', 'not stated', 'observed',
                'no single usable FX rate; units misstated is the measure')
    else:
        add('downstream', 'prices', 'price used', 'no eligible quote on the snapshot date', 'observed',
            'exposure in USD cannot be stated; units misstated is the measure')

    # The control that would have caught it.
    add('control', 'docs/RULES.md', 'TXN_FIELDS_VALID',
        f'quantity > 0 and |gross_amount - quantity x price| <= {GROSS_TOLERANCE}', 'inferred',
        'Trade check at capture; owner: Trade operations lead; on fail: reject')

TRACE_COLUMNS = ['hop', 'table', 'field', 'value', 'evidence', 'note']
trace_table = pd.DataFrame(trace_lines, columns=TRACE_COLUMNS)
trace_table['value'] = trace_table['value'].astype(str)
trace_table.to_csv(OUTPUTS / '02_trace.csv', index=False, lineterminator='\n')
if TRACE_FOUND:
    print(f'traced key: account {ACCOUNT_ID}, security {SECURITY_ID}, transaction {TXN_ID} | class: {chosen["main_class"]}')
trace_table[~trace_table['field'].isin(transactions.columns) | (trace_table['hop'] != 'source')]

In [ ]:
# CHECK: the trace helpers on small made-up inputs. Timestamps may come in either date format.
assert booked_before_cutoff('10/06/2026 10:15:00', '2026-06-02 18:00:00') == False, 'mixed formats compared as text'
assert booked_before_cutoff('02/06/2026 10:15:00', '2026-06-02 18:00:00') == True
two_dates = pd.DataFrame({'fund_id': ['F1', 'F1', 'F1'], 'net_assets': ['100', '20', '50'],
                          'as_of_date': ['2026-07-10', '10/07/2026', '2026-07-09']})
assert fund_net_assets(two_dates, 'F1', '2026-07-10') == (120.0, 2), 'another date was added into the fund total'

In [ ]:
# CHECK: the traced record follows the selection rule, every hop is marked observed or inferred,
# and the notebook source holds no literal record IDs (they must come from the data, never from the code).
import json, re
if TRACE_FOUND:
    chosen = trace_candidates.iloc[0]
    assert chosen[TRACE['flag']] and chosen['in_rebuild'] and chosen['in_book'] and chosen['in_custodian']
    assert not chosen['not_evaluable'] and not chosen['pattern'].startswith('not evaluable')
    if TRACE_RULE_USED == 'full rule':          # the "no other flag" condition is asserted only when it was applied
        assert not chosen[OTHER_FLAGS].any(), 'the traced key carries another flag'
    assert tuple(chosen[TRACE['sort_by']]) == min(map(tuple, trace_candidates[TRACE['sort_by']].to_numpy()))
    assert set(trace_table['evidence']) <= {'observed', 'inferred'} and trace_table['evidence'].notna().all()
    assert {'source', 'pipeline', 'position', 'custodian', 'downstream'} <= set(trace_table['hop'])
else:
    assert trace_table.empty, 'a trace was written without a selected key'
    print('no trace: no key meets the selection rule (see the message above)')
notebook_file = ROOT / 'notebooks' / 'deliverable.ipynb'
source_text = ''.join(''.join(cell['source']) for cell in json.loads(notebook_file.read_text())['cells'])
literal_ids = re.findall(r'\b(?:ACC|SEC|TXN|CA|FND|LDG|POS|CMP)\d{4,}\b', source_text)
assert not literal_ids, f'literal record IDs in the notebook source: {sorted(set(literal_ids))}'
print(f"trace selected by rule ({TRACE_RULE_USED if TRACE_FOUND else 'no key qualified'}); "
      'no literal record IDs in the notebook source')

## 7. Ledger arithmetic: does net_assets / shares_outstanding give nav_per_share?

Rule `NAV_ARITHMETIC` (RULES §7): fund + share class + date unique; shares_outstanding > 0; |net_assets / shares_outstanding − nav_per_share| ≤ the tolerance in the config (one unit of nav_per_share's stored 6th decimal). Missing, unreadable or zero inputs, and an unreadable `as_of_date` (the fund + class + date key cannot be checked), are **not evaluable** and reported apart. We never sum nav_per_share. Cash, accruals and fees are not supplied, so a residual is something to investigate, not a proven NAV error.

In [ ]:
NAV_TOLERANCE = CONFIG['nav']['tolerance']
nav_check = ledger_nav.copy()
for column in ['net_assets', 'shares_outstanding', 'nav_per_share']:
    nav_check[f'{column}_number'] = pd.to_numeric(nav_check[column], errors='coerce')
nav_check['as_of_day'] = to_day(nav_check['as_of_date'])
nav_check['repeated key'] = nav_check.assign(as_of_date=normalise_date(nav_check['as_of_date'])).duplicated(
    ['fund_id', 'share_class', 'as_of_date'], keep=False)            # either date format names the same day

missing = nav_check[['net_assets_number', 'shares_outstanding_number', 'nav_per_share_number']].isna().any(axis=1)
zero_shares = nav_check['shares_outstanding_number'] <= 0
date_unreadable = nav_check['as_of_day'].isna()                     # the fund + class + date key cannot be checked
nav_check['implied_nav_per_share'] = nav_check['net_assets_number'] / nav_check['shares_outstanding_number'].where(~zero_shares)
nav_check['residual'] = nav_check['implied_nav_per_share'] - nav_check['nav_per_share_number']
nav_check['residual (bps of nav_per_share)'] = nav_check['residual'] / nav_check['nav_per_share_number'] * 1e4
nav_check['result'] = np.select([missing | zero_shares | date_unreadable, nav_check['residual'].abs() <= NAV_TOLERANCE],
                                ['not evaluable', 'pass'], default='fail')
nav_check['reason'] = np.select([missing, zero_shares, date_unreadable],
                                ['blank or unreadable input', 'shares_outstanding <= 0', 'as_of_date unreadable'], default='')
nav_check.to_csv(OUTPUTS / '02_nav_check.csv', index=False, lineterminator='\n')

nav_summary = pd.DataFrame({
    'rows': [len(nav_check)],
    'repeated fund + class + date': [int(nav_check['repeated key'].sum())],
    'pass': [int((nav_check['result'] == 'pass').sum())],
    'fail': [int((nav_check['result'] == 'fail').sum())],
    'not evaluable': [int((nav_check['result'] == 'not evaluable').sum())],
    'tolerance': [NAV_TOLERANCE]}).T.rename(columns={0: 'value'})
fails = nav_check[nav_check['result'] == 'fail']
if len(fails):
    print('residual on failing rows (bps of nav_per_share): min / median / max =',
          f"{fails['residual (bps of nav_per_share)'].min():.4f} / {fails['residual (bps of nav_per_share)'].median():.4f}"
          f" / {fails['residual (bps of nav_per_share)'].max():.4f}")
    display(fails.reindex(fails['residual'].abs().sort_values(ascending=False).index)
            [['ledger_id', 'fund_id', 'share_class', 'net_assets', 'shares_outstanding', 'nav_per_share',
              'implied_nav_per_share', 'residual', 'residual (bps of nav_per_share)']].head(5))
print('not evaluable by reason:', nav_check.loc[nav_check['result'] == 'not evaluable', 'reason'].value_counts().to_dict())

In [ ]:
# CHECK: every ledger row is pass, fail or not evaluable, and the three add up to the rows in ledger_nav.
nav_counts = nav_check['result'].value_counts()
assert set(nav_counts.index) <= {'pass', 'fail', 'not evaluable'}, nav_counts
assert int(nav_counts.sum()) == len(ledger_nav), 'ledger rows do not add up'
nav_summary

## 8. Price inputs: vendor spread and scale, for every held security on the snapshot date

Rule `PRC_VENDOR_DISPERSION` (RULES §6), run on every security held in the book. For each one, the valid quotes on the snapshot date (price > 0) across vendors give a spread = (max − min) / mean. A max/min ratio in the configured band around **100** is flagged as a **scale** issue (percent-of-par vs per-unit, or a unit error). It is never auto-corrected, because quote basis and quantity denomination are not explicit in the data. Securities with no quote, or with quotes from a single vendor, are reported apart (nothing to compare across vendors), and so are quotes in **different currencies** (they cannot be compared without a stated conversion). A lower `vendor_priority` is a preference, not proof of accuracy.

Holding coverage, quote freshness and a look inside the spread and scale flags (`PRC_QUOTE_ELIGIBLE`, `PRC_VENDOR_DISPERSION`) run in section 8b; FX validity (`FX_RATE_VALID`) in section 9c.

In [ ]:
PRICE_CONFIG = CONFIG['prices']
quotes_all = prices.copy()
quotes_all['price_day'] = normalise_date(quotes_all['price_date'])
quotes_all['price_number'] = to_number(quotes_all['price'])
snapshot_quotes = quotes_all[quotes_all['price_day'] == AS_OF_DATE]
log_rows('prices on the snapshot date', len(quotes_all), len(snapshot_quotes), 'price_date (either format) = snapshot')
invalid = snapshot_quotes['price_number'].isna() | (snapshot_quotes['price_number'] <= 0)
repeated_quotes = int(snapshot_quotes.duplicated(['security_id', 'vendor']).sum())
print(f'snapshot quotes: {len(snapshot_quotes):,} | blank, unreadable or <= 0: {int(invalid.sum())} | '
      f'repeated security + vendor: {repeated_quotes}')
valid = snapshot_quotes[~invalid]

held = positions[['security_id']].drop_duplicates()
stats = valid.groupby('security_id').agg(quotes=('price_number', 'size'), vendors=('vendor', 'nunique'),
                                         currencies=('currency', lambda c: '/'.join(sorted(set(c)))),
                                         low=('price_number', 'min'), high=('price_number', 'max'),
                                         mean=('price_number', 'mean')).reset_index()
vendor_check = held.merge(stats, on='security_id', how='left').merge(
    securities[['security_id', 'asset_class', 'currency']].rename(columns={'currency': 'master_currency'}),
    on='security_id', how='left')
vendor_check['quotes'] = vendor_check['quotes'].fillna(0).astype(int)
vendor_check['vendors'] = vendor_check['vendors'].fillna(0).astype(int)
vendor_check['spread'] = (vendor_check['high'] - vendor_check['low']) / vendor_check['mean']
vendor_check['max / min'] = vendor_check['high'] / vendor_check['low']
currency_conflict = vendor_check['currencies'].fillna('').str.contains('/')
scale = vendor_check['max / min'].between(PRICE_CONFIG['scale_ratio_low'], PRICE_CONFIG['scale_ratio_high'])
vendor_check['result'] = np.select(
    [vendor_check['quotes'] == 0, vendor_check['vendors'] == 1, currency_conflict, scale,
     vendor_check['spread'] > PRICE_CONFIG['spread_threshold']],
    ['no quote (not evaluable)', 'single vendor (not evaluable)', 'currency conflict (not evaluable)',
     'scale flag (ratio near 100)', 'spread above threshold'], default='pass')
vendor_check['quote currency differs from master'] = (vendor_check['currencies'].notna()
                                                      & vendor_check['master_currency'].notna()
                                                      & (vendor_check['currencies'] != vendor_check['master_currency']))
vendor_check['not in securities'] = vendor_check['asset_class'].isna()
vendor_check.to_csv(OUTPUTS / '02_vendor_check.csv', index=False, lineterminator='\n')

vendor_summary = vendor_check['result'].value_counts().rename('securities').to_frame()
vendor_summary['share of held'] = (vendor_summary['securities'] / len(held)).round(6)
vendor_summary['of which BOND'] = vendor_check[vendor_check['asset_class'] == 'BOND']['result'].value_counts().reindex(
    vendor_summary.index).fillna(0).astype(int)
vendor_summary['of which not in securities'] = vendor_check[vendor_check['not in securities']]['result'].value_counts().reindex(
    vendor_summary.index).fillna(0).astype(int)
print(f"held securities: {len(held):,} | thresholds: spread {PRICE_CONFIG['spread_threshold']}, scale band "
      f"[{PRICE_CONFIG['scale_ratio_low']}, {PRICE_CONFIG['scale_ratio_high']}] | quote currency differs from master: "
      f"{int(vendor_check['quote currency differs from master'].sum())}")
evaluated = vendor_check[(vendor_check['vendors'] > 1) & ~currency_conflict]
print('spread on securities with 2+ vendors in one currency: median / 99th pct / max =',
      f"{evaluated['spread'].median():.5f} / {evaluated['spread'].quantile(.99):.5f} / {evaluated['spread'].max():.3f}")

In [ ]:
# CHECK: every held security has exactly one result, and the results add up to the held securities.
held_securities = positions['security_id'].drop_duplicates()
assert vendor_check['security_id'].is_unique, 'a security appears twice'
assert set(vendor_check['security_id']) == set(held_securities), 'held securities missing from the vendor check'
assert int(vendor_summary['securities'].sum()) == len(held_securities), 'results do not add up to held securities'
vendor_summary

### 8b. Price data quality: coverage, freshness, flat runs and a look inside the vendor flags

Rule `PRC_QUOTE_ELIGIBLE` (RULES §6, D-041), on section 8's held set, so 8b and section 8 always line up. For each held security, a quote dated the snapshot is **eligible** if its price reads and is above zero and its currency is the security's master currency. The **chosen quote** is the eligible quote with the lowest `vendor_priority`, then the first vendor name, then the lowest price: a sort on values, never on file order. The **priority vendor** is the lowest `vendor_priority` in the file, as in the "preferred vendor" line.

- **Coverage** (unit: one `positions` row on the snapshot date). Pass: its security has a chosen quote. Fail: no eligible quote from any vendor, including no quote at all (the absence is the evidence, D-041). Not evaluable: no chosen quote, and a quote that cannot be judged: the security is missing from `securities` or its master currency is blank (no currency to compare) and it is quoted on the snapshot, or one of its quotes has a `price_date` that does not read, or a snapshot quote's price does not read as a finite number (`inf` and `nan` do not read). A security held on another day only is reported as "not held on the snapshot". **Fallback-only**, reported beside it, counts the passes whose chosen quote is not from the priority vendor.
- **Freshness** (unit: the chosen quote of a security held on the snapshot). Pass: `price_timestamp` falls on the quote's `price_date`, at or before `[prices] nav_cutoff_time` (exactly at the cutoff is on time). Fail: after the cutoff, or on another day. Not evaluable: the timestamp is blank or does not read; never counted as fresh. D-041 judges the chosen quote and does not re-pick; **fresh fallback** shows whether another eligible snapshot quote would pass.
- **Flat run**: the number of consecutive business dates (Mon–Fri) ending on the snapshot on which the chosen vendor gave a valid quote with the same price, in the same currency. A missing date ends the run. A run of `[prices] flat_run_dates` or more is a look-alike, not stale by itself (D-041).
- **Inside the spread and scale flags** (rule `PRC_VENDOR_DISPERSION`, on the securities section 8 flags). Two prices **agree** when |x − y| / mean(x, y) is within `[prices] spread_threshold`, section 8's own spread measure. On the snapshot date the chosen quote is **backed** when another vendor's valid quote agrees with it; the chosen quote **is odd** when it is not backed and two other vendors agree with each other; otherwise **no two agree**. A flagged security with no chosen quote is not evaluable. The **odd vendor**: when the chosen quote is backed, never the chosen vendor, and the vendors that disagree with the chosen quote come first; when the chosen quote is odd, the chosen vendor; otherwise any vendor. Within that, the one farthest from the median of the snapshot's valid quotes (distances rounded to `[tolerances] float_decimals`, ties by vendor name). Beside it, section 8's test is run on every readable price date (valid quotes from two or more vendors in one currency: the scale band first, then the spread threshold) to count the dates each security is scale- or spread-flagged.

One row per held security goes to `outputs/02_price_quality.csv`.

In [ ]:
# Section 8b. Every quote is read once more from section 8's text columns; nothing in section 8 changes.
MASTER_CURRENCY = securities.set_index('security_id')['currency']          # unique per security (section 8's check)
MASTER_CURRENCY = MASTER_CURRENCY.where(MASTER_CURRENCY != '')             # a blank currency is no currency
snapshot_rows = positions[normalise_date(positions['as_of_date']) == AS_OF_DATE]
quotes = quotes_all.assign(priority=pd.to_numeric(quotes_all['vendor_priority'], errors='coerce'),
                           master_currency=quotes_all['security_id'].map(MASTER_CURRENCY))
quote_on_snapshot = quotes['price_day'] == AS_OF_DATE
quote_date_unreadable = to_day(quotes['price_date']).isna() & (quotes['price_date'] != '')
quote_price_unreadable = quotes['price_number'].isna() & (quotes['price'] != '')
PRIORITY_VENDOR = quotes['priority'].min()            # the lowest vendor_priority in the file


def stamp_reason(frame):
    """Why a quote is not fresh ('' = fresh): its price_timestamp is blank, does not read, falls on a day other than
    its price_date, or is after the NAV cutoff (exactly at the cutoff is on time)."""
    stamp = to_time(frame['price_timestamp'])
    other_day = stamp.dt.strftime('%Y-%m-%d') != frame['price_day']
    after_cutoff = (stamp - stamp.dt.normalize()) > pd.Timedelta(NAV_CUTOFF_TIME)
    return pd.Series(np.select([frame['price_timestamp'] == '', stamp.isna(), other_day, after_cutoff],
                               ['timestamp missing', 'timestamp unreadable', 'stamped on another day',
                                'after the NAV cutoff'], default=''), index=frame.index)


eligible_quotes = quotes[quote_on_snapshot & (quotes['price_number'] > 0) & (quotes['currency'] == quotes['master_currency'])
                         & quotes['security_id'].isin(vendor_check['security_id'])]
eligible_quotes = eligible_quotes.assign(stamp_reason=stamp_reason(eligible_quotes)).sort_values(
    ['security_id', 'priority', 'vendor', 'price_number', 'price_timestamp'], kind='stable')
chosen_rows = eligible_quotes.drop_duplicates('security_id')
CHOSEN = chosen_rows[['security_id', 'vendor', 'priority', 'price_number', 'price_timestamp']].reset_index(drop=True)
fresh_alternative = set(eligible_quotes.loc[~eligible_quotes.index.isin(chosen_rows.index)
                                            & (eligible_quotes['stamp_reason'] == ''), 'security_id'])

# Flat run: walk back one business day at a time from the snapshot while the chosen vendor has a valid quote with the
# same price (and currency) on that day; the snapshot itself counts as the first date.
valid_quotes = quotes[quotes['price_number'] > 0]
VALID_QUOTE_KEYS = set(zip(valid_quotes['security_id'], valid_quotes['vendor'], valid_quotes['price_day'],
                           valid_quotes['currency'], valid_quotes['price_number']))
flat_run = pd.Series(1, index=chosen_rows.index)
running = pd.Series(True, index=chosen_rows.index)
back = 1
while running.any():
    day = str(np.busday_offset(np.datetime64(AS_OF_DATE), -back, roll='forward'))
    same = pd.Series([(s, v, day, c, p) in VALID_QUOTE_KEYS for s, v, c, p in
                      zip(chosen_rows['security_id'], chosen_rows['vendor'], chosen_rows['currency'],
                          chosen_rows['price_number'])], index=chosen_rows.index)
    running &= same
    flat_run += running.astype(int)
    back += 1

# Missing from securities, or mastered with a blank currency: no currency to compare the quotes with.
quoted_without_currency = set(quotes.loc[quote_on_snapshot & quotes['master_currency'].isna(), 'security_id'])
unreadable_quote = set(quotes.loc[quote_date_unreadable | (quote_on_snapshot & quote_price_unreadable), 'security_id'])
chosen = chosen_rows.assign(flat_run=flat_run).set_index('security_id')
price_quality = vendor_check[['security_id']].sort_values('security_id').reset_index(drop=True)
security = price_quality['security_id']
price_quality['mastered'] = security.isin(securities['security_id'])
price_quality['master_currency'] = security.map(MASTER_CURRENCY)
price_quality['snapshot_position_rows'] = security.map(snapshot_rows['security_id'].value_counts()).fillna(0).astype(int)
price_quality['chosen_vendor'] = security.map(chosen['vendor'])
price_quality['chosen_priority'] = security.map(chosen['priority']).map(lambda p: '' if pd.isna(p) else f'{p:g}')
price_quality['chosen_price'] = security.map(chosen['price_number'])
price_quality['chosen_timestamp'] = security.map(chosen['price_timestamp'])
held_on_snapshot = price_quality['snapshot_position_rows'] > 0
has_chosen = price_quality['chosen_vendor'].notna()
cannot_judge = security.isin(quoted_without_currency | unreadable_quote)
price_quality['coverage'] = np.select([~held_on_snapshot, has_chosen, cannot_judge],
                                      ['not held on the snapshot', 'pass', 'not evaluable'], default='fail')
covered = price_quality['coverage'] == 'pass'
from_priority_vendor = security.map(chosen['priority']) == PRIORITY_VENDOR
price_quality['fallback_only'] = np.where(covered, np.where(from_priority_vendor, 'no', 'yes'), '')
freshness_judged = held_on_snapshot & has_chosen
reason = security.map(chosen['stamp_reason']).where(freshness_judged, '')
price_quality['freshness'] = np.select(
    [~freshness_judged, reason.isin(['timestamp missing', 'timestamp unreadable']), reason != ''],
    ['', 'not evaluable', 'fail'], default='pass')
price_quality['freshness_reason'] = reason
stale = price_quality['freshness'] == 'fail'
price_quality['fresh_fallback'] = np.where(stale, np.where(security.isin(fresh_alternative), 'yes', 'no'), '')
price_quality['flat_run_dates'] = security.map(chosen['flat_run']).fillna(0).astype(int)
price_quality['vendor_check_result'] = security.map(vendor_check.set_index('security_id')['result'])

# Inside the spread and scale flags (PRC_VENDOR_DISPERSION): on the snapshot, does another vendor back the chosen quote,
# or do two other vendors agree against it? The quotes are section 8's valid snapshot quotes (price > 0).
SPREAD_THRESHOLD = PRICE_CONFIG['spread_threshold']
FLAG_KIND = {'scale flag (ratio near 100)': 'scale', 'spread above threshold': 'spread'}


def agrees(x, y):
    """Two prices agree when |x - y| / mean(x, y) is within [prices] spread_threshold (section 8's spread measure)."""
    return abs(x - y) / ((x + y) / 2) <= SPREAD_THRESHOLD


flag_kind = price_quality['vendor_check_result'].map(FLAG_KIND)            # NaN = not flagged
flagged_quotes = quotes[quote_on_snapshot & (quotes['price_number'] > 0)
                        & quotes['security_id'].isin(security[flag_kind.notna()])][['security_id', 'vendor', 'price_number']]
others = flagged_quotes.merge(CHOSEN[['security_id', 'vendor', 'price_number']], on='security_id', suffixes=('', '_chosen'))
others = others[others['vendor'] != others['vendor_chosen']]               # other vendors' quotes, beside the chosen one
backed = set(others.loc[agrees(others['price_number'], others['price_number_chosen']), 'security_id'])
other_pairs = others.merge(others[['security_id', 'vendor', 'price_number']], on='security_id', suffixes=('', '_b'))
other_pairs = other_pairs[other_pairs['vendor'] < other_pairs['vendor_b']]   # two different other vendors
others_agree = set(other_pairs.loc[agrees(other_pairs['price_number'], other_pairs['price_number_b']), 'security_id'])
price_quality['flag_class'] = np.select(
    [flag_kind.isna(), ~has_chosen, security.isin(backed), security.isin(others_agree)],
    ['', 'not evaluable: no chosen quote', 'backed', 'chosen is odd'], default='no two agree')
# Odd vendor. Backed: never the chosen vendor; the vendors that disagree with the chosen quote come first. Chosen is
# odd: the chosen vendor. Otherwise any vendor. Within that, the farthest from the median quote, ties by vendor name.
distance = (flagged_quotes['price_number']
            - flagged_quotes.groupby('security_id')['price_number'].transform('median')).abs().round(FLOAT_DECIMALS)
odd_candidates = flagged_quotes.assign(distance=distance).merge(
    CHOSEN[['security_id', 'vendor', 'price_number']], how='left', on='security_id', suffixes=('', '_chosen'))
odd_class = odd_candidates['security_id'].map(price_quality.set_index('security_id')['flag_class'])
is_chosen = odd_candidates['vendor'] == odd_candidates['vendor_chosen']
backs_chosen = agrees(odd_candidates['price_number'], odd_candidates['price_number_chosen'])
odd_rank = np.select([(odd_class == 'backed') & is_chosen, (odd_class == 'backed') & backs_chosen,
                      (odd_class == 'chosen is odd') & ~is_chosen], [2, 1, 1], default=0)
ODD_VENDOR = (odd_candidates.assign(rank=odd_rank)
              .sort_values(['security_id', 'rank', 'distance', 'vendor'], ascending=[True, True, False, True], kind='stable')
              .drop_duplicates('security_id').set_index('security_id')['vendor'])
price_quality['odd_vendor'] = security.map(ODD_VENDOR).where(flag_kind.notna()).fillna('')

# Dates flagged: section 8's test on every readable price date (valid quotes from 2+ vendors in one currency; the scale
# band first, then the spread threshold), one groupby over all quotes, then counted per security.
dated_quotes = quotes[(quotes['price_number'] > 0) & (quotes['price_date'] != '') & ~quote_date_unreadable
                      & quotes['security_id'].isin(security)]
per_date = dated_quotes.groupby(['security_id', 'price_day']).agg(
    vendors=('vendor', 'nunique'), currencies=('currency', 'nunique'), low=('price_number', 'min'),
    high=('price_number', 'max'), mean=('price_number', 'mean'))
comparable = (per_date['vendors'] > 1) & (per_date['currencies'] == 1)
date_scale = comparable & (per_date['high'] / per_date['low']).between(PRICE_CONFIG['scale_ratio_low'],
                                                                       PRICE_CONFIG['scale_ratio_high'])
date_spread = comparable & ~date_scale & ((per_date['high'] - per_date['low']) / per_date['mean'] > SPREAD_THRESHOLD)
DATE_FLAGS = pd.DataFrame({'scale': date_scale, 'spread': date_spread}).reset_index()
dates_by_security = DATE_FLAGS.groupby('security_id')[['scale', 'spread', 'price_day']].agg(
    {'scale': 'sum', 'spread': 'sum', 'price_day': 'size'})
DATES_QUOTED = security.map(dates_by_security['price_day']).fillna(0).astype(int)   # readable dates with a valid quote
price_quality['dates_spread_flagged'] = security.map(dates_by_security['spread']).fillna(0).astype(int)
price_quality['dates_scale_flagged'] = security.map(dates_by_security['scale']).fillna(0).astype(int)
price_quality.to_csv(OUTPUTS / '02_price_quality.csv', index=False, lineterminator='\n')

coverage_rows = price_quality.groupby('coverage')['snapshot_position_rows'].sum()
quality_summary = pd.DataFrame({
    'coverage: securities': price_quality['coverage'].value_counts(),
    'coverage: position rows on the snapshot': coverage_rows,
}).fillna(0).astype(int)
print(f'held securities (section 8): {len(price_quality):,} | held on the snapshot: {int(held_on_snapshot.sum()):,} | '
      f'position rows on the snapshot: {len(snapshot_rows):,} | priority vendor: vendor_priority {PRIORITY_VENDOR:g}')
print(f"fallback-only: {int((price_quality['fallback_only'] == 'yes').sum())} securities | freshness (NAV cutoff "
      f"{NAV_CUTOFF_TIME}): {price_quality.loc[freshness_judged, 'freshness'].value_counts().to_dict()} | reasons: "
      f"{price_quality.loc[reason != '', 'freshness_reason'].value_counts().to_dict()} | fresh fallback on "
      f"{int((price_quality['fresh_fallback'] == 'yes').sum())} | flat runs of {FLAT_RUN_DATES}+ dates: "
      f"{int((freshness_judged & (price_quality['flat_run_dates'] >= FLAT_RUN_DATES)).sum())}")
print('inside the flags:', price_quality.loc[flag_kind.notna()].groupby(['vendor_check_result', 'flag_class']).size().to_dict())
quality_summary

In [ ]:
# CHECK: one row per section 8 held security; coverage results add up to the position rows on the snapshot; freshness
# judges exactly the chosen quotes of securities held on the snapshot, and a reason goes with every fail and not evaluable;
# an odd vendor goes with every spread or scale flag, and the per-date test agrees with section 8 on the snapshot. (The
# per-flag identity of the look-inside lines is checked on the register, section 10.)
assert price_quality['security_id'].is_unique, 'a security appears twice in the price-quality table'
assert set(price_quality['security_id']) == set(vendor_check['security_id']), 'price quality and section 8 differ'
assert set(snapshot_rows['security_id']) <= set(price_quality['security_id']), 'a snapshot holding is missing'
judged_rows = price_quality.loc[price_quality['coverage'].isin(['pass', 'fail', 'not evaluable']), 'snapshot_position_rows']
assert int(judged_rows.sum()) == len(snapshot_rows), 'coverage rows do not add up to the snapshot position rows'
assert (held_on_snapshot == (price_quality['coverage'] != 'not held on the snapshot')).all()
assert (price_quality.loc[price_quality['coverage'] == 'pass', 'chosen_vendor'].notna()).all()
assert (price_quality.loc[price_quality['coverage'].isin(['fail', 'not evaluable']), 'chosen_vendor'].isna()).all()
freshness_results = price_quality['freshness'].value_counts()
freshness_eligible = int(freshness_results.get('pass', 0) + freshness_results.get('fail', 0))
assert freshness_eligible + int(freshness_results.get('not evaluable', 0)) == int((held_on_snapshot & has_chosen).sum()), \
    'freshness does not judge every chosen quote held on the snapshot'
assert ((price_quality['freshness'].isin(['fail', 'not evaluable'])) == (price_quality['freshness_reason'] != '')).all()
assert ((price_quality['fresh_fallback'] != '') == stale).all(), 'fresh fallback outside the stale quotes'
assert ((price_quality['flat_run_dates'] >= 1) == has_chosen).all(), 'a flat run without a chosen quote, or the reverse'
assert len(CHOSEN) == int(has_chosen.sum()) and CHOSEN['security_id'].is_unique
assert ((price_quality['odd_vendor'] != '') == flag_kind.notna()).all(), 'an odd vendor missing, or outside the flags'
snapshot_date_flags = DATE_FLAGS[DATE_FLAGS['price_day'] == AS_OF_DATE].set_index('security_id')
for kind in ['scale', 'spread']:
    assert set(snapshot_date_flags.index[snapshot_date_flags[kind]]) == set(security[flag_kind == kind]), \
        f'the per-date {kind} test and section 8 disagree on the snapshot'
print('price quality checks passed')

## 9. Corporate actions: supported or quarantined?

`corporate_actions` lists issuer events, not which accounts received them. Before any event-adjusted scenario (that comes for the final, kept separate from the trade-only rebuild), each event is sorted by rule `CA_TERMS_COMPLETE` (RULES §4), using the required terms per type in the config:

- **Supported:** the type's required terms are present and valid, the source security is mastered, and for MERGER and SPINOFF the target exists **and is in `securities`**.
- **Quarantined:** anything required is missing, with a reason code and the contract that would release it. REDEMPTION is quarantined by rule: the Guide gives no ratio convention for extinguished units, and we do not invent one.
- **Flags (unresolved, not quarantine):** a term the type does not use (for example a cash_rate on a split), and an ex-date after the effective date (the date definitions are an open question for faculty).

**Candidate holders** = accounts holding the source security in our trade-date rebuild just before the ex-date. They are candidates, not proven recipients: the extract has no posting log. A key with a trade whose date cannot be read could hold the security before the ex-date, so it is counted under *holder keys not evaluable*, never silently left out.

In [ ]:
CA_TERMS = {k: v for k, v in CONFIG['corporate_actions'].items() if isinstance(v, dict)}
RELEASE = {  # the contract or evidence that would release each quarantine reason
    'TYPE_NOT_CONFIGURED': 'a rule for this action type in config/reconciliation.toml',
    'SOURCE_NOT_MASTERED': 'the source security set up in securities',
    'MISSING_RATIO': 'the ratio from the issuer notice',
    'MISSING_CASH_RATE': 'the cash rate from the issuer notice',
    'MISSING_TARGET_SECURITY_ID': 'the resulting security named on the event',
    'TARGET_NOT_MASTERED': 'the target security set up in securities (identifiers, currency, asset class)',
    'NO_RATIO_CONVENTION': 'a stated convention for units extinguished per source unit, and the cash treatment',
    'DATE_UNREADABLE': 'readable ex_date and effective_date',
}
mastered = set(securities['security_id'])
ca = corporate_actions.copy()
ca['ex_day'], ca['effective_day'] = to_day(ca['ex_date']), to_day(ca['effective_date'])
ca['ratio_number'] = pd.to_numeric(ca['ratio'], errors='coerce')
ca['cash_rate_number'] = pd.to_numeric(ca['cash_rate'], errors='coerce')
present = {'ratio': ca['ratio_number'] > 0, 'cash_rate': ca['cash_rate_number'] > 0,
           'target_security_id': ca['target_security_id'] != '',
           'extinguished_units_convention': pd.Series(False, index=ca.index)}   # no such field in the extract
ca['target mastered'] = ca['target_security_id'].isin(mastered)
ca['source mastered'] = ca['security_id'].isin(mastered)

reasons, flags = [], []
for i, event in ca.iterrows():
    terms = CA_TERMS.get(event['action_type'])
    found, flagged = [], []
    if terms is None:
        found.append('TYPE_NOT_CONFIGURED')
    else:
        for term in terms['required']:
            if not present[term][i]:
                found.append('NO_RATIO_CONVENTION' if term == 'extinguished_units_convention' else f'MISSING_{term.upper()}')
        if 'target_security_id' in terms['required'] and present['target_security_id'][i] and not event['target mastered']:
            found.append('TARGET_NOT_MASTERED')
        extra = [term for term in terms['unused'] if present[term][i]]
        if extra:
            flagged.append('EXTRA_TERMS: ' + ', '.join(extra))
    if not event['source mastered']:
        found.append('SOURCE_NOT_MASTERED')
    if pd.isna(event['ex_day']) or pd.isna(event['effective_day']):
        found.append('DATE_UNREADABLE')
    elif event['ex_day'] > event['effective_day']:
        flagged.append('EX_AFTER_EFFECTIVE')
    reasons.append(' | '.join(found))
    flags.append(' | '.join(flagged))
ca['reason codes'] = reasons
ca['unresolved flags'] = flags
ca['verdict'] = np.where(ca['reason codes'] == '', 'supported', 'quarantined')
ca['what would release it'] = ca['reason codes'].map(
    lambda codes: ' | '.join(RELEASE[c] for c in codes.split(' | ')) if codes else '')

# Candidate holders: accounts holding the source security in the trade-date rebuild just before the ex-date.
# A row with an unreadable trade date may fall before the ex-date: it is kept and marks its key not evaluable,
# as in the main chain, so the key is never counted on a quantity that leaves that row out.
TRADE_DATE_ROWS = MAIN_ROWS if POLICY['holding_date'] == 'trade' else OTHER_ROWS
ca_rows = TRADE_DATE_ROWS[TRADE_DATE_ROWS['security_id'].isin(ca['security_id'])]
holders, holders_not_evaluable = [], []
for _, event in ca.iterrows():
    if pd.isna(event['ex_day']):
        holders.append(np.nan); holders_not_evaluable.append(np.nan); continue
    of_security = ca_rows[ca_rows['security_id'] == event['security_id']]
    date_unknown = of_security['trade_day'].isna()
    before = of_security[date_unknown | (of_security['trade_day'] < event['ex_day'])]
    before = before.assign(mark_not_evaluable=before['mark_not_evaluable'] | before['trade_day'].isna())
    rebuilt_before = rebuild(before) if len(before) else pd.DataFrame(columns=['rebuilt_quantity', 'not_evaluable'])
    evaluable_before = rebuilt_before[~rebuilt_before['not_evaluable'].astype(bool)]
    holders.append(int((evaluable_before['rebuilt_quantity'].abs() > TOLERANCE).sum()))
    holders_not_evaluable.append(int(rebuilt_before['not_evaluable'].astype(bool).sum()))
ca['candidate holders'] = holders
ca['holder keys not evaluable'] = holders_not_evaluable
ca_readiness = ca[['action_id', 'action_type', 'security_id', 'target_security_id', 'ex_date', 'effective_date',
                   'ratio', 'cash_rate', 'source mastered', 'target mastered', 'verdict', 'reason codes',
                   'what would release it', 'unresolved flags', 'candidate holders', 'holder keys not evaluable']]
ca_readiness.to_csv(OUTPUTS / '02_ca_readiness.csv', index=False, lineterminator='\n')

ca_summary = (ca_readiness.groupby(['action_type', 'verdict'])
              .agg(events=('action_id', 'size'), candidate_holders=('candidate holders', 'sum'),
                   with_unresolved_flags=('unresolved flags', lambda f: int((f != '').sum())))
              .reset_index())
reason_counts = ca_readiness.loc[ca_readiness['verdict'] == 'quarantined', 'reason codes'].str.split(' | ', regex=False).explode().value_counts()
flag_counts = ca_readiness['unresolved flags'].str.split(' | ', regex=False).explode().replace('', np.nan).dropna().value_counts()
print('quarantine reasons (an event can have several):', reason_counts.to_dict())
print('unresolved flags (an event can have several):', flag_counts.to_dict())

In [ ]:
# CHECK: one row per event; each is supported or quarantined with a reason; an event whose type needs a
# target that is missing or not in securities is never supported.
assert len(ca_readiness) == len(corporate_actions) and ca_readiness['action_id'].is_unique
assert set(ca_readiness['verdict']) <= {'supported', 'quarantined'}
assert (ca_readiness.loc[ca_readiness['verdict'] == 'quarantined', 'reason codes'] != '').all()
needs_target = ca_readiness['action_type'].map(lambda t: 'target_security_id' in CA_TERMS.get(t, {}).get('required', []))
assert (ca_readiness.loc[needs_target & ~ca_readiness['target mastered'], 'verdict'] == 'quarantined').all()
ca_summary

## 9b. Timeliness: latency and cutoff by feed and day

Rules `TXN_TIMESTAMPS_IN_ORDER` and `TXN_AFTER_CUTOFF` (RULES §3; choices in D-065). From the five timestamps in `pipeline_events`, for each transaction_id:

| Measure | What is measured | Judged by |
|---|---|---|
| Feed latency | received − event (our reading: how late the feed reached the platform) | the **daily median per feed** against `[timeliness] feed_latency_limit_minutes` |
| Processing latency | booked − received (brief App. B "booking latency") | the daily median per feed against `[timeliness] processing_latency_limit_minutes` |
| Booking cutoff | booked ≤ cutoff (brief App. A "cutoff adherence") | each record |
| Publication cutoff | published ≤ cutoff (the data dictionary: compare booking and publication separately) | each record |
| Timestamp order | event ≤ received ≤ booked ≤ published | each record |

- **Late is late.** A record booked or published after its cutoff is late for that measure (carried to the next cycle), even though it was booked or published in the end. Exactly at the cutoff is on time.
- **Never counted as timely when it can't be judged.** A record is *not evaluable* for a measure when that measure's own timestamps are **missing** (blank, or no pipeline record), **unparseable**, or **out of order** (a negative interval is an exception, never zero latency). One reason per record, in that order. A cutoff is out of order only when its own step comes before an earlier one (booked before event or received; published before any earlier step). Eligible + not evaluable = population on every row.
- **One record per transaction_id.** Where a repeated ID's pipeline copies differ, the cutoff checks use its latest booked and published time and earliest cutoff, and the latencies and timestamp order are judged copy by copy: the longest interval counts, and a backward pair in any copy counts. A repeated ID never gets the copy that looks better. Repeated IDs and copies that disagree on a cutoff are counted beside.
- **Day** is the cutoff date (else the event date). The limits are team assumptions set before looking at this release's latencies; the App. B median is applied as the brief states it. These are historical measurements from a static extract, not live monitoring.

Output: `outputs/02_timeliness.csv`, one row per measure × feed × day, plus each feed's total (`day = all`) and the platform (`source_system = all`). Totals are recomputed from the records, never summed from medians.

In [ ]:
STEPS = ['event', 'received', 'booked', 'published']          # the order the process implies
TIMESTAMP_FIELDS = STEPS + ['cutoff']
# A repeated ID's cutoff checks use its latest booked and published time and earliest cutoff (never the copy that looks
# better); its latencies and timestamp order are judged copy by copy below.
WORST_COPY = {'event': 'min', 'received': 'max', 'booked': 'max', 'published': 'max', 'cutoff': 'min'}
TIMELINESS_MEASURES = ['feed_latency', 'processing_latency', 'booking_cutoff', 'publication_cutoff', 'timestamp_order']
LATENCY = {'feed_latency': ('event', 'received', FEED_LATENCY_LIMITS),
           'processing_latency': ('received', 'booked', PROCESSING_LATENCY_LIMITS)}
CUTOFF_STEP = {'booking_cutoff': 'booked', 'publication_cutoff': 'published'}
TIMELINESS_COUNTS = ['population', 'eligible', 'pass', 'fail', 'not_evaluable_missing', 'not_evaluable_unparseable',
                     'not_evaluable_out_of_order', 'repeated_ids', 'copies_disagree']
TIMELINESS_COLUMNS = ['measure', 'source_system', 'day'] + TIMELINESS_COUNTS + [
    'limit_minutes', 'median_minutes', 'p95_minutes', 'max_minutes', 'median_above_limit']

# One row per pipeline record: each timestamp parsed (both date formats), with blank and unreadable kept apart.
pipe = pipeline_events[pipeline_events['transaction_id'] != ''].copy()
for field in TIMESTAMP_FIELDS:
    text = pipe[f'{field}_timestamp'].str.strip()
    pipe[field] = to_time(text)
    pipe[f'{field}_blank'] = text == ''
    pipe[f'{field}_bad'] = (text != '') & pipe[field].isna()
# Copy by copy: a backward pair in any copy counts (another copy in order cannot hide it), and each latency is the
# longest interval among the copies.
STEP_PAIRS = [(a, b) for i, a in enumerate(STEPS) for b in STEPS[i + 1:]]
for earlier, later in STEP_PAIRS:
    pipe[f'back_{earlier}_{later}'] = pipe[later] < pipe[earlier]
for measure, (start, end, _) in LATENCY.items():
    pipe[f'{measure}_copy_minutes'] = (pipe[end] - pipe[start]).dt.total_seconds() / 60
aggregations = {'pipeline_records': ('transaction_id', 'size'), 'pipeline_source': ('source_system', 'min'),
                'booked_first': ('booked', 'min'), 'published_first': ('published', 'min')}
for field in TIMESTAMP_FIELDS:
    aggregations[field] = (field, WORST_COPY[field])
    aggregations[f'{field}_blank'] = (f'{field}_blank', 'any')
    aggregations[f'{field}_bad'] = (f'{field}_bad', 'any')
for earlier, later in STEP_PAIRS:
    aggregations[f'back_{earlier}_{later}'] = (f'back_{earlier}_{later}', 'any')
for measure in LATENCY:
    aggregations[f'{measure}_copy_minutes'] = (f'{measure}_copy_minutes', 'max')


def per_id(frame, aggregations):
    """Group by transaction_id. Most IDs appear once, so only the repeated ones go through groupby (a groupby over
    every ID costs about 30 s per table on the release); an ID seen once keeps its own values."""
    repeated = frame['transaction_id'].duplicated(keep=False)
    once = frame[~repeated].set_index('transaction_id')
    once = pd.DataFrame({name: 1 if how == 'size' else once[column] for name, (column, how) in aggregations.items()},
                        index=once.index)
    return pd.concat([once, frame[repeated].groupby('transaction_id').agg(**aggregations)])


pipe_per_id = per_id(pipe, aggregations)

# Population: every transaction_id in transactions or pipeline_events. A trade with no pipeline record stays in,
# missing on every measure, on its own feed and event date.
named = transactions[transactions['transaction_id'] != '']
txn_per_id = per_id(named.assign(txn_event=to_time(named['event_timestamp'].str.strip())),
                    {'txn_source': ('source_system', 'min'), 'txn_event': ('txn_event', 'min')})
blank_id_rows = int((transactions['transaction_id'] == '').sum()), int((pipeline_events['transaction_id'] == '').sum())
timing = (pd.DataFrame(index=pd.Index(sorted(set(txn_per_id.index) | set(pipe_per_id.index)), name='transaction_id'))
          .join(pipe_per_id).join(txn_per_id))
timing['pipeline_records'] = timing['pipeline_records'].fillna(0).astype(int)
no_record = timing['pipeline_records'] == 0
usable = {}
for field in TIMESTAMP_FIELDS:
    missing = no_record | timing[f'{field}_blank'].fillna(False).astype(bool)
    unparseable = ~missing & timing[f'{field}_bad'].fillna(False).astype(bool)
    timing[f'{field}_problem'] = np.select([missing, unparseable], ['missing', 'unparseable'], '')
    usable[field] = timing[field].where(timing[f'{field}_problem'] == '')
timing['source_system'] = (timing['pipeline_source'].replace('', np.nan)
                           .fillna(timing['txn_source'].replace('', np.nan)).fillna('unknown'))
timing['day'] = (usable['cutoff'].fillna(usable['event']).fillna(timing['txn_event'])
                 .dt.strftime('%Y-%m-%d').fillna('unknown'))


def backwards(earlier, later):
    """In some pipeline copy, both readable and the later step's timestamp comes first (NaT compares False)."""
    return timing[f'back_{earlier}_{later}'].fillna(False).astype(bool)


def judge(fields, out_of_order, verdict):
    """One result per record: missing > unparseable > out of order, else the verdict."""
    problems = timing[[f'{f}_problem' for f in fields]]
    return (pd.Series(verdict, index=timing.index, dtype=object).mask(out_of_order, 'out of order')
            .mask((problems == 'unparseable').any(axis=1), 'unparseable')
            .mask((problems == 'missing').any(axis=1), 'missing'))


for measure, (start, end, _) in LATENCY.items():
    timing[measure] = judge([start, end], backwards(start, end), 'eligible')
    timing[f'{measure}_minutes'] = timing[f'{measure}_copy_minutes'].where(timing[measure] == 'eligible')
for measure, step in CUTOFF_STEP.items():
    earlier = STEPS[:STEPS.index(step)]
    out_of_order = pd.concat([backwards(e, step) for e in earlier], axis=1).any(axis=1)
    timing[measure] = judge([step, 'cutoff'], out_of_order, np.where(usable[step] <= usable['cutoff'], 'pass', 'fail'))
    first = timing[f'{step}_first'].where(timing[f'{step}_problem'] == '')
    timing[f'{measure}_copies_disagree'] = (timing['pipeline_records'] > 1) & timing[measure].isin(['pass', 'fail']) \
        & ((first <= usable['cutoff']) != (usable[step] <= usable['cutoff']))
# Order: a backward pair among readable timestamps fails; all four readable and in order passes; otherwise not evaluable.
any_backwards = pd.concat([backwards(a, b) for a, b in STEP_PAIRS], axis=1).any(axis=1)
timing['timestamp_order'] = judge(STEPS, pd.Series(False, index=timing.index), 'pass').mask(any_backwards, 'fail')


def summarise(group, measure, feed, day):
    result = group[measure]
    row = {'measure': measure, 'source_system': feed, 'day': day, 'population': len(group),
           'eligible': int(result.isin(['eligible', 'pass', 'fail']).sum()),
           'not_evaluable_missing': int((result == 'missing').sum()),
           'not_evaluable_unparseable': int((result == 'unparseable').sum()),
           'not_evaluable_out_of_order': int((result == 'out of order').sum()),
           'repeated_ids': int((group['pipeline_records'] > 1).sum())}
    if measure in LATENCY:
        minutes = group[f'{measure}_minutes'].dropna()
        if len(minutes):
            row.update(median_minutes=minutes.median(), p95_minutes=minutes.quantile(0.95), max_minutes=minutes.max())
        if day != 'all' and feed != 'all':   # the App. B threshold is daily, per feed
            limits = LATENCY[measure][2]
            row['limit_minutes'] = limits.get(feed)
            row['median_above_limit'] = ('no day' if day == 'unknown'   # no readable date: not a daily median
                                         else 'no limit for this feed' if feed not in limits
                                         else 'no eligible record' if not len(minutes)
                                         else 'yes' if row['median_minutes'] > limits[feed] else 'no')
    else:
        row.update({'pass': int((result == 'pass').sum()), 'fail': int((result == 'fail').sum())})
        if measure in CUTOFF_STEP:
            row['copies_disagree'] = int(group[f'{measure}_copies_disagree'].sum())
    return row


timeliness_rows = []
for measure in TIMELINESS_MEASURES:
    for (feed, day), group in timing.groupby(['source_system', 'day']):
        timeliness_rows.append(summarise(group, measure, feed, day))
    for feed, group in timing.groupby('source_system'):
        timeliness_rows.append(summarise(group, measure, feed, 'all'))
    timeliness_rows.append(summarise(timing, measure, 'all', 'all'))
timeliness = pd.DataFrame(timeliness_rows, columns=TIMELINESS_COLUMNS)
timeliness[TIMELINESS_COUNTS] = timeliness[TIMELINESS_COUNTS].astype('Int64')   # counts stay whole; blank where n/a
timeliness['limit_minutes'] = timeliness['limit_minutes'].astype(float)
timeliness.to_csv(OUTPUTS / '02_timeliness.csv', index=False, lineterminator='\n')
log_rows('timeliness: one record per transaction_id', len(pipeline_events), len(timing),
         'pipeline records grouped by transaction_id, plus trades with no pipeline record; blank IDs left out')
print(f'timeliness: {len(timing):,} transaction_ids ({int(no_record.sum()):,} with no pipeline record, '
      f'{int((timing["pipeline_records"] > 1).sum()):,} repeated) | rows with a blank transaction_id left out: '
      f'{blank_id_rows[0]} in transactions, {blank_id_rows[1]} in pipeline_events')

In [ ]:
# CHECK: one record per transaction_id; every record has one result per measure; the denominators add up on every
# row (eligible + not evaluable = population; pass + fail = eligible); totals equal the sum of their parts; no
# eligible latency is negative.
assert timing.index.is_unique
RESULTS = {'feed_latency': {'eligible', 'missing', 'unparseable', 'out of order'},
           'processing_latency': {'eligible', 'missing', 'unparseable', 'out of order'},
           'booking_cutoff': {'pass', 'fail', 'missing', 'unparseable', 'out of order'},
           'publication_cutoff': {'pass', 'fail', 'missing', 'unparseable', 'out of order'},
           'timestamp_order': {'pass', 'fail', 'missing', 'unparseable'}}
for measure, allowed in RESULTS.items():
    assert set(timing[measure]) <= allowed, (measure, set(timing[measure]) - allowed)
for measure in LATENCY:
    assert (timing[f'{measure}_minutes'].dropna() >= 0).all(), f'{measure}: a negative latency was measured'
counts = timeliness.fillna({c: 0 for c in TIMELINESS_COUNTS})
not_evaluable = counts[['not_evaluable_missing', 'not_evaluable_unparseable', 'not_evaluable_out_of_order']].sum(axis=1)
assert (counts['eligible'] + not_evaluable == counts['population']).all(), 'eligible + not evaluable != population'
judged = ~counts['measure'].isin(list(LATENCY))
assert (counts.loc[judged, 'pass'] + counts.loc[judged, 'fail'] == counts.loc[judged, 'eligible']).all()
SUMMED = TIMELINESS_COUNTS
day_rows = counts[(counts['source_system'] != 'all') & (counts['day'] != 'all')]
feed_rows = counts[(counts['source_system'] != 'all') & (counts['day'] == 'all')].set_index(['measure', 'source_system'])
platform_rows = counts[counts['source_system'] == 'all'].set_index('measure')
assert day_rows.groupby(['measure', 'source_system'])[SUMMED].sum().sort_index().equals(feed_rows[SUMMED].sort_index())
assert feed_rows.groupby('measure')[SUMMED].sum().sort_index().equals(platform_rows[SUMMED].sort_index())
assert (platform_rows['population'] == len(timing)).all()
platform_rows.reset_index()[['measure'] + TIMELINESS_COUNTS + ['median_minutes', 'p95_minutes', 'max_minutes']]

## 9c. Trade and FX field checks

Rule `FX_RATE_VALID` (quality-checks spec §6.6), here for the FX rates; the trade field checks follow in the cells below. Each `fx_rates` row gets three results, read from the text columns:

- **Repeated:** the (currency, rate date) has more than one row. A row whose date does not read, or whose currency is blank, is not evaluable for this check.
- **Unusable:** `rate_to_usd` is blank, unreadable (`inf` and `nan` included) or not above zero.
- **Move from the previous business day:** the previous Monday to Friday date, not the previous row. `|rate / previous rate - 1|` is rounded to `[tolerances] float_decimals` and compared with `[fx] day_on_day_band`; a move above the band is a fail. The row is **not evaluable** when it is the first date of its currency, when the previous business day has no row, more than one row or an unusable rate (a gap is never compared across), or when this row is repeated or unusable. Every currency is in the check, USD included.

In [ ]:
# Section 9c. FX rates (FX_RATE_VALID), one result row per fx_rates row, in file order.
fx_check = fx_rates[['currency']].copy()
fx_check['rate_day'] = to_day(fx_rates['rate_date'])
fx_check['rate_number'] = to_number(fx_rates['rate_to_usd'])
fx_key_readable = fx_check['rate_day'].notna() & (fx_check['currency'] != '')
fx_check['repeated'] = fx_key_readable & fx_check.duplicated(['currency', 'rate_day'], keep=False)
fx_check['unusable'] = ~(fx_check['rate_number'] > 0)

# The previous business day (Monday to Friday, no holidays) and the rate rows found there.
fx_previous_day = pd.Series(pd.NaT, index=fx_check.index, dtype='datetime64[ns]')
fx_previous_day[fx_key_readable] = np.busday_offset(
    fx_check.loc[fx_key_readable, 'rate_day'].to_numpy().astype('datetime64[D]'), -1, roll='forward').astype('datetime64[ns]')
fx_by_day = (fx_check[fx_key_readable].groupby(['currency', 'rate_day'])
             .agg(rows=('rate_number', 'size'), rate=('rate_number', 'first')).reset_index()
             .rename(columns={'rate_day': 'previous_day', 'rows': 'previous_rows', 'rate': 'previous_rate'}))
fx_lookup = (pd.DataFrame({'currency': fx_check['currency'], 'previous_day': fx_previous_day})
             .merge(fx_by_day, how='left', on=['currency', 'previous_day']))
fx_lookup.index = fx_check.index
fx_previous_usable = (fx_lookup['previous_rows'] == 1) & (fx_lookup['previous_rate'] > 0)
fx_judged = (fx_key_readable & ~fx_check['repeated'] & ~fx_check['unusable'] & fx_previous_usable)
fx_move = (fx_check['rate_number'] / fx_lookup['previous_rate'] - 1).abs().round(FLOAT_DECIMALS)
fx_check['band_result'] = np.where(~fx_judged, 'not evaluable', np.where(fx_move > FX_BAND, 'fail', 'pass'))

In [ ]:
# CHECK: one result per fx_rates row; a repeated row is never judged on the band, and no unusable rate is passed.
assert len(fx_check) == len(fx_rates)
assert set(fx_check['band_result']) <= {'pass', 'fail', 'not evaluable'}
assert (fx_check.loc[fx_check['repeated'] | fx_check['unusable'], 'band_result'] == 'not evaluable').all()
assert (fx_check['repeated'] <= (fx_check['rate_day'].notna() & (fx_check['currency'] != ''))).all()
fx_check.groupby('band_result').size().rename('rate rows').to_frame()

### 9c (continued). Trade fields: price, amount and settlement lag

Rule `TXN_FIELDS_VALID` (quality-checks spec §6.7). Each `transactions` row gets three results, read from the text columns:

- **Price:** fail when `price` is blank or does not read as a finite number (`inf` and `nan` do not read). Not evaluable only when the copies of a repeated `transaction_id` get different results (below).
- **Amount:** `|gross_amount - |quantity| x price|`, rounded to `[tolerances] float_decimals`, above `gross_amount_tolerance` is a fail (a difference that is not a number never passes) (the absolute quantity, because gross carries the size, not the side). Not evaluable when quantity, price or gross is blank or unreadable.
- **Settlement lag:** `numpy.busday_count(trade day, settlement day)` on a Monday to Friday week, no holidays. Later than `[tolerances] settlement_lag_limit_business_days` is `late`, or `late with extension` when `settlement_note` is the agreed extension note. Not evaluable when either date is blank or unreadable, or settlement is before trade.
- **Copies that disagree:** `apply_copy_rule` makes every row of a repeated `transaction_id` not evaluable on a line when its copies get more than one distinct result there. Section 9c's duplicate and correction-link checks reuse it.

In [ ]:
# Section 9c. Trade fields (TXN_FIELDS_VALID), one result row per transactions row, on trades.index.
def apply_copy_rule(results, ids):
    """Every row of a transaction_id whose copies hold more than one distinct result becomes 'not evaluable'.
    A blank id names no trade, so its rows are never grouped together."""
    distinct = results.groupby(ids).transform('nunique')
    return results.where((distinct <= 1) | (ids == ''), 'not evaluable')

trade_checks = pd.DataFrame(index=trades.index)
trade_checks['price_check'] = np.where(trades['price_number'].notna(), 'pass', 'fail')

amount_readable = trades[['quantity_number', 'price_number', 'gross_number']].notna().all(axis=1)
amount_difference = (trades['gross_number'] - trades['quantity_number'].abs() * trades['price_number']).abs().round(FLOAT_DECIMALS)
trade_checks['amount'] = np.where(~amount_readable, 'not evaluable',
                                  np.where(amount_difference <= GROSS_TOLERANCE, 'pass', 'fail'))   # a NaN never passes

lag_readable = trades['trade_day'].notna() & trades['settlement_day'].notna()
trade_checks['lag_business_days'] = np.nan
trade_checks.loc[lag_readable, 'lag_business_days'] = np.busday_count(
    trades.loc[lag_readable, 'trade_day'].to_numpy().astype('datetime64[D]'),
    trades.loc[lag_readable, 'settlement_day'].to_numpy().astype('datetime64[D]'))
lag = trade_checks['lag_business_days']
extended = trades['settlement_note'] == EXTENSION_NOTE
trade_checks['lag_class'] = np.select(
    [lag.isna() | (lag < 0), (lag > LAG_LIMIT) & extended, lag > LAG_LIMIT],
    ['not evaluable', 'late with extension', 'late'], default='pass')

for column in ['price_check', 'amount', 'lag_class']:
    trade_checks[column] = apply_copy_rule(trade_checks[column], trades['transaction_id'])

In [ ]:
# CHECK: one result per transactions row, every result in its allowed set, and a repeated transaction_id never has two results on one line.
assert len(trade_checks) == len(trades) and trade_checks.index.equals(trades.index)
assert set(trade_checks['price_check']) <= {'pass', 'fail', 'not evaluable'}
assert set(trade_checks['amount']) <= {'pass', 'fail', 'not evaluable'}
assert set(trade_checks['lag_class']) <= {'pass', 'late', 'late with extension', 'not evaluable'}
for column in ['price_check', 'amount', 'lag_class']:
    judged = trades['transaction_id'] != ''
    assert (trade_checks.loc[judged, column].groupby(trades.loc[judged, 'transaction_id']).nunique() == 1).all()
# apply_copy_rule on a small made-up case: the two copies of B disagree, and so do those of C; A has one row.
_probe = apply_copy_rule(pd.Series(['pass', 'pass', 'fail', 'pass', 'fail']), pd.Series(['A', 'B', 'B', 'C', 'C']))
assert _probe.tolist() == ['pass', 'not evaluable', 'not evaluable', 'not evaluable', 'not evaluable']
trade_checks[['price_check', 'amount', 'lag_class']].apply(lambda column: column.value_counts()).fillna(0).astype(int)

### 9c (continued). Economic duplicates and correction links

Rules `TXN_DUPLICATE` and `TXN_CORRECTION_LINK` (quality-checks spec §6.7), read from the text columns:

- **Correction family:** IDs joined by `related_transaction_id` links, in either direction and through any chain (a direct link and a shared target are the simplest cases). Every ID and link target starts as its own label; each round a node takes the smallest label among itself and its link neighbours, until nothing changes. The label is the smallest ID in the family, so it never depends on row order.
- **Same economics under a different transaction_id** (one result per non-blank `transaction_id`): a copy is evaluable when account, security, type, trade date, quantity and price are all present and read (quantity and price compared as numbers, the trade date after normalisation). The ID fails when an evaluable copy shares all six with a copy of an ID in another family. Copies of one ID are never duplicates of each other, a repeated ID fails if any copy does, and every status takes part. Not evaluable when no copy is evaluable.
- **Correction link** (one result per row with a `related_transaction_id`): fail when the link is the row's own ID, when the target ID does not exist on a row that is not CANCELLED, or when no copy of the target has the row's account and security. Not evaluable when the target ID does not exist on a CANCELLED row (that has its own line), an account or security is blank on either side, or the target's copies disagree on account or security while one of them matches. Pass otherwise. `apply_copy_rule` then runs over the linked rows, so the linked copies of one ID never disagree. Rows with no link get a blank result.

In [ ]:
# Section 9c. Economic duplicates (TXN_DUPLICATE) and correction links (TXN_CORRECTION_LINK).
def correction_families(ids, edges):
    """Label every ID and link target with the smallest ID joined to it through the links, in either direction.
    `edges` has columns transaction_id and related_transaction_id. Each round moves labels one link further; chains
    are short, so the loop runs a few times over the links only, never once per row."""
    nodes = pd.Index(sorted(set(ids) | set(edges['transaction_id']) | set(edges['related_transaction_id'])),
                     name='transaction_id')
    labels = pd.Series(nodes, index=nodes, name='family')
    node = np.concatenate([edges['transaction_id'].to_numpy(), edges['related_transaction_id'].to_numpy()])
    other = np.concatenate([edges['related_transaction_id'].to_numpy(), edges['transaction_id'].to_numpy()])
    while True:
        best = pd.Series(labels.reindex(other).to_numpy(), index=node).groupby(level=0).min()
        smaller = best < labels.reindex(best.index)
        if not smaller.any():
            return labels
        labels.loc[smaller.index[smaller]] = best[smaller]


trade_ids = trades['transaction_id']
has_id = trade_ids != ''
linked_row = has_id & (trades['related_transaction_id'] != '')
link_edges = trades.loc[linked_row, ['transaction_id', 'related_transaction_id']].drop_duplicates()
family = correction_families(trade_ids[has_id], link_edges)

# Same economics under a different transaction_id: one result per non-blank transaction_id.
ECONOMIC_FIELDS = ['account_id', 'security_id', 'transaction_type', 'trade_day', 'quantity_number', 'price_number']
economic_copies = trades.loc[has_id, ['transaction_id', *ECONOMIC_FIELDS]]
copy_evaluable = (economic_copies[ECONOMIC_FIELDS].notna().all(axis=1)
                  & (economic_copies[['account_id', 'security_id', 'transaction_type']] != '').all(axis=1))
judged_copies = economic_copies[copy_evaluable].assign(family=lambda frame: frame['transaction_id'].map(family))
# A group holding more than one family gives every member a partner outside its own family; copies share a family.
families_in_group = judged_copies.groupby(ECONOMIC_FIELDS)['family'].transform('nunique')
outside_partner = (families_in_group > 1).groupby(judged_copies['transaction_id']).any()
# Supporting evidence for the how-detected text: of the groups holding more than one family, how many share one event
# minute (event_timestamp normalised, minute precision) and one feed (source_system).
in_duplicate_group = judged_copies[families_in_group > 1]
group_evidence = in_duplicate_group.assign(
    event_minute=trades.loc[in_duplicate_group.index, 'event_time'].dt.floor('min').astype('int64'),
    feed=trades.loc[in_duplicate_group.index, 'source_system'])
group_spread = group_evidence.groupby(ECONOMIC_FIELDS).agg(
    minutes=('event_minute', 'nunique'), feeds=('feed', 'nunique'), event_minute=('event_minute', 'min'))
DUPLICATE_GROUPS = len(group_spread)
DUPLICATE_GROUPS_SHARING = int(((group_spread['minutes'] == 1) & (group_spread['feeds'] == 1)
                                & (group_spread['event_minute'] != pd.NaT.value)).sum())
economic = pd.DataFrame(index=pd.Index(sorted(economic_copies['transaction_id'].unique()), name='transaction_id'))
economic['family'] = family.reindex(economic.index)
partner = outside_partner.reindex(economic.index)     # missing where no copy of the ID is evaluable
economic['result'] = np.where(partner.isna(), 'not evaluable', np.where(partner.eq(True), 'fail', 'pass'))

# Correction links: one result per row with a related_transaction_id; blank where there is no link.
link_rows = trades.loc[linked_row, ['transaction_id', 'account_id', 'security_id', 'related_transaction_id', 'status']]
target_pairs = trades.loc[has_id, ['transaction_id', 'account_id', 'security_id']].drop_duplicates()
pairs_per_target = target_pairs.groupby('transaction_id').size()
blank_pair_targets = target_pairs.loc[(target_pairs['account_id'] == '') | (target_pairs['security_id'] == ''),
                                      'transaction_id']
link_target = link_rows['related_transaction_id']
target_pair_count = link_target.map(pairs_per_target)   # missing where the target ID does not exist
target_missing = target_pair_count.isna()
some_copy_matches = pd.MultiIndex.from_frame(link_rows[['related_transaction_id', 'account_id', 'security_id']]).isin(
    pd.MultiIndex.from_frame(target_pairs))
blank_pair = (link_rows['account_id'] == '') | (link_rows['security_id'] == '') | link_target.isin(blank_pair_targets)
link_result = pd.Series(np.select(
    [link_target == link_rows['transaction_id'], target_missing & (link_rows['status'] == 'CANCELLED'), target_missing,
     blank_pair, ~some_copy_matches, target_pair_count > 1],
    ['fail', 'not evaluable', 'fail', 'not evaluable', 'fail', 'not evaluable'], default='pass'), index=link_rows.index)
trade_checks['link'] = ''
trade_checks.loc[link_rows.index, 'link'] = apply_copy_rule(link_result, link_rows['transaction_id'])

In [ ]:
# CHECK: one family label per ID and link target, the smallest in its family; linked IDs share a label; one economic
# result per non-blank transaction_id; a link result on exactly the linked rows, and linked copies of one ID agree.
assert family.index.is_unique and (family <= family.index.to_series()).all()
assert (family.reindex(family).to_numpy() == family.to_numpy()).all()
assert (family.reindex(link_edges['transaction_id']).to_numpy()
        == family.reindex(link_edges['related_transaction_id']).to_numpy()).all()
assert len(economic) == trade_ids[has_id].nunique() and economic.index.is_unique
assert set(economic['result']) <= {'pass', 'fail', 'not evaluable'}
assert set(trade_checks['link']) <= {'pass', 'fail', 'not evaluable', ''}
assert ((trade_checks['link'] != '') == linked_row).all()
assert (trade_checks.loc[linked_row, 'link'].groupby(trade_ids[linked_row]).nunique() == 1).all()
# correction_families on a small made-up chain given out of order: D -> C, B -> C, B -> A is one family labelled A,
# found only after several rounds; E has no link and stays its own family.
_probe = correction_families(pd.Series(['D', 'C', 'B', 'E']),
                             pd.DataFrame({'transaction_id': ['D', 'B', 'B'], 'related_transaction_id': ['C', 'C', 'A']}))
assert _probe.to_dict() == {'A': 'A', 'B': 'A', 'C': 'A', 'D': 'A', 'E': 'E'}
pd.DataFrame({'economic duplicate': economic['result'].value_counts(),
              'correction link': trade_checks.loc[linked_row, 'link'].value_counts()}).fillna(0).astype(int)

## 10. Defect register

One line per finding: plain name, evidence status (**confirmed**, **unresolved** or **cleared**), the rule that tests it (a rule id from `docs/RULES.md`; rules added on 16 Oct where the draft had a gap are marked *added*), the population (affected, out of how many eligible, and how many could not be evaluated; eligible never includes a not-evaluable record, so eligible + not evaluable = the population the rule looks at), where it entered, and 2 or 3 example IDs. The examples are **produced by the code** (the first by sort order), never typed in. Notebook 01's 27 findings are recomputed here from the raw tables; the Check-in 2 findings come from sections 4 to 9. Every break class from section 5 lands in exactly one register line (checked below).

The full seven-question write-ups of the classes we take forward live in `docs/DEFECT_CATALOG.md`.

In [ ]:
def key_ids(frame):
    """account_id/security_id text for key-level examples."""
    return frame['account_id'].astype(str) + '/' + frame['security_id'].astype(str)


ID_UNIVERSE = {
    'transaction_id': set(transactions['transaction_id']),
    'account/security': set(key_ids(three_way)),
    'security_id': set(securities['security_id']) | set(positions['security_id']) | set(prices['security_id'])
                   | set(corporate_actions['target_security_id']),
    'account_id': set(positions['account_id']) | set(accounts['account_id']),
    'action_id': set(corporate_actions['action_id']),
    'ledger_id': set(ledger_nav['ledger_id']),
    'currency': set(prices['currency']) | set(fx_rates['currency']),
}
ID_UNIVERSE['transaction_id'] |= set(pipeline_events['transaction_id'])   # timeliness also counts pipeline-only IDs
TL_DAY_ROWS = timeliness[(timeliness['source_system'] != 'all') & (timeliness['day'] != 'all')]
ID_UNIVERSE['feed/day'] = set(TL_DAY_ROWS['source_system'] + ' ' + TL_DAY_ROWS['day'])   # e.g. 'OMS_FRONT 2026-06-01'
register_lines = []
KEY_LEVEL_IDS = {}      # finding -> every affected account/security key (for the coverage check below)
FINDING_MEMBERS = []    # every affected ID per finding, not only the examples (the dashboard's trace reads them)


def finding(name, status, rule, stage, unit, affected_ids, eligible, not_evaluable, population, id_kind, how,
            affected=None):
    """One register line. Convention: `eligible` never includes a record counted in `not evaluable`, and
    eligible + not evaluable = `population` (the records the rule looks at). Checked in the next cell."""
    ids = pd.Series(list(affected_ids), dtype=object).dropna().astype(str)
    if id_kind == 'account/security':
        KEY_LEVEL_IDS[name] = set(ids)
    register_lines.append({
        'finding': name, 'status': status, 'rule': rule, 'stage': stage, 'unit': unit,
        'affected': int(affected if affected is not None else len(ids)), 'eligible': int(eligible),
        'not evaluable': int(not_evaluable), 'population': int(population),
        'examples': ', '.join(sorted(ids.unique())[:3]), 'id kind': id_kind, 'how detected': how})
    FINDING_MEMBERS.extend({'finding': name, 'record_id': record_id} for record_id in sorted(ids.unique()))


n_rows = len(trades)
pair = trades['pair_type']
# --- Trade capture and pipeline (transactions, pipeline_events)
finding('Repeated transaction_id: exact full-row copy', 'confirmed', 'TXN_DUPLICATE', 'pipeline', 'rows',
        trades.loc[pair == 'exact copy', 'transaction_id'], n_rows, 0, n_rows, 'transaction_id',
        'transaction_id appears twice and all columns are equal', affected=(pair == 'exact copy').sum())
finding('Repeated transaction_id: copies differ only by date format', 'confirmed', 'TXN_DUPLICATE', 'pipeline', 'rows',
        trades.loc[pair == 'differs only by date format', 'transaction_id'], n_rows, 0, n_rows, 'transaction_id',
        'copies equal after normalising both date formats', affected=(pair == 'differs only by date format').sum())
finding('Repeated transaction_id: copies conflict on a fact', 'unresolved', 'TXN_DUPLICATE', 'pipeline', 'rows',
        trades.loc[pair == 'real conflict', 'transaction_id'], n_rows, 0, n_rows, 'transaction_id',
        'copies still differ after date normalisation; which copy is right is unknown', affected=(pair == 'real conflict').sum())
pipe_repeated = pipeline_events['transaction_id'].duplicated(keep=False)
finding('Repeated transaction_id in the pipeline', 'confirmed', 'TXN_PIPELINE_ONE_TO_ONE (added)', 'pipeline', 'rows',
        pipeline_events.loc[pipe_repeated, 'transaction_id'], len(pipeline_events), 0, len(pipeline_events),
        'transaction_id', 'pipeline_events not one-to-one with transactions', affected=pipe_repeated.sum())
slash_trade_date = trades['trade_date'].str.fullmatch(r'\d{2}/\d{2}/\d{4}')
finding('trade_date written as DD/MM/YYYY', 'confirmed', 'TXN_FIELDS_VALID', 'trade capture', 'rows',
        trades.loc[slash_trade_date, 'transaction_id'], n_rows, 0, n_rows, 'transaction_id',
        'format count; readable, logged as a conformity issue', affected=slash_trade_date.sum())
blank_settlement = trades['settlement_date'] == ''
finding('Blank settlement_date', 'confirmed', 'TXN_FIELDS_VALID', 'trade capture', 'rows',
        trades.loc[blank_settlement, 'transaction_id'], n_rows, 0, n_rows, 'transaction_id', 'blank field',
        affected=blank_settlement.sum())
dates_known = trades['trade_day'].notna() & trades['settlement_day'].notna()
settles_before = dates_known & (trades['settlement_day'] < trades['trade_day'])
finding('settlement_date before trade_date', 'confirmed', 'TXN_FIELDS_VALID', 'trade capture', 'rows',
        trades.loc[settles_before, 'transaction_id'], dates_known.sum(), (~dates_known).sum(), n_rows, 'transaction_id',
        'both dates normalised and compared', affected=settles_before.sum())
quantity_known = trades['quantity_number'].notna()
negative = trades['quantity_number'] < 0
negative_fits = negative & ((trades['gross_number'] - trades['quantity_number'].abs() * trades['price_number']).abs()
                            <= GROSS_TOLERANCE)
negative_types = ', '.join(f'{kind or "blank"} {n}' for kind, n in
                           trades.loc[negative, 'transaction_type'].value_counts().sort_index().items())
finding('Negative trade quantity', 'confirmed', 'TXN_FIELDS_VALID',
        'trade capture', 'rows', trades.loc[negative, 'transaction_id'], quantity_known.sum(), (~quantity_known).sum(),
        n_rows, 'transaction_id',
        f'quantity < 0 (by type: {negative_types}); gross_amount fits the absolute value on {int(negative_fits.sum())} '
        f'of {int(negative.sum())}, so it confirms the size'
        + ('; it is positive on every BUY and SELL row, so it does not show the side' if GROSS_HAS_NO_SIDE else '')
        + f'; {PARTNER_SUMMARY}; the book keeps the raw sign, so the reconciliation matches (trace in section 6)',
        affected=negative.sum())
finding('Blank trade quantity', 'confirmed', 'TXN_FIELDS_VALID', 'trade capture', 'rows',
        trades.loc[trades['quantity_blank'], 'transaction_id'], n_rows, 0, n_rows, 'transaction_id',
        f"blank field; the size is still in the row on {int(trades['derivable_quantity'].notna().sum())} of "
        f"{int(trades['quantity_blank'].sum())} (gross_amount / price is a "
        f"whole number and the row has no link: TXN_BLANK_QUANTITY_EVIDENCE, section 2b)", affected=trades['quantity_blank'].sum())
finding('Book leaves out the units of trades with a blank quantity', 'unresolved', 'POS_REBUILD_MATCHES', 'positions', 'keys',
        key_ids(BLANK_OMITTED), BLANK_KEYS_TESTABLE, BLANK_KEYS_POPULATION - BLANK_KEYS_TESTABLE, BLANK_KEYS_POPULATION,
        'account/security',
        'rebuilt with blank quantities derived from gross_amount / price, the key differs from the book (or the book has no row), '
        'and the book equals the blank-as-zero rebuild; the book still counts the row in transaction_count (section 2b). '
        'The size is derived, '
        'not confirmed by the source')
linked_cancelled = (trades['status'] == 'CANCELLED') & (trades['related_transaction_id'] != '')
dangling = linked_cancelled & ~trades['related_transaction_id'].isin(transactions['transaction_id'])
finding('CANCELLED row linked to an ID that does not exist', 'confirmed', 'TXN_CORRECTION_LINK', 'trade capture', 'rows',
        trades.loc[dangling, 'transaction_id'], linked_cancelled.sum(), 0, linked_cancelled.sum(), 'transaction_id',
        'related_transaction_id not found in transactions', affected=dangling.sum())
reversed_rows = trades[trades['status'] == 'REVERSED']
linked = reversed_rows['related_transaction_id'] != ''      # a REVERSED row with no link is its own original (as in R3)
reversals_per_original = reversed_rows[linked].groupby('related_transaction_id')['transaction_id'].nunique()
several_reversals = linked & reversed_rows['related_transaction_id'].map(reversals_per_original).gt(1)
finding('More than one REVERSED row per original', 'unresolved', 'TXN_ONE_REVERSAL_PER_ORIGINAL (added)', 'trade capture',
        'rows', reversed_rows.loc[several_reversals, 'transaction_id'], len(reversed_rows), 0, len(reversed_rows),
        'transaction_id', 'distinct REVERSED transaction_ids per related_transaction_id (copies of one ID count once); '
        'meaning of REVERSED is an open question', affected=several_reversals.sum())
all_rebooks = trades[(trades['status'] == 'BOOKED') & trades['related_transaction_id'].isin(cancelled.index)].copy()
all_rebooks['cancelled_quantity'] = all_rebooks['related_transaction_id'].map(cancelled)
rebook_evaluable_all, new_quantity, old_gross = rebook_tests(all_rebooks)
rebook_blank = ~rebook_evaluable_all
finding('Rebook changes the quantity but keeps the cancelled gross_amount', 'unresolved', 'TXN_FIELDS_VALID', 'trade capture',
        'rows', all_rebooks.loc[old_gross, 'transaction_id'], new_quantity.sum(), rebook_blank.sum(),
        new_quantity.sum() + rebook_blank.sum(), 'transaction_id',
        'BOOKED -> CANCELLED links with a new quantity: gross_amount fits the old quantity, not its own; which is right is '
        'unknown. A blank quantity on either side is not evaluable (D-048)')

# --- Master data
missing_account = ~positions['account_id'].isin(accounts['account_id'])
finding('Position on an account_id missing from accounts', 'confirmed', 'POS_KEYS_MASTERED (added)', 'master data', 'rows',
        positions.loc[missing_account, 'account_id'], len(positions), 0, len(positions), 'account_id',
        'positions.account_id not in accounts', affected=missing_account.sum())
missing_security = ~positions['security_id'].isin(securities['security_id'])
finding('Position on a security_id missing from securities', 'confirmed', 'POS_KEYS_MASTERED (added)', 'master data', 'rows',
        positions.loc[missing_security, 'security_id'], len(positions), 0, len(positions), 'security_id',
        'positions.security_id not in securities', affected=missing_security.sum())

# --- Corporate actions
has_target = ca['target_security_id'] != ''
target_missing = has_target & ~ca['target mastered']
finding('Corporate-action target security missing from securities', 'confirmed', 'CA_TERMS_COMPLETE', 'corporate actions',
        'events', ca.loc[target_missing, 'action_id'], has_target.sum(), 0, has_target.sum(), 'action_id',
        'target_security_id not in securities; event quarantined (section 9)', affected=target_missing.sum())
is_merger = ca['action_type'] == 'MERGER'
finding('MERGER without a target_security_id', 'confirmed', 'CA_TERMS_COMPLETE', 'corporate actions', 'events',
        ca.loc[is_merger & ~has_target, 'action_id'], is_merger.sum(), 0, is_merger.sum(), 'action_id',
        'blank target on a merger; event quarantined', affected=(is_merger & ~has_target).sum())
is_redemption = ca['action_type'] == 'REDEMPTION'
finding('REDEMPTION with no convention for extinguished units', 'unresolved', 'CA_TERMS_COMPLETE', 'corporate actions', 'events',
        ca.loc[is_redemption, 'action_id'], is_redemption.sum(), 0, is_redemption.sum(), 'action_id',
        'the Guide defines no ratio convention for redemptions; quarantined, not guessed')
extra_terms = ca['unresolved flags'].str.contains('EXTRA_TERMS')
finding('ratio or cash_rate filled for an event type that does not use it', 'unresolved', 'CA_TERMS_COMPLETE',
        'corporate actions', 'events', ca.loc[extra_terms, 'action_id'], len(ca), 0, len(ca), 'action_id',
        'terms present per action_type compared with the config table', affected=extra_terms.sum())
dates_readable = ca['ex_day'].notna() & ca['effective_day'].notna()
ex_after = dates_readable & (ca['ex_day'] > ca['effective_day'])
finding('ex_date after effective_date', 'unresolved', 'CA_TERMS_COMPLETE', 'corporate actions', 'events',
        ca.loc[ex_after, 'action_id'], dates_readable.sum(), (~dates_readable).sum(), len(ca), 'action_id',
        'date definitions to confirm with faculty', affected=ex_after.sum())

# --- Prices and FX
# Preferred vendor: only a valid quote (readable price > 0) counts as a price; a blank, unreadable or <= 0 quote
# from the preferred vendor means the preferred vendor has no price on that date.
quote_priority = pd.to_numeric(quotes_all['vendor_priority'], errors='coerce')
valid_quote = quotes_all['price_number'] > 0
best_priority = quote_priority.min()
quote_grid = quotes_all[['security_id', 'price_day']].drop_duplicates()
best_valid = (quotes_all[valid_quote].assign(priority=quote_priority[valid_quote])
              .groupby(['security_id', 'price_day'], as_index=False)['priority'].min())
quote_grid = quote_grid.merge(best_valid, on=['security_id', 'price_day'], how='left', validate='one_to_one')
no_preferred = quote_grid[~quote_grid['priority'].eq(best_priority)]    # no valid quote at all also counts
finding('Preferred vendor has no valid price on a date', 'confirmed', 'PRC_QUOTE_ELIGIBLE', 'pricing', 'security-dates',
        no_preferred['security_id'], len(quote_grid), 0, len(quote_grid), 'security_id',
        'security x date grid: no valid quote (price > 0) from the lowest vendor_priority; fallback needed',
        affected=len(no_preferred))
unpriced = ~positions['security_id'].isin(prices['security_id'])
finding('Held security with no price at all', 'confirmed', 'PRC_QUOTE_ELIGIBLE', 'pricing', 'position rows',
        positions.loc[unpriced, 'security_id'], len(positions), 0, len(positions), 'security_id',
        'positions.security_id absent from prices', affected=unpriced.sum())
# FX coverage per (currency, price date), as FX_RATE_COVERAGE states. Both dates read in either format; a rate
# counts only if it is readable and > 0. Prices in the reporting currency need no conversion. A price row with an
# unreadable date or a blank currency cannot be looked up: not evaluable.
usable_rates = (fx_rates.assign(price_day=to_day(fx_rates['rate_date']),
                                rate_number=pd.to_numeric(fx_rates['rate_to_usd'], errors='coerce'))
                .query('rate_number > 0').dropna(subset=['price_day'])[['currency', 'price_day']].drop_duplicates())
price_fx = prices[['currency']].assign(price_day=to_day(prices['price_date']))
has_rate = price_fx.merge(usable_rates.assign(has_rate=True), on=['currency', 'price_day'], how='left',
                          validate='many_to_one')['has_rate'].fillna(False).astype(bool).to_numpy()
fx_evaluable = price_fx['price_day'].notna() & (price_fx['currency'] != '')
no_fx = fx_evaluable & ~has_rate & (price_fx['currency'] != REPORTING_CURRENCY)
finding('Price with no FX rate for its currency on the price date', 'confirmed', 'FX_RATE_COVERAGE (added)', 'pricing',
        'price rows', prices.loc[no_fx, 'currency'], fx_evaluable.sum(), (~fx_evaluable).sum(), len(prices), 'currency',
        f'rate looked up per currency and price date (both date formats); {REPORTING_CURRENCY} prices need no rate',
        affected=no_fx.sum())
usd = fx_rates[fx_rates['currency'] == REPORTING_CURRENCY]
usd_rate = pd.to_numeric(usd['rate_to_usd'], errors='coerce')
usd_readable = usd_rate.notna()                         # a rate that does not parse is not evaluable, not "not 1"
usd_not_one = usd_readable & (usd_rate != 1)
finding(f'{REPORTING_CURRENCY} rate to USD is not 1', 'unresolved', 'FX_RATE_VALID', 'pricing', 'rate rows',
        usd.loc[usd_not_one, 'currency'], usd_readable.sum(), (~usd_readable).sum(), len(usd), 'currency',
        'error or convention: open question for faculty', affected=usd_not_one.sum())
for result, status in [('spread above threshold', 'unresolved'), ('scale flag (ratio near 100)', 'unresolved')]:
    hit = vendor_check['result'] == result
    evaluable_securities = ~vendor_check['result'].str.contains('not evaluable')
    finding(f'Vendor quotes on the snapshot date: {result}', status, 'PRC_VENDOR_DISPERSION', 'pricing', 'securities',
            vendor_check.loc[hit, 'security_id'], evaluable_securities.sum(), (~evaluable_securities).sum(),
            len(vendor_check), 'security_id', 'section 8; quote basis not explicit, so never auto-corrected')

# --- Positions: reconciliation (sections 3 to 5)
book_only = book_vs_custodian['side'] == 'only in positions'
finding('Position in the book with no custodian record', 'unresolved', 'POS_CUSTODIAN_BREAK', 'positions', 'keys',
        key_ids(book_vs_custodian[book_only]), len(book_vs_custodian), 0, len(book_vs_custodian), 'account/security',
        'full outer join book vs custodian; trades support the book on most (section 4): likely a custodian feed gap')

# Every break class (is a break = yes) goes to exactly one register line. Denominator for all three-way lines:
# keys in the union; eligible = evaluable keys, not evaluable = the keys reported as not evaluable (no overlap).
breaks = three_way['is a break'] == 'yes'
three_way_not_evaluable = int(three_way['is a break'].eq('not evaluable').sum())
three_way_eligible = len(three_way) - three_way_not_evaluable
SAME_BOOK_AND_CUSTODIAN = {   # rebuild is the odd one out, so book = custodian: one named line per cause
    'book_counts_duplicate_copies': 'Book and custodian count duplicate copies as trades',
    'book_double_offsets_reversed': 'Book and custodian subtract every REVERSED row (double offset)',
    'duplicates_and_reversed': 'Book and custodian count duplicate copies and subtract every REVERSED row',
    'sign_flipped_position': 'Book and custodian hold the opposite sign to the trades',
    'unexplained': 'Book and custodian agree on a quantity the trades do not explain'}


def register_line_for(pattern, cause):
    """(name, rule, how detected) of the one register line that covers a break class (pattern + cause)."""
    label = CAUSE_LABELS.get(cause, cause)
    if pattern == 'rebuild is the odd one out':
        return (SAME_BOOK_AND_CUSTODIAN.get(cause, f'Book and custodian agree; the trades differ ({label})'),
                'POS_REBUILD_MATCHES',
                'layer-1 pattern with no cause rule that fits (section 5); book = custodian' if cause == 'unexplained'
                else 'cause rule in section 5; book = custodian, so the custodian cannot catch it')
    if pattern == 'custodian is the odd one out':
        return ('Custodian differs from both the book and the trades' + ('' if cause == 'unexplained' else f' ({label})'),
                'POS_CUSTODIAN_BREAK', 'layer-1 pattern with no cause rule that fits (section 5)' if cause == 'unexplained'
                else 'cause rule in section 5, tested on custodian - rebuild')
    if pattern == 'book is the odd one out':
        return (f'Book differs from both the custodian and the trades ({label})', 'POS_REBUILD_MATCHES',
                'layer-1 pattern; cause rules tested on book - rebuild (section 5)')
    if pattern == 'all three differ':
        if cause == 'unexplained':
            return ('Book, custodian and trades all differ; no cause rule fits', 'POS_REBUILD_MATCHES',
                    'book and custodian each tested against the rebuild; no rule fits either (section 5)')
        return ('Book, custodian and trades all differ; a cause rule fits the book or the custodian', 'POS_REBUILD_MATCHES',
                'book and custodian each tested against the rebuild; the first rule either one meets sets the class (section 5)')
    return (f'Sources disagree ({pattern}; {label})', 'POS_REBUILD_MATCHES', 'layer-1 pattern (section 4)')


break_keys = three_way.loc[breaks, KEY + ['pattern', 'cause']]
line_info = {(p, c): register_line_for(p, c) for p, c in break_keys[['pattern', 'cause']].drop_duplicates().itertuples(index=False)}
break_keys = break_keys.assign(line=[line_info[(p, c)][0] for p, c in zip(break_keys['pattern'], break_keys['cause'])])
CLASS_LINES = {}       # register line -> (rule, how detected), in a fixed order: most keys first, then by name
line_sizes = break_keys['line'].value_counts()
for name in sorted(line_sizes.index, key=lambda n: (-line_sizes[n], n)):
    rule, how = next(info[1:] for info in line_info.values() if info[0] == name)
    members = break_keys[break_keys['line'] == name]
    if members['cause'].nunique() > 1:     # a line that groups several causes says how many keys each one has
        how += '; ' + ', '.join(f'{CAUSE_LABELS.get(c, c)} {n}' for c, n in members['cause'].value_counts().sort_index().items())
    CLASS_LINES[name] = rule
    finding(name, 'unresolved', rule, 'positions', 'keys', key_ids(members), three_way_eligible, three_way_not_evaluable,
            len(three_way), 'account/security', how)

no_trades = three_way['pattern'] == 'book + custodian, no trades'
finding('Position with no contributing trade', 'unresolved', 'POS_REBUILD_MATCHES', 'positions', 'keys',
        key_ids(three_way[no_trades]), three_way_eligible, three_way_not_evaluable, len(three_way), 'account/security',
        'no opening balance is provided; never back-solved from the closing position')
booked_only = compare_groups(rebuild(apply_steps(trades, [raw_rows, drop_cancelled, reversed_R1])), REPORTED['positions'])
booked_only_groups = booked_only['group'].value_counts()
BOOKED_ONLY_FINDING = "Rebuilt quantity differs from the book (Guide's BOOKED-only benchmark)"
booked_only_eligible = booked_only_groups.get('matched', 0) + booked_only_groups.get('different', 0)
booked_only_not_evaluable = (booked_only_groups.get('not evaluable (in both)', 0)
                             + booked_only_groups.get('not evaluable (only in rebuild)', 0))
finding(BOOKED_ONLY_FINDING, 'unresolved', 'POS_REBUILD_MATCHES', 'positions', 'keys',
        key_ids(booked_only[booked_only['group'] == 'different']), booked_only_eligible, booked_only_not_evaluable,
        booked_only_eligible + booked_only_not_evaluable, 'account/security',
        'waterfall in section 2 (keys in both, or only in the rebuild and not evaluable); classified in sections 4 and 5')

# The cleared-flag candidate: rule written in the plan before looking. The pattern already means neither the book nor
# the custodian has the key; the evidence that this absence is their convention, not a gap, is that neither source
# ever stores a zero-quantity row (each source's own convention for flat positions).
nets_to_zero = three_way['pattern'] == 'rebuild only, nets to zero'
book_zero_rows = int((REPORTED['positions']['reported_quantity'].abs() <= TOLERANCE).sum())
custodian_zero_rows = int((REPORTED['comparison_positions']['reported_quantity'].abs() <= TOLERANCE).sum())
finding('Closed position not reported (trades net to zero, absent from book and custodian)',
        'cleared' if book_zero_rows == 0 and custodian_zero_rows == 0 else 'unresolved',
        'POS_REBUILD_MATCHES', 'positions', 'keys', key_ids(three_way[nets_to_zero]), three_way_eligible,
        three_way_not_evaluable, len(three_way), 'account/security',
        f'positions stores {book_zero_rows} and comparison_positions {custodian_zero_rows} zero-quantity rows: '
        + ('neither source reports flat positions, so their absence is expected. ' if book_zero_rows + custodian_zero_rows == 0
           else 'a source reports flat positions, so its absence here is not explained. ')
        + 'Would reopen: a zero row in either source, or a later non-zero record')

# --- NAV
finding('NAV arithmetic does not hold (net_assets / shares_outstanding vs nav_per_share)', 'confirmed', 'NAV_ARITHMETIC',
        'NAV', 'ledger rows', nav_check.loc[nav_check['result'] == 'fail', 'ledger_id'],
        int((nav_check['result'] != 'not evaluable').sum()), int((nav_check['result'] == 'not evaluable').sum()),
        len(nav_check), 'ledger_id', 'section 7; which input is wrong is unknown, and cash or accruals are not supplied')

# --- Timeliness (section 9b, D-065). Appended after every other line so existing lines keep their order (the dashboard
# headline takes a record's lowest line, D-058). Not evaluable = a record (or feed-day) the rule cannot judge.
TL_IDS = len(timing)
tl_problems = timing[[f'{field}_problem' for field in TIMESTAMP_FIELDS]]
for problem in ['missing', 'unparseable']:
    finding(f'Pipeline timestamp {problem}', 'confirmed', 'TXN_TIMESTAMPS_IN_ORDER', 'pipeline', 'transaction_ids',
            timing.index[(tl_problems == problem).any(axis=1)], TL_IDS, 0, TL_IDS, 'transaction_id',
            f'section 9b: an event, received, booked, published or cutoff timestamp is {problem} in a pipeline record'
            + (' (or the trade has no pipeline record)' if problem == 'missing' else ' (neither date format reads)')
            + '; such a record is never counted as on time, and the measures it blocks report it as not evaluable')
tl_judged = timing['timestamp_order'].isin(['pass', 'fail'])
finding('Pipeline timestamps out of order', 'confirmed', 'TXN_TIMESTAMPS_IN_ORDER', 'pipeline', 'transaction_ids',
        timing.index[timing['timestamp_order'] == 'fail'], int(tl_judged.sum()), int((~tl_judged).sum()), TL_IDS,
        'transaction_id', 'section 9b: two readable timestamps among event, received, booked and published run backwards '
        '(a negative interval is an exception, never zero latency); not evaluable = a timestamp missing or unreadable '
        'with no backward pair among the rest')
for measure, label, interval, key in [
        ('feed_latency', 'Feed latency', 'received - event', 'feed_latency_limit_minutes'),
        ('processing_latency', 'Processing latency', 'booked - received', 'processing_latency_limit_minutes')]:
    tl_days = TL_DAY_ROWS[TL_DAY_ROWS['measure'] == measure]
    tl_judged = tl_days['median_above_limit'].isin(['yes', 'no'])
    tl_above = tl_days[tl_days['median_above_limit'] == 'yes']
    finding(f"{label}: daily median above the feed's limit", 'unresolved', 'TXN_TIMESTAMPS_IN_ORDER', 'pipeline',
            'feed-days', tl_above['source_system'] + ' ' + tl_above['day'], int(tl_judged.sum()), int((~tl_judged).sum()),
            len(tl_days), 'feed/day',
            f'section 9b: the median {interval} of a feed on a day is above [timeliness] {key} for that feed; brief App. B '
            'judges the daily median by feed. Unresolved: the limit is a team assumption, not an SLA. Not evaluable = '
            'no limit for the feed or no eligible record that day')
for measure, step, name in [('booking_cutoff', 'booked', 'Booked after cutoff'),
                            ('publication_cutoff', 'published', 'Published after cutoff')]:
    tl_judged = timing[measure].isin(['pass', 'fail'])
    finding(name, 'confirmed', 'TXN_AFTER_CUTOFF', 'pipeline', 'transaction_ids', timing.index[timing[measure] == 'fail'],
            int(tl_judged.sum()), int((~tl_judged).sum()), TL_IDS, 'transaction_id',
            f'section 9b: {step}_timestamp after the cutoff_timestamp of its pipeline record (exactly at the cutoff is on '
            f'time); the record is carried to the next cycle. A repeated ID uses its latest {step} time. Not evaluable '
            f'= {step} or cutoff missing or unreadable, or {step} before an earlier step')

# --- Price data quality (section 8b; quality-checks spec §6.9). Appended after the timeliness lines, in spec order.
quality_rows = snapshot_rows[['security_id']].merge(price_quality[['security_id', 'coverage', 'fallback_only']],
                                                    on='security_id', how='left', validate='many_to_one')
coverage_fail = quality_rows['coverage'] == 'fail'
coverage_unjudged = quality_rows['coverage'] == 'not evaluable'
never_priced = coverage_fail & ~quality_rows['security_id'].isin(prices['security_id'])
finding('Holding with no eligible quote on the snapshot date', 'confirmed', 'PRC_QUOTE_ELIGIBLE', 'pricing',
        'position rows', quality_rows.loc[coverage_fail, 'security_id'], int((~coverage_unjudged).sum()),
        int(coverage_unjudged.sum()), len(quality_rows), 'security_id',
        f'section 8b: no quote dated {AS_OF_DATE} whose price reads, is above zero and is in the security\'s master '
        'currency, from any vendor (the absence is the evidence, D-041). Not evaluable = no such quote, and the security '
        'is quoted on the snapshot but missing from securities, or a price_date or snapshot price does not read. '
        f'Overlap: {int(never_priced.sum())} of these {int(coverage_fail.sum())} rows are also on "Held security with no '
        'price at all" (no quote on any date)', affected=coverage_fail.sum())
covered_rows = quality_rows['coverage'] == 'pass'
fallback_rows = covered_rows & (quality_rows['fallback_only'] == 'yes')
fallback_securities = set(quality_rows.loc[fallback_rows, 'security_id'])
also_preferred_line = fallback_securities & set(no_preferred.loc[no_preferred['price_day'] == AS_OF_DATE, 'security_id'])
finding('Holding priced only by a fallback vendor on the snapshot date', 'confirmed', 'PRC_QUOTE_ELIGIBLE', 'pricing',
        'position rows', quality_rows.loc[fallback_rows, 'security_id'], int(covered_rows.sum()), 0,
        int(covered_rows.sum()), 'security_id',
        f'section 8b: the chosen quote (lowest vendor_priority among the eligible quotes, then vendor, then price) is not '
        f'from the priority vendor (vendor_priority {PRIORITY_VENDOR:g}); population = rows with a chosen quote '
        f'(coverage passes). Overlap: {len(also_preferred_line)} of these {len(fallback_securities)} securities are also '
        f'on "Preferred vendor has no valid price on a date" for {AS_OF_DATE}; for the others the priority vendor\'s '
        'valid quote is in another currency', affected=fallback_rows.sum())
fresh_judged = price_quality['freshness'].isin(['pass', 'fail', 'not evaluable'])
fresh_unjudged = price_quality['freshness'] == 'not evaluable'
stale_quotes = price_quality['freshness'] == 'fail'
stale_reasons = price_quality.loc[stale_quotes, 'freshness_reason'].value_counts()
in_flat_run = fresh_judged & (price_quality['flat_run_dates'] >= FLAT_RUN_DATES)
finding('Chosen quote stamped after the NAV cutoff', 'unresolved', 'PRC_QUOTE_ELIGIBLE', 'pricing', 'securities',
        price_quality.loc[stale_quotes, 'security_id'], int((fresh_judged & ~fresh_unjudged).sum()),
        int(fresh_unjudged.sum()), int(fresh_judged.sum()), 'security_id',
        f'section 8b: the chosen quote\'s price_timestamp is after [prices] nav_cutoff_time {NAV_CUTOFF_TIME} '
        f'({int(stale_reasons.get("after the NAV cutoff", 0))}) or on a day other than its price_date '
        f'({int(stale_reasons.get("stamped on another day", 0))}); exactly at the cutoff is on time. Unresolved: the '
        'cutoff is our reading of the pipeline cutoff, not a stated NAV strike time. Not evaluable = timestamp blank or '
        'unreadable. Beside it: another eligible quote is fresh for '
        f'{int((price_quality["fresh_fallback"] == "yes").sum())} of these (D-041 does not re-pick); '
        f'{int(in_flat_run.sum())} chosen quotes repeat the same price on {FLAT_RUN_DATES} or more business dates '
        'ending on the snapshot (a look-alike, not stale by itself)')
LOOK_REOPEN = ('Would reopen if the backing vendor is shown not to be independent of the chosen one, or a stated '
               'quote basis contradicts it')
for kinds, flag_class, name, status, test, verdict in [
        (['scale'], 'chosen is odd', 'Scale flag: the chosen quote is the odd one out', 'confirmed',
         'the chosen quote agrees with no other vendor, while two other vendors agree with each other',
         'Confirmed: a gap near 100x against two vendors that agree, so the chosen price is on another basis; never '
         'auto-corrected'),
        (['scale'], 'backed', 'Scale flag: the chosen quote is backed by another vendor', 'cleared',
         'another vendor\'s valid quote agrees with the chosen quote',
         f'Cleared: the price we would use is corroborated; the odd vendor is logged for the vendor owner. {LOOK_REOPEN}'),
        (['spread'], 'chosen is odd', 'Spread flag: the chosen quote is the odd one out', 'unresolved',
         'the chosen quote agrees with no other vendor, while two other vendors agree with each other',
         'Unresolved: vendor_priority is a preference, not proof of accuracy (Dataset Guide §4)'),
        (['spread'], 'backed', 'Spread flag: the chosen quote is backed by another vendor', 'cleared',
         'another vendor\'s valid quote agrees with the chosen quote',
         f'Cleared: the price we would use is corroborated; the odd vendor is logged for the vendor owner. {LOOK_REOPEN}'),
        (['scale', 'spread'], 'no two agree', 'Spread or scale flag: no two vendors agree', 'unresolved',
         'no two vendors\' valid quotes agree, the chosen quote included', 'Unresolved: no majority of vendors to price from')]:
    in_scope = price_quality[price_quality['vendor_check_result'].map(FLAG_KIND).isin(kinds)]
    hit = in_scope['flag_class'] == flag_class
    unjudged = in_scope['flag_class'] == 'not evaluable: no chosen quote'
    odd_counts = in_scope.loc[hit, 'odd_vendor'].value_counts().sort_index()
    dates_flagged = in_scope.loc[hit, [f'dates_{k}_flagged' for k in kinds]].sum(axis=1)
    dates_quoted = DATES_QUOTED[in_scope.index[hit]]
    median_text = (f'{dates_flagged.median():g} of a median {dates_quoted.median():g} dates with a valid quote'
                   if hit.any() else 'none (no security on this line)')
    finding(name, status, 'PRC_VENDOR_DISPERSION', 'pricing', 'securities', in_scope.loc[hit, 'security_id'],
            int((~unjudged).sum()), int(unjudged.sum()), len(in_scope), 'security_id',
            f'section 8b, on {AS_OF_DATE}, among the securities section 8 flags as {" or ".join(kinds)}: {test}. Two '
            f'quotes agree when |x - y| / mean(x, y) is within [prices] spread_threshold {SPREAD_THRESHOLD:g}. Odd vendor '
            '(the chosen vendor when its quote is odd; never the chosen vendor when it is backed, those that disagree '
            'with it first; otherwise any; then the farthest from the median snapshot quote): '
            f'{", ".join(f"{v} {n}" for v, n in odd_counts.items()) or "none"}. Median dates flagged ({" or ".join(kinds)}): '
            f'{median_text}. Chosen quotes in a flat run of {FLAT_RUN_DATES} or more business dates: '
            f'{int((in_scope.loc[hit, "flat_run_dates"] >= FLAT_RUN_DATES).sum())}. {verdict}. Not evaluable = a flagged '
            'security with no chosen quote')

# --- FX rates (section 9c; quality-checks spec §6.6)
fx_rows = len(fx_check)
fx_readable_key = fx_check['rate_day'].notna() & (fx_check['currency'] != '')
finding('FX rate repeated for a currency and date', 'confirmed', 'FX_RATE_VALID', 'pricing', 'rate rows',
        fx_check.loc[fx_check['repeated'], 'currency'], fx_readable_key.sum(), (~fx_readable_key).sum(), fx_rows, 'currency',
        'section 9c: the same currency has more than one rate row for the same date. Not evaluable = a rate date that '
        'does not read, or a blank currency', affected=fx_check['repeated'].sum())
finding('FX rate not positive or unreadable', 'confirmed', 'FX_RATE_VALID', 'pricing', 'rate rows',
        fx_check.loc[fx_check['unusable'], 'currency'], fx_rows, 0, fx_rows, 'currency',
        'section 9c: rate_to_usd is blank, unreadable, zero or negative', affected=fx_check['unusable'].sum())
fx_band_judged = fx_check['band_result'] != 'not evaluable'
finding('FX rate moves more than the band from the previous business day', 'unresolved', 'FX_RATE_VALID', 'pricing',
        'rate rows', fx_check.loc[fx_check['band_result'] == 'fail', 'currency'], fx_band_judged.sum(),
        (~fx_band_judged).sum(), fx_rows, 'currency',
        f'section 9c: |rate / previous business day\'s rate - 1| above [fx] day_on_day_band {FX_BAND:g}, every currency '
        'included. Not evaluable = the first date of a currency, a previous business day without exactly one usable '
        'rate, or a repeated or unusable rate. Unresolved: a band is a team assumption and a real market can move',
        affected=(fx_check['band_result'] == 'fail').sum())

# --- Trade fields (section 9c; quality-checks spec §6.7)
LAG_NAME = 'Settlement later than the lag limit'   # a stable name: the limit is in the how-detected text
# Blank and unreadable prices are counted over the failing rows; prices on copies that disagree are not evaluable.
price_blank = trades['price'] == ''
price_unreadable = ~price_blank & trades['price_number'].isna()
price_failing = trade_checks['price_check'] == 'fail'
price_disagreeing = int(((price_blank | price_unreadable) & (trade_checks['price_check'] == 'not evaluable')).sum())
price_text = (f'{int((price_blank & price_failing).sum())} blank, {int((price_unreadable & price_failing).sum())} unreadable'
              + (f' ({price_disagreeing} more on copies that disagree, not evaluable)' if price_disagreeing else '')
              + '; not evaluable only where the copies of a repeated transaction_id disagree')
for name, status, column, bad, test, text in [
        ('Blank or unreadable trade price', 'confirmed', 'price_check', 'fail',
         'price is blank or does not read as a number', price_text),
        ('gross_amount does not equal quantity x price', 'confirmed', 'amount', 'fail',
         f'|gross_amount - |quantity| x price| above [tolerances] gross_amount_tolerance {GROSS_TOLERANCE:g} (absolute quantity: '
         'gross carries the size, not the side), compared after rounding to [tolerances] float_decimals. Not evaluable = '
         'quantity, price or gross_amount blank or unreadable. The row\'s own arithmetic fails; which field is wrong is '
         'unknown, and a rebooked cancellation can sit on this line and on the rebook line', ''),
        (f'{LAG_NAME} with no agreed extension', 'unresolved', 'lag_class', 'late',
         f'business days from trade to settlement (Monday to Friday, no holidays) above T+{LAG_LIMIT} ([tolerances] '
         f'settlement_lag_limit_business_days {LAG_LIMIT}) and settlement_note is not {EXTENSION_NOTE}', ''),
        (f'{LAG_NAME} with an agreed extension', 'cleared', 'lag_class', 'late with extension',
         f'business days from trade to settlement above T+{LAG_LIMIT} ([tolerances] settlement_lag_limit_business_days '
         f'{LAG_LIMIT}), with settlement_note {EXTENSION_NOTE}', '')]:
    result = trade_checks[column]
    unjudged = result == 'not evaluable'
    how = f'section 9c, per row: {test}'
    if text:
        how += f' ({text})'
    if column == 'lag_class':
        how += ('. Not evaluable = a blank or unreadable trade or settlement date, settlement before trade, or copies of '
                'a repeated transaction_id that disagree. ' + ('Unresolved: a late settlement may be an error or a real delay'
                                                                if status == 'unresolved' else
                                                                'Cleared on the Dataset Guide\'s wording of the note; the only '
                                                                'evidence is the note in the same row, so it would reopen if a '
                                                                'counterparty confirmation showed no agreed extension'))
    elif column == 'amount':
        how += '. Copies of a repeated transaction_id that disagree are not evaluable'
    finding(name, status, 'TXN_FIELDS_VALID', 'trade capture', 'rows', trades.loc[result == bad, 'transaction_id'],
            (~unjudged).sum(), unjudged.sum(), n_rows, 'transaction_id', how, affected=(result == bad).sum())

# --- Economic duplicates and correction links (section 9c; quality-checks spec §6.7)
economic_judged = economic['result'] != 'not evaluable'
finding('Same economics under a different transaction_id', 'unresolved', 'TXN_DUPLICATE', 'trade capture',
        'transaction_ids', economic.index[economic['result'] == 'fail'], economic_judged.sum(), (~economic_judged).sum(),
        len(economic), 'transaction_id',
        'section 9c, per transaction_id: a copy shares account_id, security_id, transaction_type, trade date, quantity and '
        'price (numbers compared as numbers, dates after normalisation) with a copy of another ID outside its correction '
        'family (IDs joined by related_transaction_id in either direction, through any chain). Copies of one ID are never '
        'duplicates of each other; a repeated ID matches if any copy does; every status takes part. Not evaluable = no '
        'copy with all six fields present and readable. Unresolved: two real trades can share their economics. Supporting '
        f'evidence: {DUPLICATE_GROUPS_SHARING} of {DUPLICATE_GROUPS} groups share one event minute and one feed')
link_check = trade_checks['link']
link_unjudged = link_check == 'not evaluable'
link_population = link_check != ''
finding('Correction link points to itself, to a missing ID, or to another account or security', 'confirmed',
        'TXN_CORRECTION_LINK',
        'trade capture', 'rows', trades.loc[link_check == 'fail', 'transaction_id'], (link_population & ~link_unjudged).sum(),
        link_unjudged.sum(), link_population.sum(), 'transaction_id',
        'section 9c, per row with a related_transaction_id: the link is the row\'s own ID, the target ID does not exist '
        '(rows that are not CANCELLED), or no copy of the target has the row\'s account_id and security_id. Not evaluable '
        '= a CANCELLED row whose target ID does not exist (its own line, "CANCELLED row linked to an ID that does not '
        'exist"), a blank account_id or security_id on either side, the target\'s copies disagree on account or security '
        'while one of them matches, or linked copies of a repeated transaction_id that disagree',
        affected=(link_check == 'fail').sum())

register = pd.DataFrame(register_lines)
register.to_csv(OUTPUTS / '02_defect_register.csv', index=False, lineterminator='\n')
finding_members = pd.DataFrame(FINDING_MEMBERS, columns=['finding', 'record_id'])
finding_members.to_csv(OUTPUTS / '02_finding_members.csv', index=False, lineterminator='\n')

In [ ]:
# CHECK: every finding has a status, a rule and coherent denominators, and its example IDs exist in the data.
assert set(register['status']) <= {'confirmed', 'unresolved', 'cleared'}, register['status'].unique()
assert (register['rule'] != '').all() and (register['stage'] != '').all()
assert register['finding'].is_unique, 'two register lines share a name'
assert (register['eligible'] >= 0).all() and (register['affected'] <= register['eligible']).all()
assert (register['not evaluable'] >= 0).all()
assert ((register['affected'] == 0) | (register['examples'] != '')).all(), 'a finding with records has no example'
members_by_finding = finding_members.groupby('finding')['record_id'].apply(set)
for _, line in register.iterrows():
    examples = [e for e in line['examples'].split(', ') if e]
    missing_ids = [e for e in examples if e not in ID_UNIVERSE[line['id kind']]]
    assert not missing_ids, f"{line['finding']}: examples not found in the data: {missing_ids}"
    # The full list behind each line (02_finding_members.csv) holds its examples and only IDs from the data.
    members = members_by_finding.get(line['finding'], set())
    assert set(examples) <= members, f"{line['finding']}: an example is missing from the member list"
    assert members <= ID_UNIVERSE[line['id kind']], f"{line['finding']}: a member ID is not in the data"
    assert (line['affected'] > 0) == bool(members), f"{line['finding']}: affected and member list disagree"

# CHECK: the two lag lines never share a member (the extension note splits them).
assert not (members_by_finding.get(f'{LAG_NAME} with no agreed extension', set())
            & members_by_finding.get(f'{LAG_NAME} with an agreed extension', set())), 'a record is on both lag lines'

# CHECK (denominators): eligible and not evaluable never overlap: together they are exactly the population.
bad_denominator = register[register['eligible'] + register['not evaluable'] != register['population']]
assert bad_denominator.empty, bad_denominator[['finding', 'eligible', 'not evaluable', 'population']]

# CHECK (look inside the vendor flags, quality-checks spec §7): per flag, the affected of its odd and backed lines, its
# share of the "no two vendors agree" members and its not evaluable add up to section 8's count of that flag.
register_by_name = register.set_index('finding')
section8_result = vendor_check.set_index('security_id')['result']
no_two_agree = sorted(members_by_finding.get('Spread or scale flag: no two vendors agree', set()))
for result, odd_line, backed_line in [
        ('scale flag (ratio near 100)', 'Scale flag: the chosen quote is the odd one out',
         'Scale flag: the chosen quote is backed by another vendor'),
        ('spread above threshold', 'Spread flag: the chosen quote is the odd one out',
         'Spread flag: the chosen quote is backed by another vendor')]:
    odd_row, backed_row = register_by_name.loc[odd_line], register_by_name.loc[backed_line]
    from_lines = (int(odd_row['affected']) + int(backed_row['affected'])
                  + int((section8_result[no_two_agree] == result).sum()) + int(odd_row['not evaluable']))
    assert from_lines == int((vendor_check['result'] == result).sum()), \
        f'{result}: the look-inside lines add up to {from_lines}, section 8 flags {int((vendor_check["result"] == result).sum())}'

# CHECK (register against classes): every key with is a break = yes is in exactly one break-class line, the lines
# hold nothing else, and their affected counts add up to the break count.
break_id_list = key_ids(three_way[breaks])
in_class_lines = [KEY_LEVEL_IDS[name] for name in CLASS_LINES]
assert sum(len(ids) for ids in in_class_lines) == len(break_id_list), 'a break key is in two lines, or in none'
assert set().union(*in_class_lines) == set(break_id_list), 'the break-class lines and the break keys differ'
assert int(register.loc[register['finding'].isin(CLASS_LINES), 'affected'].sum()) == int(breaks.sum())
# The other key-level lines from the three-way (one-sided, cleared) hold no break key. The BOOKED-only line is a
# different rebuild (the Guide's benchmark), so it overlaps the breaks by design and is left out of this check.
for name, ids in KEY_LEVEL_IDS.items():
    if name not in CLASS_LINES and name != BOOKED_ONLY_FINDING:
        assert not ids & set(break_id_list), f'{name} holds keys that are already break-class lines'
print(f"findings: {len(register)} | confirmed {int((register['status'] == 'confirmed').sum())}, unresolved "
      f"{int((register['status'] == 'unresolved').sum())}, cleared {int((register['status'] == 'cleared').sum())} | "
      f"breaks {int(breaks.sum()):,} in {len(CLASS_LINES)} register lines")
register[['finding', 'status', 'rule', 'stage', 'affected', 'eligible', 'not evaluable', 'population', 'unit', 'examples']]

## 11. Error severity: NAV sensitivity and distinct accounts

This section reuses the classifications above (merged from the former `notebooks/03_error_severity.ipynb`, D-060) and writes `02_severity.csv`, `02_fund_impact.csv`, `02_impact_detail.csv`, `02_account_membership.csv` and `02_severity.png` / `.pdf`. No raw data is modified.

**Financial axis:** alternative minus current investment value, USD and basis points of each fund's reported net assets. These are indicative, investment-only scenarios; missing cash/accruals and unknown corrections prevent a claim of actual NAV misstatement or loss. NAV arithmetic uses a separate equivalent-dollar inconsistency, assuming net assets and shares are authoritative. Vendor flags use high-minus-low quote scenarios, not a verified correction.

**Breadth axis:** distinct account IDs in each defect/scenario population. NAV arithmetic uses the affected fund/share-class account cohort, explicitly distinguished from directly implicated holding accounts. Fund-wide account cohorts are also reported for holding scenarios; these are possible downstream reach, not confirmed investor harm.

Only unique-master EQUITY records with eligible, unambiguous quotes and FX are monetized. Bonds have unresolved quote/quantity basis. Generic holding scenarios exclude existing vendor spread/scale flags. Price scenarios intentionally quantify those flags. Unknowns remain blank, with coverage and reasons. No materiality thresholds or combined score are invented. Classes and scenarios overlap and must not be summed.


In [ ]:
# Unique mappings; quarantine duplicate master keys instead of expanding joins.
impact_accounts = accounts.loc[~accounts.duplicated('account_id', keep=False), ['account_id','fund_id','share_class']]
impact_securities = securities.loc[~securities.duplicated('security_id', keep=False), ['security_id','asset_class','currency']].rename(columns={'currency':'master_currency'})
impact_fx = fx_rates.assign(day=normalise_date(fx_rates['rate_date']), fx=pd.to_numeric(fx_rates['rate_to_usd'], errors='coerce'))
impact_fx = impact_fx[impact_fx['day'].eq(AS_OF_DATE)]
impact_fx = impact_fx[~impact_fx.duplicated('currency',keep=False) & impact_fx['fx'].gt(0)]
impact_quotes = snapshot_quotes.copy().merge(impact_securities,on='security_id',how='left',validate='many_to_one')
impact_quotes['priority'] = pd.to_numeric(impact_quotes['vendor_priority'],errors='coerce')
impact_quotes['timestamp'] = to_time(impact_quotes['price_timestamp'])
impact_quotes = impact_quotes[impact_quotes['price_number'].gt(0) & impact_quotes['priority'].notna()
    & impact_quotes['currency'].eq(impact_quotes['master_currency'])
    & impact_quotes['timestamp'].dt.strftime('%Y-%m-%d').eq(AS_OF_DATE)]
# A duplicate vendor quote is ambiguous, even if one row looks usable.
impact_duplicate_quotes = snapshot_quotes[snapshot_quotes.duplicated(['security_id','vendor'],keep=False)][['security_id','vendor']].drop_duplicates()
impact_quotes = impact_quotes.merge(impact_duplicate_quotes.assign(duplicate=True),on=['security_id','vendor'],how='left',validate='many_to_one')
impact_quotes = impact_quotes[~impact_quotes['duplicate'].eq(True)]
impact_best = impact_quotes[impact_quotes['priority'].eq(impact_quotes.groupby('security_id')['priority'].transform('min'))]
impact_best = impact_best[~impact_best.duplicated('security_id',keep=False)]
impact_price = impact_securities.merge(impact_best[['security_id','price_number','vendor']],on='security_id',how='left',validate='one_to_one')
impact_price = impact_price.merge(impact_fx[['currency','fx']].rename(columns={'currency':'master_currency'}),on='master_currency',how='left',validate='many_to_one')
impact_price.loc[impact_price['master_currency'].eq(REPORTING_CURRENCY),'fx']=1.0
impact_price = impact_price.merge(vendor_check[['security_id','result']].drop_duplicates(),on='security_id',how='left',validate='one_to_one')
impact_price['unit_usd'] = impact_price['price_number'] * impact_price['fx']
impact_price['valuation_reason'] = np.select([
    ~impact_price['asset_class'].eq('EQUITY'), impact_price['price_number'].isna(), impact_price['fx'].isna(),
    ~impact_price['result'].eq('pass')],
    ['quote/quantity basis not verified for asset class','no unique eligible snapshot quote','no unique positive snapshot FX',
     'vendor dispersion or insufficient independent quotes'],default='eligible equity scenario')
impact_price.loc[impact_price['valuation_reason'].ne('eligible equity scenario'),'unit_usd']=np.nan

impact_ledger = ledger_nav.assign(day=normalise_date(ledger_nav['as_of_date']), na=pd.to_numeric(ledger_nav['net_assets'],errors='coerce'))
impact_ledger = impact_ledger[impact_ledger['day'].eq(AS_OF_DATE)].copy()
impact_bad_funds = set(impact_ledger.loc[impact_ledger.duplicated(['fund_id','share_class'],keep=False) | impact_ledger['na'].isna(),'fund_id'])
impact_funds = impact_ledger.groupby('fund_id',as_index=False).agg(fund_net_assets_usd=('na',lambda x:x.sum(min_count=1)))
impact_funds.loc[impact_funds['fund_id'].isin(impact_bad_funds) | impact_funds['fund_net_assets_usd'].le(0),'fund_net_assets_usd']=np.nan
impact_details, impact_members, impact_specs = [], [], []


def impact_add(name, frame, interpretation, evidence='unresolved', zero=False, price_span=False):
    """A scenario has one financial row per account/security key, not per raw trade row."""
    frame=frame.copy()
    assert not frame.duplicated(KEY).any(), name
    impact_specs.append(dict(scenario=name, interpretation=interpretation, evidence=evidence, account_scope='directly implicated account IDs'))
    d=frame.merge(impact_accounts,on='account_id',how='left',validate='many_to_one')
    d=d.merge(impact_price,on='security_id',how='left',validate='many_to_one')
    d['valuation_reason']=d['valuation_reason'].fillna('missing or nonunique security master')
    d['delta_usd']=d['delta_units']*d['unit_usd']
    d.loc[d['delta_units'].eq(0),'delta_usd']=0.0
    d.loc[d['delta_units'].eq(0),'valuation_reason']='zero investment-unit change under this scenario'
    if price_span:
        span=impact_quotes.groupby('security_id',as_index=False).agg(low=('price_number','min'),high=('price_number','max'),vendors=('vendor','nunique'))
        d=d.merge(span,on='security_id',how='left',validate='many_to_one')
        eligible=d['asset_class'].eq('EQUITY') & d['fx'].notna() & d['vendors'].ge(2)
        d['delta_usd']=(d['quantity']*(d['high']-d['low'])*d['fx']).where(eligible)
        d['valuation_reason']=np.where(eligible,'equity high-minus-low vendor scenario','quote basis, quotes or FX unavailable')
    if zero:
        d['delta_usd']=0.0
        d['valuation_reason']='zero investment-unit effect under this reading; not evidence the record is valid'
    d['scenario']=name
    impact_details.append(d[['scenario',*KEY,'fund_id','share_class','delta_units','delta_usd','valuation_reason']])
    impact_members.append(d[['scenario','account_id','fund_id','share_class']].drop_duplicates())

# Negative rows: isolate the effect of the sign interpretation; do not repair the book.
impact_neg=sign_conflict[KEY].merge(negative_rows.groupby(KEY,as_index=False).agg(delta_units=('quantity_used',lambda x:-2*x.sum())),on=KEY,how='left',validate='one_to_one').fillna({'delta_units':0.0})
impact_add('Negative quantities: wrong-sign scenario',impact_neg,'Flip the signed contribution of active negative rows; all other treatments fixed')
impact_add('Negative quantities: wrong-type scenario',impact_neg.assign(delta_units=0.0),'Change BUY/SELL interpretation while retaining its signed holding contribution',zero=True)
# Blanks: isolate derived-versus-zero units only on keys already selected in section 2b.
impact_blank=BLANK_OMITTED[KEY].merge(derived_vs_book[KEY+['rebuilt_quantity']],on=KEY,validate='one_to_one').merge(
    zero_vs_book[KEY+['rebuilt_quantity']].rename(columns={'rebuilt_quantity':'zero_quantity'}),on=KEY,validate='one_to_one')
impact_blank['delta_units']=impact_blank['rebuilt_quantity']-impact_blank['zero_quantity']
impact_add('Blank quantity: derived-units scenario',impact_blank[KEY+['delta_units']],'Gross amount / price derivation versus blank-as-zero; source confirmation still needed')
# Reconciliation classes are disjoint by construction, but may overlap the flags above.
for impact_class, impact_group in three_way[three_way['is a break'].eq('yes')].groupby('main_class',sort=True):
    impact_frame=impact_group[KEY].copy()
    impact_frame['delta_units']=(impact_group['rebuilt_quantity']-impact_group['book_quantity']).to_numpy()
    impact_add('Reconciliation: '+impact_class,impact_frame,'Main-chain rebuild minus book; benchmark sensitivity, not a verified correction')
# Existence gaps: no price and no authoritative corrected units, so retain unknown dollars.
for impact_name,impact_mask in [
    ('Missing security master / spin-off target',~positions['security_id'].isin(securities['security_id'])),
    ('Held security with no price',~positions['security_id'].isin(prices['security_id']))]:
    impact_add(impact_name,positions.loc[impact_mask,KEY].assign(delta_units=np.nan),'Observed affected holdings; entitlement and valuation remain unknown','confirmed')
impact_add('Book holding absent from custodian',book_vs_custodian.loc[book_vs_custodian['side'].eq('only in positions'),KEY].assign(delta_units=np.nan),
    'Missing confirmation is not zero custody; no corrected quantity is known')
impact_add('Closed flat positions: cleared',three_way.loc[three_way['pattern'].eq('rebuild only, nets to zero'),KEY].assign(delta_units=0.),
    'Closed positions omitted by both sources under the documented reporting convention','cleared',zero=True)
for impact_result in ['spread above threshold','scale flag (ratio near 100)']:
    impact_flagged=positions[positions['security_id'].isin(vendor_check.loc[vendor_check['result'].eq(impact_result),'security_id'])][KEY+['quantity']].copy()
    impact_flagged['quantity']=pd.to_numeric(impact_flagged['quantity'],errors='coerce')
    impact_flagged['delta_units']=np.nan
    impact_add('Vendor quotes: '+impact_result,impact_flagged,'High quote versus low quote at the same holding; neither vendor is assumed correct',price_span=True)

# NAV arithmetic: dollar equivalent is kept once per ledger row, never repeated once per account.
impact_nav=nav_check[nav_check['result'].eq('fail') & nav_check['as_of_day'].eq(pd.Timestamp(AS_OF_DATE))].copy()
impact_nav_name='NAV arithmetic: net-assets/shares versus published NAV'
impact_specs.append(dict(scenario=impact_nav_name,interpretation='net_assets - nav_per_share * shares; equivalent inconsistency, not a known correction',evidence='confirmed',account_scope='accounts mapped to failing fund/share-class rows'))
impact_nav['delta_usd']=impact_nav['net_assets_number']-impact_nav['nav_per_share_number']*impact_nav['shares_outstanding_number']
impact_nav['scenario']=impact_nav_name
impact_nav['account_id']=''
impact_nav['security_id']=''
impact_nav['delta_units']=np.nan
impact_nav['valuation_reason']='arithmetic dollar equivalent; net assets and shares assumed authoritative'
impact_nav.loc[impact_nav['repeated key'],'delta_usd']=np.nan
impact_details.append(impact_nav[['scenario',*KEY,'fund_id','share_class','ledger_id','delta_units','delta_usd','valuation_reason']])
impact_nav_cohort=impact_accounts.merge(impact_nav[['fund_id','share_class']].drop_duplicates(),on=['fund_id','share_class'],how='inner',validate='many_to_one').assign(scenario=impact_nav_name)
impact_members.append(impact_nav_cohort[['scenario','account_id','fund_id','share_class']])
impact_detail=pd.concat(impact_details,ignore_index=True)
impact_membership=pd.concat(impact_members,ignore_index=True).drop_duplicates()
impact_spec=pd.DataFrame(impact_specs)
print('Scenarios:',len(impact_spec),'| financial detail rows:',len(impact_detail))


In [ ]:
# Aggregate within funds first. Never add basis points with different fund denominators.
def impact_aggregate(detail):
    return detail.groupby(['scenario','fund_id'],as_index=False,dropna=False).agg(
        signed_delta_usd=('delta_usd',lambda s:s.sum(min_count=1)),
        gross_delta_usd=('delta_usd',lambda s:s.abs().sum(min_count=1)),
        detail_rows=('delta_usd','size'),valued_rows=('delta_usd','count'))

impact_by_fund=impact_aggregate(impact_detail).merge(impact_funds,on='fund_id',how='left',validate='many_to_one')
impact_by_fund['signed_delta_bps']=impact_by_fund['signed_delta_usd']/impact_by_fund['fund_net_assets_usd']*1e4
impact_by_fund['gross_delta_bps']=impact_by_fund['gross_delta_usd']/impact_by_fund['fund_net_assets_usd']*1e4
impact_by_fund['valuation_complete']=impact_by_fund['valued_rows'].eq(impact_by_fund['detail_rows'])
impact_fund_counts=impact_membership.groupby(['scenario','fund_id'],as_index=False,dropna=False).agg(distinct_accounts=('account_id','nunique'))
impact_by_fund=impact_by_fund.merge(impact_fund_counts,on=['scenario','fund_id'],how='left',validate='one_to_one')
impact_summaries=[]
for impact_name,impact_group in impact_detail.groupby('scenario',sort=False):
    impact_m=impact_membership[impact_membership['scenario'].eq(impact_name)]
    impact_f=impact_by_fund[impact_by_fund['scenario'].eq(impact_name)]
    impact_fund_ids=impact_m['fund_id'].dropna().unique()
    impact_summaries.append(dict(scenario=impact_name,financial_rows=len(impact_group),
        distinct_accounts=impact_m['account_id'].nunique(),
        accounts_without_fund_mapping=impact_m.loc[impact_m['fund_id'].isna(),'account_id'].nunique(),
        affected_funds=len(impact_fund_ids),fund_wide_account_cohort=impact_accounts.loc[impact_accounts['fund_id'].isin(impact_fund_ids),'account_id'].nunique(),
        valued_rows=int(impact_group['delta_usd'].notna().sum()),unknown_rows=int(impact_group['delta_usd'].isna().sum()),
        valuation_coverage=impact_group['delta_usd'].notna().mean(),
        signed_delta_usd_valued_subset=impact_group['delta_usd'].sum(min_count=1),
        gross_delta_usd_valued_subset=impact_group['delta_usd'].abs().sum(min_count=1),
        max_abs_fund_net_bps_valued_subset=impact_f['signed_delta_bps'].abs().max(),
        max_fund_gross_bps_valued_subset=impact_f['gross_delta_bps'].max(),
        funds_without_NAV_denominator=int((impact_f['fund_id'].notna() & impact_f['fund_net_assets_usd'].isna()).sum())))
impact_summary=impact_spec.merge(pd.DataFrame(impact_summaries),on='scenario',how='left',validate='one_to_one')
# A scenario with no affected records counts 0 rows, accounts and funds; its dollar measures stay unknown (blank).
impact_counts=['financial_rows','distinct_accounts','accounts_without_fund_mapping','affected_funds',
               'fund_wide_account_cohort','valued_rows','unknown_rows','funds_without_NAV_denominator']
impact_summary[impact_counts]=impact_summary[impact_counts].fillna(0).astype(int)
impact_summary['account_ids_in_extract']=pd.concat([accounts['account_id'],positions['account_id'],transactions['account_id']]).nunique()
impact_summary['account_share_of_extract']=impact_summary['distinct_accounts']/impact_summary['account_ids_in_extract']
# Checks catch aggregation bugs, offsets and accidental orphan drops.
assert len(impact_detail)==len(impact_detail.drop_duplicates(['scenario','account_id','security_id','ledger_id']))
assert (impact_summary['valued_rows']+impact_summary['unknown_rows']).eq(impact_summary['financial_rows']).all()
assert impact_summary['distinct_accounts'].le(impact_summary['account_ids_in_extract']).all()
assert np.isclose(impact_detail['delta_usd'].abs().sum(),impact_by_fund['gross_delta_usd'].sum())
assert np.isclose(impact_detail['delta_usd'].sum(),impact_by_fund['signed_delta_usd'].sum())
# Synthetic checks: opposite errors must net within a fund while remaining visible gross; unknowns stay unknown.
impact_toy=pd.DataFrame({'scenario':['x','x','x','x'],'fund_id':['a','a','b',None],'delta_usd':[100.,-100.,np.nan,30.]})
impact_toy_result=impact_aggregate(impact_toy)
assert impact_toy_result.loc[impact_toy_result['fund_id'].eq('a'),'signed_delta_usd'].iloc[0]==0
assert impact_toy_result.loc[impact_toy_result['fund_id'].eq('a'),'gross_delta_usd'].iloc[0]==200
assert impact_toy_result.loc[impact_toy_result['fund_id'].eq('b'),'signed_delta_usd'].isna().all()
assert impact_toy_result['detail_rows'].sum()==4
pd.testing.assert_frame_equal(impact_aggregate(impact_detail),impact_aggregate(impact_detail.sample(frac=1,random_state=7)),check_exact=False)
for impact_file,impact_data in [('severity',impact_summary),('fund_impact',impact_by_fund),('impact_detail',impact_detail),('account_membership',impact_membership)]:
    impact_data.to_csv(OUTPUTS / f'02_{impact_file}.csv',index=False,lineterminator='\n')
display(impact_summary[['scenario','distinct_accounts','affected_funds','gross_delta_usd_valued_subset','max_abs_fund_net_bps_valued_subset','valuation_coverage']])


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, NullLocator
from matplotlib.patches import FancyBboxPatch

# Presentation only: all values are read from the existing severity summary.
impact_design = [
    ('A', 'Negative quantities: wrong-sign scenario', 'Negative quantity', 'Wrong-sign scenario', '#2563EB'),
    ('B', 'Reconciliation: rebuild is the odd one out: counts duplicate copies', 'Duplicate trades', 'Book + custodian count copies', '#7856B5'),
    ('C', 'Reconciliation: rebuild is the odd one out: double-offsets REVERSED', 'Reversal treatment', 'Book + custodian double-offset', '#087E8B'),
    ('D', 'Blank quantity: derived-units scenario', 'Missing trade quantity', 'Derived-units scenario', '#C27A28'),
    ('E', 'Vendor quotes: spread above threshold', 'Vendor price spread', 'High-quote vs. low-quote scenario', '#C34B71'),
    ('F', 'NAV arithmetic: net-assets/shares versus published NAV', 'NAV arithmetic', 'Equivalent-dollar inconsistency', '#344C67'),
    ('G', 'Negative quantities: wrong-type scenario', 'Negative quantity', 'Wrong-type scenario; zero unit effect', '#6F7C8D'),
    ('H', 'Closed flat positions: cleared', 'Closed positions', 'Cleared reporting convention', '#859387'),
]
impact_lookup=impact_summary.set_index('scenario',verify_integrity=True)
# A scenario exists only when its break class occurs in the data: plot the ones this run produced.
impact_design=[row for row in impact_design if row[1] in impact_lookup.index]
impact_ink='#192A40'; impact_muted='#637489'; impact_grid='#E7EDF4'
with plt.rc_context({'font.family':'DejaVu Sans','font.size':11,'axes.unicode_minus':False}):
    fig=plt.figure(figsize=(18,10.5),facecolor='#F7F9FC')
    fig.text(.055,.935,'ERROR SEVERITY',fontsize=11,fontweight='bold',color='#3564A8')
    fig.text(.055,.887,'Financial exposure × account reach',fontsize=28,fontweight='bold',color=impact_ink)
    fig.text(.055,.848,'Selected scenarios · valuation snapshot '+AS_OF_DATE+' · indicative sensitivity, not verified NAV loss',fontsize=12,color=impact_muted)

    ax=fig.add_axes([.075,.345,.45,.445],facecolor='white')
    ax.set_xscale('log')
    impact_positive=impact_summary.loc[impact_summary['scenario'].isin([r[1] for r in impact_design]),'max_abs_fund_net_bps_valued_subset'].dropna()
    impact_ymax=max(float(impact_positive.max())*1.15 if len(impact_positive) else 0.,100.)   # no valued scenario: default scale
    ax.set_xlim(25,50000); ax.set_ylim(-impact_ymax*.055,impact_ymax)
    ax.set_xticks([50,100,500,1000,5000,10000,30000])
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v,_: f'{v/1000:g}k' if v>=1000 else f'{v:g}'))
    ax.xaxis.set_minor_locator(NullLocator())
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v,_: f'{v:,.0f}'))
    ax.tick_params(axis='both',length=0,labelsize=10,pad=9,colors=impact_muted)
    ax.grid(axis='y',color=impact_grid,zorder=0)
    ax.grid(axis='x',color=impact_grid,alpha=.55,zorder=0)
    for edge in ax.spines.values(): edge.set_visible(False)
    ax.axhline(0,color='#CED8E4',linewidth=1,zorder=1)
    ax.set_xlabel('Distinct accounts · logarithmic scale',fontsize=12,labelpad=14,color=impact_ink)
    ax.set_ylabel('Largest fund sensitivity · bps',fontsize=12,labelpad=12,color=impact_ink)
    fig.text(.075,.805,'FINANCIAL MAGNITUDE',fontsize=10,fontweight='bold',color=impact_muted)
    for code,name,title,subtitle,color in impact_design:
        row=impact_lookup.loc[name]
        y=row['max_abs_fund_net_bps_valued_subset']
        if pd.notna(y):
            ax.scatter(row['distinct_accounts'],y,s=360,color=color,edgecolors='white',linewidths=2,zorder=4)
            ax.text(row['distinct_accounts'],y,code,color='white',fontsize=10,fontweight='bold',ha='center',va='center',zorder=5)

    # A matching letter connects each plotted point to its exact values; no labels collide in the scatter.
    panel=fig.add_axes([.575,.318,.38,.49]); panel.axis('off')
    panel.text(0,1,'SCENARIO',fontsize=10,fontweight='bold',color=impact_muted)
    for x,label in [(.61,'ACCOUNTS'),(.80,'BPS'),(.99,'VALUED')]:
        panel.text(x,1,label,ha='right',fontsize=9,fontweight='bold',color=impact_muted)
    for i,(code,name,title,subtitle,color) in enumerate(impact_design):
        row=impact_lookup.loc[name]; y=.916-i*.113
        panel.axhline(y-.071,color=impact_grid,linewidth=.8)
        panel.scatter(.023,y,s=245,color=color,clip_on=False)
        panel.text(.023,y,code,color='white',fontsize=9,fontweight='bold',ha='center',va='center')
        panel.text(.078,y+.008,title,fontsize=11,fontweight='bold',color=impact_ink,va='bottom')
        panel.text(.078,y-.035,subtitle,fontsize=8.4,color=impact_muted,va='center')
        panel.text(.61,y,f"{int(row['distinct_accounts']):,}",ha='right',va='center',fontsize=11,color=impact_ink)
        bps=row['max_abs_fund_net_bps_valued_subset']
        panel.text(.80,y,'—' if pd.isna(bps) else f'{bps:,.1f}',ha='right',va='center',fontsize=11,fontweight='bold',color=impact_ink)
        panel.text(.99,y,f"{row['valuation_coverage']:.0%}",ha='right',va='center',fontsize=11,color=impact_muted)
    panel.set_xlim(-.015,1.015); panel.set_ylim(-.03,1.035)

    fig.text(.075,.268,'How to read it',fontsize=10,fontweight='bold',color=impact_ink)
    fig.text(.075,.244,'Higher = larger modeled fund effect. Further right = more accounts. 100 bps = 1% of fund net assets.',fontsize=10,color=impact_muted)
    fig.text(.075,.222,'BPS uses the largest absolute net fund effect on the valued subset. VALUED is scenario-wide record coverage.',fontsize=10,color=impact_muted)

    fig.text(.055,.181,'FINANCIAL IMPACT STILL UNKNOWN',fontsize=10,fontweight='bold',color=impact_muted)
    impact_unknown_cards=[
        ('Missing security master / spin-off target','Missing master / no price','Same holdings; count this cohort once'),
        ('Book holding absent from custodian','No custodian confirmation','Missing confirmation is not zero custody'),
        ('Vendor quotes: scale flag (ratio near 100)','100× vendor scale flag','Bond quote basis remains unresolved')]
    for i,(name,label,note) in enumerate(impact_unknown_cards):
        left=.055+i*.305
        fig.add_artist(FancyBboxPatch((left,.077),.288,.084,boxstyle='round,pad=0.009,rounding_size=0.008',transform=fig.transFigure,facecolor='#EBF0F6',edgecolor='none',zorder=0))
        fig.text(left+.010,.134,label,fontsize=11,fontweight='bold',color=impact_ink)
        fig.text(left+.010,.108,f"{int(impact_lookup.loc[name,'distinct_accounts']):,} accounts",fontsize=13,fontweight='bold',color=impact_ink)
        fig.text(left+.010,.087,note,fontsize=9,color=impact_muted)
    fig.text(.055,.038,'Scenarios overlap; do not sum. Account counts are direct except NAV arithmetic, which uses the fund/share-class cohort.',fontsize=9,color=impact_muted)
    fig.text(.055,.018,'Partial valuation excludes unresolved inputs. Unknown financial impact is not zero. The complete scenario list remains in 02_severity.csv.',fontsize=9,color=impact_muted)
    fig.savefig(OUTPUTS/'02_severity.png',dpi=200,facecolor=fig.get_facecolor())
    fig.savefig(OUTPUTS/'02_severity.pdf',facecolor=fig.get_facecolor())
    plt.show()


## 12. Run record

Runtime, environment and the row ledger, so another team can check what this run did. **Robustness:** set `shuffle_seed` in `config/reconciliation.toml` to any number and rerun: every table's rows are shuffled after loading, and every output must stay the same. The only exception is the keep-first and keep-second sensitivities, which depend on row order by definition.

In [ ]:
run_info = pd.DataFrame([
    ('runtime_seconds', round(time.time() - RUN_START, 1)),
    ('python', platform.python_version()), ('pandas', pd.__version__), ('numpy', np.__version__),
    ('duckdb', duckdb.__version__), ('machine', platform.machine()), ('system', f'{platform.system()} {platform.release()}'),
    ('processor', platform.processor() or 'not reported'), ('config', CONFIG_PATH.name), ('shuffle_seed', SHUFFLE_SEED),
    ('as_of_date', AS_OF_DATE), ('quantity_match_tolerance', TOLERANCE),
], columns=['item', 'value'])
run_info.to_csv(OUTPUTS / '02_run_info.csv', index=False, lineterminator='\n')
row_ledger = pd.DataFrame(ROW_LEDGER)
row_ledger.to_csv(OUTPUTS / '02_row_ledger.csv', index=False, lineterminator='\n')
print(f"outputs written: {sorted(p.name for p in OUTPUTS.glob('02_*.csv'))}")
display(run_info)
row_ledger